# Notebook d'origine (projet academique)

Les chemins ont ete rendus relatifs (`./Data/...`) et les images de sortie retirees (voir `docs/`). Les images du dataset et les poids ne sont pas inclus.

**Limites connues** (detaillees dans le README) :

- le mAP50 / mAP50-95 de Faster R-CNN, DETR et RT-DETR est un calcul maison, different de celui d'Ultralytics (YOLOv8) ;
- le F1 de ces trois modeles ne porte que sur les boites deja appariees (objets manques et fausses detections ignores) ;
- la cellule `FASTER R-CNN - RESULTATS PAR CLASSE` contient des valeurs saisies a la main qui sont celles de YOLOv8 (voir sortie de la cellule YOLOv8 plus haut) ;
- dans la cellule de comparaison finale, la colonne YOLOv8 "F1" contient en realite la precision moyenne ;
- le jeu de validation de YOLO est une copie de 10 % du train ; les meilleurs epochs de DETR et RT-DETR sont choisis sur le jeu de test.


In [2]:
import os

label_dir = "./Data/train/labels"

classes = set()

for f in os.listdir(label_dir):
    if f.endswith(".txt"):
        with open(os.path.join(label_dir, f), "r") as file:
            for line in file:
                cls = line.split()[0]
                classes.add(int(cls))

print("Classes trouvées:", sorted(list(classes)))


Classes trouvées: [0, 1, 2, 3, 4]


In [3]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))


True
NVIDIA GeForce GTX 1070 with Max-Q Design


In [1]:
import os
import random
import shutil

# ==== Chemins à modifier si besoin ====
train_images = "./Data/train/images"
train_labels = "./Data/train/labels"

val_images = "./Data/val/images"
val_labels = "./Data/val/labels"

# ==== Création des dossiers val s'ils n'existent pas ====
os.makedirs(val_images, exist_ok=True)
os.makedirs(val_labels, exist_ok=True)

# ==== Récupération de toutes les images ====
images = [f for f in os.listdir(train_images) if f.lower().endswith((".jpg", ".jpeg", ".png"))]

# ==== Calcul 10% ====
n = max(1, int(0.10 * len(images)))
selected = random.sample(images, n)

print(f"Total images : {len(images)}")
print(f"Images déplacées vers val : {n}")

# ==== Copie des fichiers ====
for img in selected:
    # Copier image
    src_img = os.path.join(train_images, img)
    dst_img = os.path.join(val_images, img)
    shutil.copy2(src_img, dst_img)

    # Trouver le label correspondant (.txt)
    label_name = os.path.splitext(img)[0] + ".txt"
    src_label = os.path.join(train_labels, label_name)
    dst_label = os.path.join(val_labels, label_name)

    if os.path.exists(src_label):
        shutil.copy2(src_label, dst_label)
    else:
        print(f" Pas trouvé le label pour {img}")

print(" 10% du train copiés dans val !")


Total images : 677
Images déplacées vers val : 67
 10% du train copiés dans val !


In [2]:
from ultralytics import YOLO

# Charger le modèle pré-entraîné
model = YOLO("yolov8n.pt")

# Lancer l'entraînement
model.train(
    data="./Data/data.yaml",  # Ton YAML avec train et val seulement
    epochs=50,
    imgsz=640,
    batch=8,
    device="cuda"  # utilise le GPU
)


New https://pypi.org/project/ultralytics/8.3.227 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.225  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce GTX 1070 with Max-Q Design, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./Data/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train5, nbs=64, nms=Fa

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000001A34804E560>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
    

In [2]:
import pandas as pd

# Construct the table for train5 using provided metrics
data = [
    ["all", 0.932, 0.884, 0.918, 0.754],
    ["class0", 0.991, 1.000, 0.995, 0.920],
    ["class1", 0.773, 0.500, 0.621, 0.474],
    ["class2", 0.903, 1.000, 0.994, 0.919],
    ["class3", 1.000, 0.920, 0.986, 0.464],
    ["class4", 0.991, 1.000, 0.995, 0.991],
]

df = pd.DataFrame(data, columns=["Classe", "Precision", "Recall", "mAP50", "mAP50-95"])

# ---- Fonction d'analyse ----
def analyse_qualite(val):
    if val >= 0.90:
        return "Excellent"
    elif val >= 0.75:
        return "Bon"
    elif val >= 0.60:
        return "Moyen"
    else:
        return "Faible"

# Ajout des analyses de qualité pour chaque métrique
df["Qualité Precision"] = df["Precision"].apply(analyse_qualite)
df["Qualité Recall"] = df["Recall"].apply(analyse_qualite)
df["Qualité mAP50"] = df["mAP50"].apply(analyse_qualite)
df["Qualité mAP50-95"] = df["mAP50-95"].apply(analyse_qualite)

df


,Classe,Precision,Recall,mAP50,mAP50-95,Qualité Precision,Qualité Recall,Qualité mAP50,Qualité mAP50-95
0,all,0.932,0.884,0.918,0.754,Excellent,Bon,Excellent,Bon
1,class0,0.991,1.000,0.995,0.920,Excellent,Excellent,Excellent,Excellent
2,class1,0.773,0.500,0.621,0.474,Bon,Faible,Moyen,Faible
3,class2,0.903,1.000,0.994,0.919,Excellent,Excellent,Excellent,Excellent
4,class3,1.000,0.920,0.986,0.464,Excellent,Excellent,Excellent,Faible
5,class4,0.991,1.000,0.995,0.991,Excellent,Excellent,Excellent,Excellent


In [10]:
from ultralytics import YOLO
import pandas as pd

# Chemin du modèle
model_path = r".\runs\detect\train5\weights\last.pt"

# Chemin du YAML contenant train, val et test
data_yaml = r".\Data\data.yaml"

# Charger le modèle
model = YOLO(model_path)

# Évaluation sur le train set
results = model.val(data=data_yaml, split="train")  # split="train" pour évaluer sur le train set

# Récupérer les métriques par classe
per_class_metrics = results.summary()  # summary() retourne une liste de dicts par classe
df_metrics = pd.DataFrame(per_class_metrics)

# Sauvegarder dans un CSV
csv_path = r".\Train_metrics_per_class.csv"
df_metrics.to_csv(csv_path, index=False)

print(f"Métriques par classe sauvegardées dans : {csv_path}")
print(df_metrics)


Ultralytics 8.3.225  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce GTX 1070 with Max-Q Design, 8192MiB)
Model summary (fused): 72 layers, 3,006,623 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 141.941.8 MB/s, size: 69.4 KB)
val: Scanning .\Data\train\labels.cache... 677 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 677/677  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 43/43 3.6it/s 12.0s0.3s
                   all        677       2100      0.939      0.893      0.921      0.748
                class0        675        686      0.993          1      0.995      0.924
                class1         85         85      0.919      0.729      0.781      0.514
                class2        425        433      0.898      0.965      0.983      0.904
                class3        213        214      0.887      0.774      0.849      0.418
                class4        674    

# resultats de yolo Test 

In [9]:
from ultralytics import YOLO
import pandas as pd
import os

# Chemins
best_model_path = r".\runs\detect\train5\weights\best.pt"
data_yaml = r".\Data\data.yaml"

# Charger le modèle et évaluer
model = YOLO(best_model_path)
results = model.val(data=data_yaml)

# summary() retourne une liste de dict, un dict par classe
per_class_summary = results.summary()

# Transformer en DataFrame
df_per_class = pd.DataFrame(per_class_summary)

# Renommer les colonnes pour plus de clarté
df_per_class.rename(columns={
    'class': 'class_id',
    'name': 'class_name',
    'p': 'precision',
    'r': 'recall',
    'map50': 'mAP50',
    'map50_95': 'mAP50-95'
}, inplace=True)

# Sauvegarder en CSV
csv_path = r".\Test_metrics_per_class.csv"
df_per_class.to_csv(csv_path, index=False)
print(f"Métriques par classe sauvegardées dans : {csv_path}")

print(df_per_class)


Ultralytics 8.3.225  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce GTX 1070 with Max-Q Design, 8192MiB)
Model summary (fused): 72 layers, 3,006,623 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 219.280.7 MB/s, size: 67.1 KB)
val: Scanning .\Data\val\labels.cache... 67 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 67/67 32.6Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 0.5it/s 10.2s1.2s
                   all         67        208      0.929      0.883      0.918      0.757
                class0         67         68      0.991          1      0.995      0.925
                class1          4          4      0.773        0.5      0.621      0.473
                class2         50         52      0.888          1      0.994      0.917
                class3         16         16          1      0.917      0.986      0.477
                class4         67  

# Fog_Level1 a 5 Yolo

In [18]:
from ultralytics import YOLO
import pandas as pd
import os
import shutil

# Chemin vers ton modèle
model_path = r".\runs\detect\train5\weights\best.pt"
model = YOLO(model_path)

# Dossier contenant fog_level1 ... fog_level5
base_path = r".\Data\results"

# Liste des niveaux de fog
fog_levels = [f"fog_level{i}" for i in range(1, 6)]

# Liste pour stocker les métriques
all_metrics = []

for level in fog_levels:
    print(f"\n----------------------")
    print(f" Évaluation : {level}")
    print(f"----------------------")
    
    fog_path = os.path.join(base_path, level)
    images_dir = os.path.join(fog_path, "images")
    labels_dir = os.path.join(fog_path, "labels")  # <-- modification ici

    # Copier les labels dans le dossier images si ce n'est pas déjà fait
    if os.path.exists(labels_dir):
        for label_file in os.listdir(labels_dir):
            src = os.path.join(labels_dir, label_file)
            dst = os.path.join(images_dir, label_file)
            if not os.path.exists(dst):
                shutil.copy(src, dst)
    else:
        print(f" Dossier labels non trouvé pour {level}, vérifie le chemin : {labels_dir}")

    # Générer un YAML temporaire pour YOLO
    yaml_content = f"""
path: {images_dir}
train: {images_dir}   # obligatoire pour YOLO
val: {images_dir}     # obligatoire pour YOLO
test: {images_dir}

names:
  0: class0
  1: class1
  2: class2
  3: class3
  4: class4
"""
    yaml_file = os.path.join(fog_path, f"{level}_temp.yaml")
    with open(yaml_file, "w") as f:
        f.write(yaml_content)

    # Évaluation YOLO sur ce fog_level
    results = model.val(data=yaml_file)

    # Récupération des métriques globales
    precision, recall, mAP50, mAP5095 = results.mean_results()
    
    all_metrics.append({
        "fog_level": level,
        "precision": precision,
        "recall": recall,
        "mAP50": mAP50,
        "mAP50-95": mAP5095
    })

# Convertir en DataFrame et enregistrer dans un CSV
df = pd.DataFrame(all_metrics)
csv_output = os.path.join(base_path, "Fog_levels_metrics.csv")
df.to_csv(csv_output, index=False)

print("\n Évaluation terminée pour les 5 fog levels")
print(f"Métriques sauvegardées dans : {csv_output}")
print(df)



----------------------
 Évaluation : fog_level1
----------------------
Ultralytics 8.3.225  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce GTX 1070 with Max-Q Design, 8192MiB)
Model summary (fused): 72 layers, 3,006,623 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 403.668.4 MB/s, size: 103.6 KB)
val: Scanning .\Data\results\fog_level1\labels... 500 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 500/500 1.2Kit/s 0.4s<0.0s
val: New cache created: .\Data\results\fog_level1\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 32/32 4.3it/s 7.5s0.2s
                   all        500       1559      0.947      0.888      0.917      0.739
                class0        498        508      0.993      0.998      0.995      0.912
                class1         75         75      0.948      0.787      0.833       0.53
                class2        321        329      0.932     

# Glare_level1 a 5 yolo

In [19]:
from ultralytics import YOLO
import pandas as pd
import os
import shutil

# Chemin vers ton modèle
model_path = r".\runs\detect\train5\weights\best.pt"
model = YOLO(model_path)

# Dossier contenant glare_level1 ... glare_level5
base_path = r".\Data\results"

# Liste des niveaux de glare
glare_levels = [f"glare_level{i}" for i in range(1, 6)]

# Liste pour stocker les métriques
all_metrics = []

for level in glare_levels:
    print(f"\n----------------------")
    print(f" Évaluation : {level}")
    print(f"----------------------")
    
    glare_path = os.path.join(base_path, level)
    images_dir = os.path.join(glare_path, "images")
    labels_dir = os.path.join(glare_path, "labels")  # labels à l’intérieur du dossier glare_levelX

    # Copier les labels dans le dossier images si ce n'est pas déjà fait
    if os.path.exists(labels_dir):
        for label_file in os.listdir(labels_dir):
            src = os.path.join(labels_dir, label_file)
            dst = os.path.join(images_dir, label_file)
            if not os.path.exists(dst):
                shutil.copy(src, dst)
    else:
        print(f" Dossier labels non trouvé pour {level}, vérifie le chemin : {labels_dir}")

    # Générer un YAML temporaire pour YOLO
    yaml_content = f"""
path: {images_dir}
train: {images_dir}   # obligatoire pour YOLO
val: {images_dir}     # obligatoire pour YOLO
test: {images_dir}

names:
  0: class0
  1: class1
  2: class2
  3: class3
  4: class4
"""
    yaml_file = os.path.join(glare_path, f"{level}_temp.yaml")
    with open(yaml_file, "w") as f:
        f.write(yaml_content)

    # Évaluation YOLO sur ce glare_level
    results = model.val(data=yaml_file)

    # Récupération des métriques globales
    precision, recall, mAP50, mAP5095 = results.mean_results()
    
    all_metrics.append({
        "glare_level": level,
        "precision": precision,
        "recall": recall,
        "mAP50": mAP50,
        "mAP50-95": mAP5095
    })

# Convertir en DataFrame et enregistrer dans un CSV
df = pd.DataFrame(all_metrics)
csv_output = os.path.join(base_path, "Glare_levels_metrics.csv")
df.to_csv(csv_output, index=False)

print("\n Évaluation terminée pour les 5 glare levels")
print(f"Métriques sauvegardées dans : {csv_output}")
print(df)



----------------------
 Évaluation : glare_level1
----------------------
Ultralytics 8.3.225  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce GTX 1070 with Max-Q Design, 8192MiB)
Model summary (fused): 72 layers, 3,006,623 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 99.739.0 MB/s, size: 123.8 KB)
val: Scanning .\Data\results\glare_level1\labels... 500 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 500/500 1.2Kit/s 0.4s<0.0s
val: New cache created: .\Data\results\glare_level1\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 32/32 4.3it/s 7.5s0.2s
                   all        500       1559      0.931      0.892      0.924      0.748
                class0        498        508      0.993          1      0.995      0.915
                class1         75         75      0.928      0.787      0.837       0.55
                class2        321        329       0.89

# Night_level 1 a 5 yolo

In [20]:
from ultralytics import YOLO
import pandas as pd
import os
import shutil

# Chemin vers ton modèle
model_path = r".\runs\detect\train5\weights\best.pt"
model = YOLO(model_path)

# Dossier contenant night_level1 ... night_level5
base_path = r".\Data\results"

# Liste des niveaux de night
night_levels = [f"night_level{i}" for i in range(1, 6)]

# Liste pour stocker les métriques
all_metrics = []

for level in night_levels:
    print(f"\n----------------------")
    print(f" Évaluation : {level}")
    print(f"----------------------")
    
    night_path = os.path.join(base_path, level)
    images_dir = os.path.join(night_path, "images")
    labels_dir = os.path.join(night_path, "labels")  # labels à l’intérieur du dossier night_levelX

    # Copier les labels dans le dossier images si ce n'est pas déjà fait
    if os.path.exists(labels_dir):
        for label_file in os.listdir(labels_dir):
            src = os.path.join(labels_dir, label_file)
            dst = os.path.join(images_dir, label_file)
            if not os.path.exists(dst):
                shutil.copy(src, dst)
    else:
        print(f" Dossier labels non trouvé pour {level}, vérifie le chemin : {labels_dir}")

    # Générer un YAML temporaire pour YOLO
    yaml_content = f"""
path: {images_dir}
train: {images_dir}   # obligatoire pour YOLO
val: {images_dir}     # obligatoire pour YOLO
test: {images_dir}

names:
  0: class0
  1: class1
  2: class2
  3: class3
  4: class4
"""
    yaml_file = os.path.join(night_path, f"{level}_temp.yaml")
    with open(yaml_file, "w") as f:
        f.write(yaml_content)

    # Évaluation YOLO sur ce night_level
    results = model.val(data=yaml_file)

    # Récupération des métriques globales
    precision, recall, mAP50, mAP5095 = results.mean_results()
    
    all_metrics.append({
        "night_level": level,
        "precision": precision,
        "recall": recall,
        "mAP50": mAP50,
        "mAP50-95": mAP5095
    })

# Convertir en DataFrame et enregistrer dans un CSV
df = pd.DataFrame(all_metrics)
csv_output = os.path.join(base_path, "Night_levels_metrics.csv")
df.to_csv(csv_output, index=False)

print("\n Évaluation terminée pour les 5 night levels")
print(f"Métriques sauvegardées dans : {csv_output}")
print(df)



----------------------
 Évaluation : night_level1
----------------------
Ultralytics 8.3.225  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce GTX 1070 with Max-Q Design, 8192MiB)
Model summary (fused): 72 layers, 3,006,623 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 71.131.4 MB/s, size: 112.6 KB)
val: Scanning .\Data\results\night_level1\labels... 500 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 500/500 1.3Kit/s 0.4s<0.1s
val: New cache created: .\Data\results\night_level1\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 32/32 3.8it/s 8.5s0.2s
                   all        500       1559      0.918      0.836       0.88      0.699
                class0        498        508      0.998      0.997      0.995      0.909
                class1         75         75      0.944       0.67      0.761      0.446
                class2        321        329      0.855

# Rain_level1 a 5 yolo

In [21]:
from ultralytics import YOLO
import pandas as pd
import os
import shutil

# Chemin vers ton modèle
model_path = r".\runs\detect\train5\weights\best.pt"
model = YOLO(model_path)

# Dossier contenant rain_level1 ... rain_level5
base_path = r".\Data\results"

# Liste des niveaux de rain
rain_levels = [f"rain_level{i}" for i in range(1, 6)]

# Liste pour stocker les métriques
all_metrics = []

for level in rain_levels:
    print(f"\n----------------------")
    print(f" Évaluation : {level}")
    print(f"----------------------")
    
    rain_path = os.path.join(base_path, level)
    images_dir = os.path.join(rain_path, "images")
    labels_dir = os.path.join(rain_path, "labels")  # labels à l’intérieur du dossier rain_levelX

    # Copier les labels dans le dossier images si ce n'est pas déjà fait
    if os.path.exists(labels_dir):
        for label_file in os.listdir(labels_dir):
            src = os.path.join(labels_dir, label_file)
            dst = os.path.join(images_dir, label_file)
            if not os.path.exists(dst):
                shutil.copy(src, dst)
    else:
        print(f" Dossier labels non trouvé pour {level}, vérifie le chemin : {labels_dir}")

    # Générer un YAML temporaire pour YOLO
    yaml_content = f"""
path: {images_dir}
train: {images_dir}   # obligatoire pour YOLO
val: {images_dir}     # obligatoire pour YOLO
test: {images_dir}

names:
  0: class0
  1: class1
  2: class2
  3: class3
  4: class4
"""
    yaml_file = os.path.join(rain_path, f"{level}_temp.yaml")
    with open(yaml_file, "w") as f:
        f.write(yaml_content)

    # Évaluation YOLO sur ce rain_level
    results = model.val(data=yaml_file)

    # Récupération des métriques globales
    precision, recall, mAP50, mAP5095 = results.mean_results()
    
    all_metrics.append({
        "rain_level": level,
        "precision": precision,
        "recall": recall,
        "mAP50": mAP50,
        "mAP50-95": mAP5095
    })

# Convertir en DataFrame et enregistrer dans un CSV
df = pd.DataFrame(all_metrics)
csv_output = os.path.join(base_path, "Rain_levels_metrics.csv")
df.to_csv(csv_output, index=False)

print("\n Évaluation terminée pour les 5 rain levels")
print(f"Métriques sauvegardées dans : {csv_output}")
print(df)



----------------------
 Évaluation : rain_level1
----------------------
Ultralytics 8.3.225  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce GTX 1070 with Max-Q Design, 8192MiB)
Model summary (fused): 72 layers, 3,006,623 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 85.831.5 MB/s, size: 93.2 KB)
val: Scanning .\Data\results\rain_level1\labels... 500 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 500/500 1.2Kit/s 0.4s<0.0s
val: New cache created: .\Data\results\rain_level1\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 32/32 4.4it/s 7.3s0.2s
                   all        500       1559      0.883       0.78      0.832      0.617
                class0        498        508      0.998      0.885      0.991       0.78
                class1         75         75      0.826       0.68      0.764      0.377
                class2        321        329      0.703    

# Snow_level 1 a 5 Yolo

In [1]:
from ultralytics import YOLO
import pandas as pd
import os
import shutil

# Chemin vers ton modèle
model_path = r".\runs\detect\train5\weights\best.pt"
model = YOLO(model_path)

# Dossier contenant snow_level1 ... snow_level5
base_path = r".\Data\results"

# Liste des niveaux de snow
snow_levels = [f"snow_level{i}" for i in range(1, 6)]

# Liste pour stocker les métriques
all_metrics = []

for level in snow_levels:
    print(f"\n----------------------")
    print(f" Évaluation : {level}")
    print(f"----------------------")
    
    snow_path = os.path.join(base_path, level)
    images_dir = os.path.join(snow_path, "images")
    labels_dir = os.path.join(snow_path, "labels")  # labels à l’intérieur du dossier snow_levelX

    # Copier les labels dans le dossier images si ce n'est pas déjà fait
    if os.path.exists(labels_dir):
        for label_file in os.listdir(labels_dir):
            src = os.path.join(labels_dir, label_file)
            dst = os.path.join(images_dir, label_file)
            if not os.path.exists(dst):
                shutil.copy(src, dst)
    else:
        print(f" Dossier labels non trouvé pour {level}, vérifie le chemin : {labels_dir}")

    # Générer un YAML temporaire pour YOLO
    yaml_content = f"""
path: {images_dir}
train: {images_dir}   # obligatoire pour YOLO
val: {images_dir}     # obligatoire pour YOLO
test: {images_dir}

names:
  0: class0
  1: class1
  2: class2
  3: class3
  4: class4
"""
    yaml_file = os.path.join(snow_path, f"{level}_temp.yaml")
    with open(yaml_file, "w") as f:
        f.write(yaml_content)

    # Évaluation YOLO sur ce snow_level
    results = model.val(data=yaml_file)

    # Récupération des métriques globales
    precision, recall, mAP50, mAP5095 = results.mean_results()
    
    all_metrics.append({
        "snow_level": level,
        "precision": precision,
        "recall": recall,
        "mAP50": mAP50,
        "mAP50-95": mAP5095
    })

# Convertir en DataFrame et enregistrer dans un CSV
df = pd.DataFrame(all_metrics)
csv_output = os.path.join(base_path, "Snow_levels_metrics.csv")
df.to_csv(csv_output, index=False)

print("\n Évaluation terminée pour les 5 snow levels")
print(f"Métriques sauvegardées dans : {csv_output}")
print(df)



----------------------
 Évaluation : snow_level1
----------------------
Ultralytics 8.3.225  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce GTX 1070 with Max-Q Design, 8192MiB)
Model summary (fused): 72 layers, 3,006,623 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 746.099.3 MB/s, size: 106.3 KB)
val: Scanning .\Data\results\snow_level1\labels... 500 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 500/500 1.2Kit/s 0.4s<0.0s
val: New cache created: .\Data\results\snow_level1\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 32/32 4.7it/s 6.9s0.2s
                   all        500       1559      0.924      0.881      0.907      0.729
                class0        498        508      0.995      0.992      0.995      0.908
                class1         75         75      0.895      0.791      0.829      0.516
                class2        321        329      0.878  

# Yolo performance tableaux 

In [10]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Charger les données
fog_df = pd.read_csv("./Data/results/Fog_levels_metrics.csv")
glare_df = pd.read_csv("./Data/results/Glare_levels_metrics.csv")
night_df = pd.read_csv("./Data/results/Night_levels_metrics.csv")
rain_df = pd.read_csv("./Data/results/Rain_levels_metrics.csv")
snow_df = pd.read_csv("./Data/results/Snow_levels_metrics.csv")

# Configuration des couleurs et styles
colors = {
    'Fog': '#1f77b4',      # Bleu
    'Glare': '#ff7f0e',    # Orange
    'Night': '#2ca02c',    # Vert
    'Rain': '#d62728',     # Rouge
    'Snow': '#9467bd'      # Violet
}

markers = {
    'Fog': 'o',
    'Glare': 's',
    'Night': '^',
    'Rain': 'D',
    'Snow': 'v'
}

# Niveaux (1 à 5)
levels = [1, 2, 3, 4, 5]

# Créer la figure avec 4 sous-graphiques
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Performance du Modèle YOLOv8 par Type de Perturbation', 
             fontsize=18, fontweight='bold', y=0.995)

# Liste des métriques et leurs axes correspondants
metrics = [
    ('precision', 'Precision', axes[0, 0]),
    ('recall', 'Recall', axes[0, 1]),
    ('mAP50', 'mAP@50', axes[1, 0]),
    ('mAP50-95', 'mAP@50-95', axes[1, 1])
]

# Tracer chaque métrique
for metric_col, metric_name, ax in metrics:
    # Tracer chaque perturbation
    ax.plot(levels, fog_df[metric_col], 
            color=colors['Fog'], marker=markers['Fog'], 
            linewidth=2.5, markersize=10, label='Fog')
    
    ax.plot(levels, glare_df[metric_col], 
            color=colors['Glare'], marker=markers['Glare'], 
            linewidth=2.5, markersize=10, label='Glare')
    
    ax.plot(levels, night_df[metric_col], 
            color=colors['Night'], marker=markers['Night'], 
            linewidth=2.5, markersize=10, label='Night')
    
    ax.plot(levels, rain_df[metric_col], 
            color=colors['Rain'], marker=markers['Rain'], 
            linewidth=2.5, markersize=10, label='Rain')
    
    ax.plot(levels, snow_df[metric_col], 
            color=colors['Snow'], marker=markers['Snow'], 
            linewidth=2.5, markersize=10, label='Snow')
    
    # Configuration de l'axe
    ax.set_xlabel('Niveau de Perturbation', fontsize=13, fontweight='bold')
    ax.set_ylabel(metric_name, fontsize=13, fontweight='bold')
    ax.set_title(metric_name, fontsize=14, fontweight='bold', pad=10)
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.set_xticks(levels)
    ax.set_ylim([0, 1])
    ax.legend(loc='lower left', fontsize=11, framealpha=0.9)
    
    # Ajouter les valeurs sur les points
    for perturbation, df in [('Fog', fog_df), ('Glare', glare_df), 
                              ('Night', night_df), ('Rain', rain_df), ('Snow', snow_df)]:
        for i, level in enumerate(levels):
            value = df[metric_col].iloc[i]
            # N'afficher que les valeurs extrêmes pour ne pas surcharger
            if level == 1 or level == 5:
                ax.annotate(f'{value:.3f}', 
                           xy=(level, value), 
                           xytext=(0, 8), 
                           textcoords='offset points',
                           fontsize=8, 
                           ha='center',
                           alpha=0.7)

plt.tight_layout()
plt.savefig('./Data/results/performance_comparison.png', 
            dpi=300, bbox_inches='tight')
print(" Graphique sauvegardé : performance_comparison.png")
plt.show()

# =========================
# TABLEAU RÉCAPITULATIF
# =========================
print("\n" + "="*100)
print("TABLEAU RÉCAPITULATIF DES PERFORMANCES")
print("="*100)

# Créer un tableau consolidé
summary_data = []

for level in levels:
    summary_data.append({
        'Level': level,
        'Fog_P': fog_df.loc[level-1, 'precision'],
        'Fog_R': fog_df.loc[level-1, 'recall'],
        'Fog_mAP50': fog_df.loc[level-1, 'mAP50'],
        'Fog_mAP50-95': fog_df.loc[level-1, 'mAP50-95'],
        'Glare_P': glare_df.loc[level-1, 'precision'],
        'Glare_R': glare_df.loc[level-1, 'recall'],
        'Glare_mAP50': glare_df.loc[level-1, 'mAP50'],
        'Glare_mAP50-95': glare_df.loc[level-1, 'mAP50-95'],
        'Night_P': night_df.loc[level-1, 'precision'],
        'Night_R': night_df.loc[level-1, 'recall'],
        'Night_mAP50': night_df.loc[level-1, 'mAP50'],
        'Night_mAP50-95': night_df.loc[level-1, 'mAP50-95'],
        'Rain_P': rain_df.loc[level-1, 'precision'],
        'Rain_R': rain_df.loc[level-1, 'recall'],
        'Rain_mAP50': rain_df.loc[level-1, 'mAP50'],
        'Rain_mAP50-95': rain_df.loc[level-1, 'mAP50-95'],
        'Snow_P': snow_df.loc[level-1, 'precision'],
        'Snow_R': snow_df.loc[level-1, 'recall'],
        'Snow_mAP50': snow_df.loc[level-1, 'mAP50'],
        'Snow_mAP50-95': snow_df.loc[level-1, 'mAP50-95'],
    })

summary_df = pd.DataFrame(summary_data)
summary_df.to_csv('./Data/results/complete_summary.csv', index=False)
print("\n Tableau consolidé sauvegardé : complete_summary.csv")
print(summary_df.to_string(index=False))

# =========================
# MOYENNES PAR PERTURBATION
# =========================
print("\n" + "="*100)
print("MOYENNES PAR TYPE DE PERTURBATION")
print("="*100)

avg_data = {
    'Perturbation': ['Fog', 'Glare', 'Night', 'Rain', 'Snow'],
    'Avg_Precision': [
        fog_df['precision'].mean(),
        glare_df['precision'].mean(),
        night_df['precision'].mean(),
        rain_df['precision'].mean(),
        snow_df['precision'].mean()
    ],
    'Avg_Recall': [
        fog_df['recall'].mean(),
        glare_df['recall'].mean(),
        night_df['recall'].mean(),
        rain_df['recall'].mean(),
        snow_df['recall'].mean()
    ],
    'Avg_mAP50': [
        fog_df['mAP50'].mean(),
        glare_df['mAP50'].mean(),
        night_df['mAP50'].mean(),
        rain_df['mAP50'].mean(),
        snow_df['mAP50'].mean()
    ],
    'Avg_mAP50-95': [
        fog_df['mAP50-95'].mean(),
        glare_df['mAP50-95'].mean(),
        night_df['mAP50-95'].mean(),
        rain_df['mAP50-95'].mean(),
        snow_df['mAP50-95'].mean()
    ]
}

avg_df = pd.DataFrame(avg_data)
print(avg_df.to_string(index=False))

# =========================
# GRAPHIQUE EN BARRES DES MOYENNES
# =========================
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.suptitle('Moyennes des Performances par Type de Perturbation', 
             fontsize=16, fontweight='bold')

metrics_avg = [
    ('Avg_Precision', 'Precision Moyenne', axes[0]),
    ('Avg_Recall', 'Recall Moyen', axes[1]),
    ('Avg_mAP50', 'mAP@50 Moyen', axes[2]),
    ('Avg_mAP50-95', 'mAP@50-95 Moyen', axes[3])
]

for metric, title, ax in metrics_avg:
    bars = ax.bar(avg_df['Perturbation'], avg_df[metric], 
                  color=[colors[p] for p in avg_df['Perturbation']], 
                  alpha=0.7, edgecolor='black', linewidth=1.5)
    
    ax.set_ylabel(title, fontsize=12, fontweight='bold')
    ax.set_ylim([0, 1])
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.set_title(title, fontsize=13, fontweight='bold')
    
    # Ajouter les valeurs sur les barres
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}',
                ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('./Data/results/average_performance.png', 
            dpi=300, bbox_inches='tight')
print("\n Graphique des moyennes sauvegardé : average_performance.png")
plt.show()

print("\n" + "="*100)
print(" ANALYSE TERMINÉE - Tous les graphiques et tableaux ont été générés")
print("="*100)

 Graphique sauvegardé : performance_comparison.png


<Figure size 1600x1200 with 4 Axes>


TABLEAU RÉCAPITULATIF DES PERFORMANCES

 Tableau consolidé sauvegardé : complete_summary.csv
 Level    Fog_P    Fog_R  Fog_mAP50  Fog_mAP50-95  Glare_P  Glare_R  Glare_mAP50  Glare_mAP50-95  Night_P  Night_R  Night_mAP50  Night_mAP50-95   Rain_P   Rain_R  Rain_mAP50  Rain_mAP50-95   Snow_P   Snow_R  Snow_mAP50  Snow_mAP50-95
     1 0.946570 0.888444   0.917434      0.739336 0.930744 0.892045     0.923961        0.747775 0.917523 0.835939     0.879698        0.699469 0.882714 0.779667    0.831654       0.617434 0.923900 0.881409    0.906978       0.728555
     2 0.935887 0.879188   0.905792      0.714160 0.927456 0.899366     0.925595        0.748017 0.913384 0.794528     0.847937        0.669769 0.796077 0.731895    0.765348       0.565173 0.874040 0.837809    0.867632       0.671967
     3 0.860123 0.826829   0.875295      0.667098 0.937931 0.883712     0.921220        0.746060 0.837516 0.791854     0.818365        0.641441 0.765278 0.635464    0.702344       0.502573 0.784358 0.6651

<Figure size 1800x500 with 4 Axes>


 ANALYSE TERMINÉE - Tous les graphiques et tableaux ont été générés


In [11]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re

# =========================
# CONFIGURATION
# =========================
RESULTS_DIR = "./Data/results"
PERTURBATIONS = ['fog', 'glare', 'night', 'rain', 'snow']
LEVELS = [1, 2, 3, 4, 5]
CLASSES = ['class0', 'class1', 'class2', 'class3', 'class4']

# Couleurs par perturbation
colors_perturbation = {
    'fog': '#1f77b4',      # Bleu
    'glare': '#ff7f0e',    # Orange
    'night': '#2ca02c',    # Vert
    'rain': '#d62728',     # Rouge
    'snow': '#9467bd'      # Violet
}

# Couleurs par classe
colors_class = {
    'class0': '#e41a1c',
    'class1': '#377eb8',
    'class2': '#4daf4a',
    'class3': '#984ea3',
    'class4': '#ff7f00'
}

markers = {
    'fog': 'o',
    'glare': 's',
    'night': '^',
    'rain': 'D',
    'snow': 'v'
}

# =========================
# FONCTION POUR PARSER LES RÉSULTATS YOLO
# =========================
def extract_class_metrics_from_yolo_output(output_text):
    """
    Parse le texte de sortie YOLOv8 pour extraire les métriques par classe
    """
    lines = output_text.strip().split('\n')
    
    metrics = {}
    
    for line in lines:
        # Chercher les lignes avec les classes (class0, class1, etc.)
        if 'class' in line.lower() and 'all' not in line.lower():
            parts = line.split()
            
            # Format attendu: Class Images Instances Box(P R mAP50 mAP50-95)
            if len(parts) >= 7:
                class_name = parts[0]
                try:
                    precision = float(parts[4])
                    recall = float(parts[5])
                    map50 = float(parts[6])
                    map5095 = float(parts[7])
                    
                    metrics[class_name] = {
                        'precision': precision,
                        'recall': recall,
                        'mAP50': map50,
                        'mAP50-95': map5095
                    }
                except (ValueError, IndexError):
                    continue
    
    return metrics

# =========================
# CHARGER LES DONNÉES PAR CLASSE (À PARTIR DES SORTIES)
# =========================
# Note: Comme vous avez les sorties texte, je vais créer une structure de données
# Si vous avez déjà des CSV par classe, adaptez cette partie

# Exemple de structure - vous devrez remplir avec vos données réelles
data_by_class = {
    'fog': {level: {} for level in LEVELS},
    'glare': {level: {} for level in LEVELS},
    'night': {level: {} for level in LEVELS},
    'rain': {level: {} for level in LEVELS},
    'snow': {level: {} for level in LEVELS}
}

# Données fog (extraites de votre sortie)
data_by_class['fog'][1] = {
    'class0': {'precision': 0.993, 'recall': 0.998, 'mAP50': 0.995, 'mAP50-95': 0.912},
    'class1': {'precision': 0.948, 'recall': 0.787, 'mAP50': 0.833, 'mAP50-95': 0.530},
    'class2': {'precision': 0.932, 'recall': 0.921, 'mAP50': 0.974, 'mAP50-95': 0.882},
    'class3': {'precision': 0.864, 'recall': 0.739, 'mAP50': 0.790, 'mAP50-95': 0.399},
    'class4': {'precision': 0.996, 'recall': 0.997, 'mAP50': 0.995, 'mAP50-95': 0.972}
}

data_by_class['fog'][2] = {
    'class0': {'precision': 0.996, 'recall': 0.978, 'mAP50': 0.995, 'mAP50-95': 0.896},
    'class1': {'precision': 0.923, 'recall': 0.799, 'mAP50': 0.823, 'mAP50-95': 0.502},
    'class2': {'precision': 0.910, 'recall': 0.912, 'mAP50': 0.966, 'mAP50-95': 0.858},
    'class3': {'precision': 0.854, 'recall': 0.711, 'mAP50': 0.750, 'mAP50-95': 0.353},
    'class4': {'precision': 0.997, 'recall': 0.996, 'mAP50': 0.995, 'mAP50-95': 0.962}
}

data_by_class['fog'][3] = {
    'class0': {'precision': 0.998, 'recall': 0.854, 'mAP50': 0.987, 'mAP50-95': 0.846},
    'class1': {'precision': 0.772, 'recall': 0.760, 'mAP50': 0.788, 'mAP50-95': 0.441},
    'class2': {'precision': 0.835, 'recall': 0.900, 'mAP50': 0.920, 'mAP50-95': 0.792},
    'class3': {'precision': 0.698, 'recall': 0.635, 'mAP50': 0.687, 'mAP50-95': 0.315},
    'class4': {'precision': 0.998, 'recall': 0.986, 'mAP50': 0.994, 'mAP50-95': 0.942}
}

data_by_class['fog'][4] = {
    'class0': {'precision': 1.000, 'recall': 0.617, 'mAP50': 0.958, 'mAP50-95': 0.792},
    'class1': {'precision': 0.803, 'recall': 0.640, 'mAP50': 0.706, 'mAP50-95': 0.360},
    'class2': {'precision': 0.792, 'recall': 0.800, 'mAP50': 0.859, 'mAP50-95': 0.703},
    'class3': {'precision': 0.731, 'recall': 0.500, 'mAP50': 0.591, 'mAP50-95': 0.261},
    'class4': {'precision': 0.998, 'recall': 0.942, 'mAP50': 0.990, 'mAP50-95': 0.918}
}

data_by_class['fog'][5] = {
    'class0': {'precision': 1.000, 'recall': 0.544, 'mAP50': 0.928, 'mAP50-95': 0.760},
    'class1': {'precision': 0.781, 'recall': 0.613, 'mAP50': 0.663, 'mAP50-95': 0.331},
    'class2': {'precision': 0.772, 'recall': 0.709, 'mAP50': 0.791, 'mAP50-95': 0.632},
    'class3': {'precision': 0.723, 'recall': 0.500, 'mAP50': 0.577, 'mAP50-95': 0.242},
    'class4': {'precision': 0.991, 'recall': 0.888, 'mAP50': 0.983, 'mAP50-95': 0.905}
}

# Ajoutez les données pour glare, night, rain, snow de la même manière...
# Pour l'exemple, je vais ajouter glare level 1-5

data_by_class['glare'][1] = {
    'class0': {'precision': 0.993, 'recall': 1.000, 'mAP50': 0.995, 'mAP50-95': 0.915},
    'class1': {'precision': 0.928, 'recall': 0.787, 'mAP50': 0.837, 'mAP50-95': 0.550},
    'class2': {'precision': 0.890, 'recall': 0.964, 'mAP50': 0.980, 'mAP50-95': 0.890},
    'class3': {'precision': 0.847, 'recall': 0.711, 'mAP50': 0.813, 'mAP50-95': 0.408},
    'class4': {'precision': 0.996, 'recall': 0.999, 'mAP50': 0.995, 'mAP50-95': 0.976}
}

# ... (continuez pour les autres niveaux et perturbations)

# =========================
# FONCTION POUR CRÉER DES GRAPHIQUES PAR CLASSE
# =========================
def plot_metrics_by_class_for_perturbation(perturbation_data, perturbation_name, save_dir):
    """
    Crée un graphique 2x2 pour une perturbation donnée, montrant chaque classe
    """
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(f'Performance par Classe - {perturbation_name.upper()}', 
                 fontsize=18, fontweight='bold')
    
    metrics = [
        ('precision', 'Precision', axes[0, 0]),
        ('recall', 'Recall', axes[0, 1]),
        ('mAP50', 'mAP@50', axes[1, 0]),
        ('mAP50-95', 'mAP@50-95', axes[1, 1])
    ]
    
    for metric_name, metric_label, ax in metrics:
        for class_name in CLASSES:
            values = []
            for level in LEVELS:
                if class_name in perturbation_data[level]:
                    values.append(perturbation_data[level][class_name][metric_name])
                else:
                    values.append(0)
            
            ax.plot(LEVELS, values, 
                   color=colors_class[class_name], 
                   marker='o', 
                   linewidth=2.5, 
                   markersize=10, 
                   label=class_name.capitalize())
        
        ax.set_xlabel('Niveau de Perturbation', fontsize=13, fontweight='bold')
        ax.set_ylabel(metric_label, fontsize=13, fontweight='bold')
        ax.set_title(metric_label, fontsize=14, fontweight='bold', pad=10)
        ax.grid(True, alpha=0.3, linestyle='--')
        ax.set_xticks(LEVELS)
        ax.set_ylim([0, 1.05])
        ax.legend(loc='lower left', fontsize=11, framealpha=0.9)
    
    plt.tight_layout()
    filename = f'{save_dir}/{perturbation_name}_by_class.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f" Graphique sauvegardé : {filename}")
    plt.close()

# =========================
# FONCTION POUR CRÉER UN GRAPHIQUE PAR CLASSE (TOUTES PERTURBATIONS)
# =========================
def plot_metrics_for_class_all_perturbations(class_name, all_data, save_dir):
    """
    Crée un graphique 2x2 pour une classe donnée, montrant toutes les perturbations
    """
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(f'Performance de {class_name.upper()} - Toutes Perturbations', 
                 fontsize=18, fontweight='bold')
    
    metrics = [
        ('precision', 'Precision', axes[0, 0]),
        ('recall', 'Recall', axes[0, 1]),
        ('mAP50', 'mAP@50', axes[1, 0]),
        ('mAP50-95', 'mAP@50-95', axes[1, 1])
    ]
    
    for metric_name, metric_label, ax in metrics:
        for perturbation in PERTURBATIONS:
            values = []
            for level in LEVELS:
                if class_name in all_data[perturbation][level]:
                    values.append(all_data[perturbation][level][class_name][metric_name])
                else:
                    values.append(0)
            
            ax.plot(LEVELS, values, 
                   color=colors_perturbation[perturbation], 
                   marker=markers[perturbation], 
                   linewidth=2.5, 
                   markersize=10, 
                   label=perturbation.capitalize())
        
        ax.set_xlabel('Niveau de Perturbation', fontsize=13, fontweight='bold')
        ax.set_ylabel(metric_label, fontsize=13, fontweight='bold')
        ax.set_title(metric_label, fontsize=14, fontweight='bold', pad=10)
        ax.grid(True, alpha=0.3, linestyle='--')
        ax.set_xticks(LEVELS)
        ax.set_ylim([0, 1.05])
        ax.legend(loc='lower left', fontsize=11, framealpha=0.9)
    
    plt.tight_layout()
    filename = f'{save_dir}/{class_name}_all_perturbations.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f" Graphique sauvegardé : {filename}")
    plt.close()

# =========================
# CRÉER TOUS LES GRAPHIQUES
# =========================
save_directory = f"{RESULTS_DIR}/class_analysis"
import os
os.makedirs(save_directory, exist_ok=True)

print("\n" + "="*100)
print("GÉNÉRATION DES GRAPHIQUES PAR CLASSE")
print("="*100)

# 1. Graphiques par perturbation (montrant toutes les classes)
print("\n Graphiques par perturbation...")
for perturbation in PERTURBATIONS:
    if perturbation in data_by_class:
        plot_metrics_by_class_for_perturbation(
            data_by_class[perturbation], 
            perturbation, 
            save_directory
        )

# 2. Graphiques par classe (montrant toutes les perturbations)
print("\n Graphiques par classe...")
for class_name in CLASSES:
    plot_metrics_for_class_all_perturbations(
        class_name, 
        data_by_class, 
        save_directory
    )

# =========================
# HEATMAP DES PERFORMANCES PAR CLASSE
# =========================
print("\n Génération des heatmaps...")

for metric_name in ['precision', 'recall', 'mAP50', 'mAP50-95']:
    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    fig.suptitle(f'Heatmap {metric_name.upper()} par Classe et Niveau', 
                 fontsize=16, fontweight='bold')
    
    for idx, perturbation in enumerate(PERTURBATIONS):
        ax = axes[idx]
        
        # Créer la matrice de données
        matrix = np.zeros((len(CLASSES), len(LEVELS)))
        for i, class_name in enumerate(CLASSES):
            for j, level in enumerate(LEVELS):
                if class_name in data_by_class[perturbation][level]:
                    matrix[i, j] = data_by_class[perturbation][level][class_name][metric_name]
        
        # Créer le heatmap
        im = ax.imshow(matrix, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
        
        # Configuration
        ax.set_xticks(range(len(LEVELS)))
        ax.set_yticks(range(len(CLASSES)))
        ax.set_xticklabels([f'L{l}' for l in LEVELS])
        ax.set_yticklabels([c.upper() for c in CLASSES])
        ax.set_xlabel('Niveau', fontweight='bold')
        ax.set_ylabel('Classe', fontweight='bold')
        ax.set_title(perturbation.capitalize(), fontweight='bold')
        
        # Ajouter les valeurs
        for i in range(len(CLASSES)):
            for j in range(len(LEVELS)):
                text = ax.text(j, i, f'{matrix[i, j]:.2f}',
                             ha="center", va="center", 
                             color="black", fontsize=9, fontweight='bold')
    
    # Colorbar
    fig.colorbar(im, ax=axes, orientation='horizontal', 
                 pad=0.1, shrink=0.8, label=metric_name.upper())
    
    plt.tight_layout()
    filename = f'{save_directory}/heatmap_{metric_name}.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f" Heatmap sauvegardé : {filename}")
    plt.close()

# =========================
# TABLEAU RÉCAPITULATIF PAR CLASSE
# =========================
print("\n" + "="*100)
print("TABLEAU RÉCAPITULATIF PAR CLASSE")
print("="*100)

for class_name in CLASSES:
    print(f"\n{'='*100}")
    print(f"CLASSE: {class_name.upper()}")
    print(f"{'='*100}")
    
    summary_data = []
    for perturbation in PERTURBATIONS:
        for level in LEVELS:
            if class_name in data_by_class[perturbation][level]:
                metrics = data_by_class[perturbation][level][class_name]
                summary_data.append({
                    'Perturbation': perturbation.capitalize(),
                    'Level': level,
                    'Precision': f"{metrics['precision']:.3f}",
                    'Recall': f"{metrics['recall']:.3f}",
                    'mAP50': f"{metrics['mAP50']:.3f}",
                    'mAP50-95': f"{metrics['mAP50-95']:.3f}"
                })
    
    df = pd.DataFrame(summary_data)
    print(df.to_string(index=False))
    
    # Sauvegarder en CSV
    csv_filename = f'{save_directory}/{class_name}_summary.csv'
    df.to_csv(csv_filename, index=False)
    print(f"\n CSV sauvegardé : {csv_filename}")

print("\n" + "="*100)
print(" ANALYSE PAR CLASSE TERMINÉE")
print(f" Tous les fichiers sont dans : {save_directory}")
print("="*100)


GÉNÉRATION DES GRAPHIQUES PAR CLASSE

 Graphiques par perturbation...
 Graphique sauvegardé : ./Data/results/class_analysis/fog_by_class.png
 Graphique sauvegardé : ./Data/results/class_analysis/glare_by_class.png
 Graphique sauvegardé : ./Data/results/class_analysis/night_by_class.png
 Graphique sauvegardé : ./Data/results/class_analysis/rain_by_class.png
 Graphique sauvegardé : ./Data/results/class_analysis/snow_by_class.png

 Graphiques par classe...
 Graphique sauvegardé : ./Data/results/class_analysis/class0_all_perturbations.png
 Graphique sauvegardé : ./Data/results/class_analysis/class1_all_perturbations.png
 Graphique sauvegardé : ./Data/results/class_analysis/class2_all_perturbations.png
 Graphique sauvegardé : ./Data/results/class_analysis/class3_all_perturbations.png
 Graphique sauvegardé : ./Data/results/class_analysis/class4_all_perturbations.png

 Génération des heatmaps...


<tmp>\761778993.py:314: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


 Heatmap sauvegardé : ./Data/results/class_analysis/heatmap_precision.png


<tmp>\761778993.py:314: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


 Heatmap sauvegardé : ./Data/results/class_analysis/heatmap_recall.png


<tmp>\761778993.py:314: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


 Heatmap sauvegardé : ./Data/results/class_analysis/heatmap_mAP50.png


<tmp>\761778993.py:314: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


 Heatmap sauvegardé : ./Data/results/class_analysis/heatmap_mAP50-95.png

TABLEAU RÉCAPITULATIF PAR CLASSE

CLASSE: CLASS0
Perturbation  Level Precision Recall mAP50 mAP50-95
         Fog      1     0.993  0.998 0.995    0.912
         Fog      2     0.996  0.978 0.995    0.896
         Fog      3     0.998  0.854 0.987    0.846
         Fog      4     1.000  0.617 0.958    0.792
         Fog      5     1.000  0.544 0.928    0.760
       Glare      1     0.993  1.000 0.995    0.915

 CSV sauvegardé : ./Data/results/class_analysis/class0_summary.csv

CLASSE: CLASS1
Perturbation  Level Precision Recall mAP50 mAP50-95
         Fog      1     0.948  0.787 0.833    0.530
         Fog      2     0.923  0.799 0.823    0.502
         Fog      3     0.772  0.760 0.788    0.441
         Fog      4     0.803  0.640 0.706    0.360
         Fog      5     0.781  0.613 0.663    0.331
       Glare      1     0.928  0.787 0.837    0.550

 CSV sauvegardé : ./Data/results/class_analysis/class1_summary.c

# RCNN 

In [3]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))


True
NVIDIA GeForce GTX 1070 with Max-Q Design


# RCNN TRAIN 

In [4]:
pip install torch torchvision


Note: you may need to restart the kernel to use updated packages.


In [5]:
import torch
print("CUDA available =", torch.cuda.is_available())
print("CUDA device =", torch.cuda.get_device_name(0))
print("Number of GPU =", torch.cuda.device_count())


CUDA available = True
CUDA device = NVIDIA GeForce GTX 1070 with Max-Q Design
Number of GPU = 1


In [6]:
!pip install torch torchvision pycocotools tqdm


In [4]:
# Imports
import os
import torch
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms import functional as F
from tqdm import tqdm
from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd
from torchvision.ops import box_iou

class YoloDataset(Dataset):
    def __init__(self, img_dir, transform=None):
        self.img_dir = img_dir
        self.transform = transform
        self.images = [f for f in os.listdir(img_dir) if f.endswith((".jpg",".png",".jpeg"))]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        file = self.images[idx]
        img_path = os.path.join(self.img_dir, file)
        label_path = img_path.replace("images", "labels").replace(".jpg", ".txt").replace(".png",".txt")

        img = Image.open(img_path).convert("RGB")
        w, h = img.size

        boxes = []
        labels = []

        if os.path.exists(label_path):
            with open(label_path, "r") as f:
                for line in f.readlines():
                    cls, x, y, bw, bh = map(float, line.split())
                    xmin = (x - bw/2) * w
                    ymin = (y - bh/2) * h
                    xmax = (x + bw/2) * w
                    ymax = (y + bh/2) * h
                    boxes.append([xmin, ymin, xmax, ymax])
                    labels.append(int(cls))

        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.int64)

        target = {"boxes": boxes, "labels": labels}
        img = F.to_tensor(img)
        return img, target

train_dir = r"./Data1/train/images"
test_dir  = r"./Data1/test/images"

train_dataset = YoloDataset(train_dir)
test_dataset  = YoloDataset(test_dir)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))
test_loader  = DataLoader(test_dataset, batch_size=4, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))


nc = 5  # nombre de classes
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", device)

model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, nc)
model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)

def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0
    for imgs, targets in tqdm(loader):
        imgs = [img.to(device) for img in imgs]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(imgs, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        total_loss += losses.item()
    return total_loss / len(loader)

def evaluate(model, loader, iou_threshold=0.5):
    model.eval()
    all_matched_gt_labels = []
    all_matched_pred_labels = []
    all_map50, all_map5095 = [], []

    with torch.no_grad():
        for imgs, targets in tqdm(loader):
            imgs = [img.to(device) for img in imgs]
            preds = model(imgs)

            for pred, target in zip(preds, targets):
                gt_boxes = target["boxes"].to(device)
                gt_labels = target["labels"].to(device)
                pred_boxes = pred["boxes"]
                pred_labels = pred["labels"]
                pred_scores = pred["scores"]

                if len(gt_boxes) == 0 or len(pred_boxes) == 0:
                    continue

                # Filtrer les prédictions avec un score >= 0.5
                score_mask = pred_scores >= 0.5
                pred_boxes = pred_boxes[score_mask]
                pred_labels = pred_labels[score_mask]

                if len(pred_boxes) == 0:
                    continue

                # Calculer IoU entre toutes les gt_boxes et pred_boxes
                iou = box_iou(gt_boxes, pred_boxes)
                
                # Pour chaque gt_box, trouver la meilleure prédiction
                max_iou_per_gt, best_pred_idx = iou.max(dim=1)
                
                # Calcul mAP50 et mAP50-95
                all_map50.append((max_iou_per_gt >= 0.5).float().mean().item())
                all_map5095.append(max_iou_per_gt.mean().item())

                # Matcher les labels pour precision/recall/f1
                # On ne garde que les matchs avec IoU >= iou_threshold
                matched_mask = max_iou_per_gt >= iou_threshold
                
                if matched_mask.sum() > 0:
                    matched_gt_labels = gt_labels[matched_mask]
                    matched_pred_labels = pred_labels[best_pred_idx[matched_mask]]
                    
                    all_matched_gt_labels.extend(matched_gt_labels.cpu().numpy().tolist())
                    all_matched_pred_labels.extend(matched_pred_labels.cpu().numpy().tolist())

    # Calcul des métriques de classification
    if len(all_matched_gt_labels) > 0 and len(all_matched_pred_labels) > 0:
        precision = precision_score(all_matched_gt_labels, all_matched_pred_labels, 
                                    average="macro", zero_division=0)
        recall = recall_score(all_matched_gt_labels, all_matched_pred_labels, 
                             average="macro", zero_division=0)
        f1 = f1_score(all_matched_gt_labels, all_matched_pred_labels, 
                     average="macro", zero_division=0)
    else:
        precision, recall, f1 = 0, 0, 0

    return {
        "Box-P": precision,
        "Box-R": recall,
        "Box-F1": f1,
        "mAP50": np.mean(all_map50) if all_map50 else 0,
        "mAP50-95": np.mean(all_map5095) if all_map5095 else 0
    }

epochs = 5
all_train_metrics = []
all_test_metrics = []

for epoch in range(epochs):
    print(f"----- Epoch {epoch+1}/{epochs} -----")
    loss = train_one_epoch(model, train_loader, optimizer)
    print("Train Loss:", loss)

    train_metrics = evaluate(model, train_loader)
    test_metrics = evaluate(model, test_loader)

    print("Train Metrics:", train_metrics)
    print("Test Metrics:", test_metrics)

    all_train_metrics.append(train_metrics)
    all_test_metrics.append(test_metrics)

# Sauvegarde CSV final
df_train = pd.DataFrame(all_train_metrics)
df_test  = pd.DataFrame(all_test_metrics)
df_train.to_csv("./RCNN_train_metrics.csv", index=False)
df_test.to_csv("./RCNN_test_metrics.csv", index=False)

print("\n Entraînement terminé! Les métriques ont été sauvegardées.")

DEVICE: cuda
----- Epoch 1/5 -----


100%|████████████████████████████████████████████████████████████████████████████████| 170/170 [02:55<00:00,  1.03s/it]


Train Loss: 0.36361990404479644


100%|██████████████████████████████████████████████████████████████████████████████████| 25/25 [00:19<00:00,  1.30it/s]


Train Metrics: {'Box-P': 0.5981265018215642, 'Box-R': 0.5997054491899853, 'Box-F1': 0.5989128468121782, 'mAP50': np.float64(0.5443694322993167), 'mAP50-95': np.float64(0.47184512617845126)}
Test Metrics: {'Box-P': 0.7383720930232558, 'Box-R': 0.75, 'Box-F1': 0.7440476190476191, 'mAP50': np.float64(0.52272728687585), 'mAP50-95': np.float64(0.4453635090077766)}
----- Epoch 2/5 -----


...

Train Metrics: {'Box-P': 0.992602501772201, 'Box-R': 0.9866271768027728, 'Box-F1': 0.9895840485745795, 'mAP50': np.float64(0.6216712559695786), 'mAP50-95': np.float64(0.5503559665234438)}
Test Metrics: {'Box-P': 0.9310905396251976, 'Box-R': 0.8857323232323232, 'Box-F1': 0.9066129273025825, 'mAP50': np.float64(0.5366666805744171), 'mAP50-95': np.float64(0.4887920132279396)}
----- Epoch 5/5 -----


100%|████████████████████████████████████████████████████████████████████████████████| 170/170 [05:04<00:00,  1.79s/it]


Train Loss: 0.15958770872915493


100%|██████████████████████████████████████████████████████████████████████████████████| 25/25 [00:24<00:00,  1.01it/s]

Train Metrics: {'Box-P': 0.9963048582922257, 'Box-R': 0.9870971935678432, 'Box-F1': 0.9916066488707493, 'mAP50': np.float64(0.6099528902742676), 'mAP50-95': np.float64(0.533013455448686)}
Test Metrics: {'Box-P': 0.9834215979737508, 'Box-R': 0.9169435215946844, 'Box-F1': 0.9442054263565891, 'mAP50': np.float64(0.5420875561357749), 'mAP50-95': np.float64(0.49117784638597506)}

 Entraînement terminé! Les métriques ont été sauvegardées.


# Yolo train test tableaux 

In [12]:
import pandas as pd
import numpy as np

# =========================
# FONCTION POUR DÉTERMINER LA QUALITÉ
# =========================
def get_quality(value):
    """Détermine la qualité basée sur la valeur"""
    if value >= 0.95:
        return "Excellent"
    elif value >= 0.85:
        return "Très Bon"
    elif value >= 0.75:
        return "Bon"
    elif value >= 0.60:
        return "Moyen"
    else:
        return "Faible"

def format_with_delta(test_value, train_value):
    """Formate une valeur avec son delta par rapport au train"""
    delta = test_value - train_value
    if abs(delta) < 0.001:
        return f"{test_value:.3f}"
    elif delta > 0:
        return f"{test_value:.3f}(+{delta:.3f})"
    else:
        return f"{test_value:.3f}({delta:.3f})"

# =========================
# DONNÉES FASTER R-CNN PAR CLASSE (MEILLEUR EPOCH)
# =========================

# Résultats Train (Epoch 5 - meilleur)
class_results_train = {
    'Classe 0': {'P': 0.993, 'R': 1.000, 'mAP50': 0.995},
    'Classe 1': {'P': 0.919, 'R': 0.729, 'mAP50': 0.781},
    'Classe 2': {'P': 0.898, 'R': 0.965, 'mAP50': 0.983},
    'Classe 3': {'P': 0.887, 'R': 0.774, 'mAP50': 0.849},
    'Classe 4': {'P': 0.997, 'R': 0.998, 'mAP50': 0.995}
}

# Résultats Test (Epoch 5 - meilleur)
class_results_test = {
    'Classe 0': {'P': 0.991, 'R': 1.000, 'mAP50': 0.995},
    'Classe 1': {'P': 0.773, 'R': 0.500, 'mAP50': 0.621},
    'Classe 2': {'P': 0.888, 'R': 1.000, 'mAP50': 0.994},
    'Classe 3': {'P': 1.000, 'R': 0.917, 'mAP50': 0.986},
    'Classe 4': {'P': 0.991, 'R': 1.000, 'mAP50': 0.995}
}

# =========================
# TABLEAU TRAIN
# =========================
print("\n" + "="*100)
print(" FASTER R-CNN - RÉSULTATS PAR CLASSE (TRAIN)")
print("="*100)

train_data = []
for class_name in sorted(class_results_train.keys()):
    metrics = class_results_train[class_name]
    train_data.append({
        'Classe': class_name,
        'P (Train)': f"{metrics['P']:.3f}",
        'R (Train)': f"{metrics['R']:.3f}",
        'mAP50 (Train)': f"{metrics['mAP50']:.3f}",
        'Qualité': get_quality(metrics['mAP50'])
    })

# Calculer les moyennes
avg_p_train = np.mean([m['P'] for m in class_results_train.values()])
avg_r_train = np.mean([m['R'] for m in class_results_train.values()])
avg_map_train = np.mean([m['mAP50'] for m in class_results_train.values()])

train_data.append({
    'Classe': 'All',
    'P (Train)': f"{avg_p_train:.3f}",
    'R (Train)': f"{avg_r_train:.3f}",
    'mAP50 (Train)': f"{avg_map_train:.3f}",
    'Qualité': get_quality(avg_map_train)
})

df_train = pd.DataFrame(train_data)
print(df_train.to_string(index=False))

# =========================
# TABLEAU TEST AVEC DELTAS
# =========================
print("\n" + "="*100)
print(" FASTER R-CNN - RÉSULTATS PAR CLASSE (TEST) - Avec Variations vs Train")
print("="*100)

test_data_with_delta = []
for class_name in sorted(class_results_test.keys()):
    test_metrics = class_results_test[class_name]
    train_metrics = class_results_train[class_name]
    
    test_data_with_delta.append({
        'Classe': class_name,
        'P (Test)': format_with_delta(test_metrics['P'], train_metrics['P']),
        'R (Test)': format_with_delta(test_metrics['R'], train_metrics['R']),
        'mAP50 (Test)': format_with_delta(test_metrics['mAP50'], train_metrics['mAP50']),
        'Qualité': get_quality(test_metrics['mAP50'])
    })

# Calculer les moyennes test
avg_p_test = np.mean([m['P'] for m in class_results_test.values()])
avg_r_test = np.mean([m['R'] for m in class_results_test.values()])
avg_map_test = np.mean([m['mAP50'] for m in class_results_test.values()])

test_data_with_delta.append({
    'Classe': 'All',
    'P (Test)': format_with_delta(avg_p_test, avg_p_train),
    'R (Test)': format_with_delta(avg_r_test, avg_r_train),
    'mAP50 (Test)': format_with_delta(avg_map_test, avg_map_train),
    'Qualité': get_quality(avg_map_test)
})

df_test_delta = pd.DataFrame(test_data_with_delta)
print(df_test_delta.to_string(index=False))

# =========================
# TABLEAU DÉTAILLÉ : TRAIN | TEST | DELTA
# =========================
print("\n" + "="*100)
print(" FASTER R-CNN - COMPARAISON DÉTAILLÉE PAR CLASSE")
print("="*100)

detailed_data = []
for class_name in sorted(class_results_test.keys()):
    test_metrics = class_results_test[class_name]
    train_metrics = class_results_train[class_name]
    
    detailed_data.append({
        'Classe': class_name,
        'P (Train)': f"{train_metrics['P']:.3f}",
        'P (Test)': f"{test_metrics['P']:.3f}",
        'Δ P': f"{test_metrics['P'] - train_metrics['P']:+.3f}",
        'R (Train)': f"{train_metrics['R']:.3f}",
        'R (Test)': f"{test_metrics['R']:.3f}",
        'Δ R': f"{test_metrics['R'] - train_metrics['R']:+.3f}",
        'mAP50 (Train)': f"{train_metrics['mAP50']:.3f}",
        'mAP50 (Test)': f"{test_metrics['mAP50']:.3f}",
        'Δ mAP50': f"{test_metrics['mAP50'] - train_metrics['mAP50']:+.3f}",
        'Qualité': get_quality(test_metrics['mAP50'])
    })

# Ajouter All
detailed_data.append({
    'Classe': 'All',
    'P (Train)': f"{avg_p_train:.3f}",
    'P (Test)': f"{avg_p_test:.3f}",
    'Δ P': f"{avg_p_test - avg_p_train:+.3f}",
    'R (Train)': f"{avg_r_train:.3f}",
    'R (Test)': f"{avg_r_test:.3f}",
    'Δ R': f"{avg_r_test - avg_r_train:+.3f}",
    'mAP50 (Train)': f"{avg_map_train:.3f}",
    'mAP50 (Test)': f"{avg_map_test:.3f}",
    'Δ mAP50': f"{avg_map_test - avg_map_train:+.3f}",
    'Qualité': get_quality(avg_map_test)
})

df_detailed = pd.DataFrame(detailed_data)
print(df_detailed.to_string(index=False))

# =========================
# ANALYSE DES VARIATIONS
# =========================
print("\n" + "="*100)
print(" ANALYSE DES VARIATIONS (TEST vs TRAIN)")
print("="*100)

analysis_data = []
for class_name in sorted(class_results_test.keys()):
    test_metrics = class_results_test[class_name]
    train_metrics = class_results_train[class_name]
    
    delta_p = test_metrics['P'] - train_metrics['P']
    delta_r = test_metrics['R'] - train_metrics['R']
    delta_map = test_metrics['mAP50'] - train_metrics['mAP50']
    
    # Déterminer la tendance
    if delta_map >= 0.05:
        trend = " Amélioration"
    elif delta_map <= -0.05:
        trend = " Dégradation"
    else:
        trend = " Stable"
    
    analysis_data.append({
        'Classe': class_name,
        'Δ Precision': f"{delta_p:+.3f}",
        'Δ Recall': f"{delta_r:+.3f}",
        'Δ mAP50': f"{delta_map:+.3f}",
        'Tendance': trend
    })

df_analysis = pd.DataFrame(analysis_data)
print(df_analysis.to_string(index=False))

# =========================
# SAUVEGARDER LES TABLEAUX
# =========================
save_dir = "./models_rcnn"

df_train.to_csv(f"{save_dir}/rcnn_class_train.csv", index=False)
df_test_delta.to_csv(f"{save_dir}/rcnn_class_test_with_delta.csv", index=False)
df_detailed.to_csv(f"{save_dir}/rcnn_class_comparison_detailed.csv", index=False)
df_analysis.to_csv(f"{save_dir}/rcnn_class_analysis.csv", index=False)

print("\n" + "="*100)
print(" TABLEAUX SAUVEGARDÉS")
print(f" Dossier: {save_dir}")
print("   - rcnn_class_train.csv")
print("   - rcnn_class_test_with_delta.csv")
print("   - rcnn_class_comparison_detailed.csv")
print("   - rcnn_class_analysis.csv")
print("="*100)

# =========================
# GRAPHIQUE COMPARATIF PAR CLASSE
# =========================
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Faster R-CNN - Comparaison Train vs Test par Classe', fontsize=16, fontweight='bold')

classes = list(class_results_train.keys())
x = np.arange(len(classes))
width = 0.35

metrics_to_plot = [
    ('P', 'Precision', axes[0]),
    ('R', 'Recall', axes[1]),
    ('mAP50', 'mAP@50', axes[2])
]

for metric_key, metric_name, ax in metrics_to_plot:
    train_values = [class_results_train[c][metric_key] for c in classes]
    test_values = [class_results_test[c][metric_key] for c in classes]
    
    bars1 = ax.bar(x - width/2, train_values, width, label='Train', color='#2ca02c', alpha=0.8)
    bars2 = ax.bar(x + width/2, test_values, width, label='Test', color='#d62728', alpha=0.8)
    
    ax.set_xlabel('Classe', fontsize=12, fontweight='bold')
    ax.set_ylabel(metric_name, fontsize=12, fontweight='bold')
    ax.set_title(metric_name, fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(classes)
    ax.legend(fontsize=11)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.set_ylim([0, 1.1])
    
    # Ajouter les valeurs sur les barres
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.2f}',
                   ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(f"{save_dir}/rcnn_class_comparison.png", dpi=300, bbox_inches='tight')
print(f" Graphique sauvegardé : rcnn_class_comparison.png")
plt.show()

print("\n" + "="*100)


 FASTER R-CNN - RÉSULTATS PAR CLASSE (TRAIN)
  Classe P (Train) R (Train) mAP50 (Train)   Qualité
Classe 0     0.993     1.000         0.995 Excellent
Classe 1     0.919     0.729         0.781       Bon
Classe 2     0.898     0.965         0.983 Excellent
Classe 3     0.887     0.774         0.849       Bon
Classe 4     0.997     0.998         0.995 Excellent
     All     0.939     0.893         0.921  Très Bon

 FASTER R-CNN - RÉSULTATS PAR CLASSE (TEST) - Avec Variations vs Train
  Classe      P (Test)      R (Test)  mAP50 (Test)   Qualité
Classe 0 0.991(-0.002)         1.000         0.995 Excellent
Classe 1 0.773(-0.146) 0.500(-0.229) 0.621(-0.160)     Moyen
Classe 2 0.888(-0.010) 1.000(+0.035) 0.994(+0.011) Excellent
Classe 3 1.000(+0.113) 0.917(+0.143) 0.986(+0.137) Excellent
Classe 4 0.991(-0.006) 1.000(+0.002)         0.995 Excellent
     All 0.929(-0.010) 0.883(-0.010) 0.918(-0.002)  Très Bon

 FASTER R-CNN - COMPARAISON DÉTAILLÉE PAR CLASSE
  Classe P (Train) P (Test)    Δ P

<Figure size 1800x600 with 3 Axes>

# Fog_level1 a 5 RCNN 

In [6]:
# Imports
import os
import torch
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms import functional as F
from tqdm import tqdm
from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd
from torchvision.ops import box_iou

# Dataset adapté pour YOLO labels
class YoloDataset(Dataset):
    def __init__(self, img_dir, labels_dir, transform=None):
        self.img_dir = img_dir
        self.labels_dir = labels_dir
        self.transform = transform
        self.images = [f for f in os.listdir(img_dir) if f.endswith((".jpg",".png",".jpeg"))]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        file = self.images[idx]
        img_path = os.path.join(self.img_dir, file)
        label_path = os.path.join(self.labels_dir, file.replace(".jpg", ".txt").replace(".png", ".txt"))

        img = Image.open(img_path).convert("RGB")
        w, h = img.size

        boxes, labels = [], []

        if os.path.exists(label_path):
            with open(label_path, "r") as f:
                for line in f.readlines():
                    cls, x, y, bw, bh = map(float, line.split())
                    xmin = (x - bw/2) * w
                    ymin = (y - bh/2) * h
                    xmax = (x + bw/2) * w
                    ymax = (y + bh/2) * h
                    boxes.append([xmin, ymin, xmax, ymax])
                    labels.append(int(cls))

        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.int64)

        target = {"boxes": boxes, "labels": labels}
        img = F.to_tensor(img)
        return img, target

# Evaluation avec métriques
def evaluate(model, loader, iou_threshold=0.5):
    model.eval()
    all_matched_gt_labels = []
    all_matched_pred_labels = []
    all_map50, all_map5095 = [], []

    with torch.no_grad():
        for imgs, targets in tqdm(loader, leave=False):
            imgs = [img.to(device) for img in imgs]
            preds = model(imgs)

            for pred, target in zip(preds, targets):
                gt_boxes = target["boxes"].to(device)
                gt_labels = target["labels"].to(device)
                pred_boxes = pred["boxes"]
                pred_labels = pred["labels"]
                pred_scores = pred["scores"]

                if len(gt_boxes) == 0 or len(pred_boxes) == 0:
                    continue

                # Filtrer les prédictions par score >= 0.5
                score_mask = pred_scores >= 0.5
                pred_boxes = pred_boxes[score_mask]
                pred_labels = pred_labels[score_mask]

                if len(pred_boxes) == 0:
                    continue

                # Calcul IoU
                iou = box_iou(gt_boxes, pred_boxes)
                max_iou_per_gt, best_pred_idx = iou.max(dim=1)

                # mAP50 et mAP50-95
                all_map50.append((max_iou_per_gt >= 0.5).float().mean().item())
                all_map5095.append(max_iou_per_gt.mean().item())

                # Matcher labels pour precision/recall/f1
                matched_mask = max_iou_per_gt >= iou_threshold
                if matched_mask.sum() > 0:
                    matched_gt_labels = gt_labels[matched_mask]
                    matched_pred_labels = pred_labels[best_pred_idx[matched_mask]]
                    all_matched_gt_labels.extend(matched_gt_labels.cpu().numpy().tolist())
                    all_matched_pred_labels.extend(matched_pred_labels.cpu().numpy().tolist())

    if len(all_matched_gt_labels) > 0:
        precision = precision_score(all_matched_gt_labels, all_matched_pred_labels, average="macro", zero_division=0)
        recall = recall_score(all_matched_gt_labels, all_matched_pred_labels, average="macro", zero_division=0)
        f1 = f1_score(all_matched_gt_labels, all_matched_pred_labels, average="macro", zero_division=0)
    else:
        precision, recall, f1 = 0, 0, 0

    return {
        "Box-P": precision,
        "Box-R": recall,
        "Box-F1": f1,
        "mAP50": np.mean(all_map50) if all_map50 else 0,
        "mAP50-95": np.mean(all_map5095) if all_map5095 else 0
    }

# Device GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", device)

# Paramètres
nc = 5  # nombre de classes
epochs = 5
batch_size = 4
fog_levels = [1,2,3,4,5]

# Boucle sur les niveaux de fog
for level in fog_levels:
    print(f"\n=== Training and evaluating fog_level{level} ===\n")
    
    img_dir = f"./Data1/results/fog_level{level}/images"
    labels_dir = f"./Data1/results/fog_level{level}/labels_yolo"

    dataset = YoloDataset(img_dir, labels_dir)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))
    
    # Charger modèle RCNN
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, nc)
    model.to(device)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)
    
    all_metrics = []

    # Entraînement
    for epoch in range(epochs):
        print(f"Epoch {epoch+1}/{epochs} - Training fog_level{level}")
        model.train()
        total_loss = 0
        for imgs, targets in tqdm(loader, leave=False):
            imgs = [img.to(device) for img in imgs]
            targets = [{k: v.to(device) for k,v in t.items()} for t in targets]

            # Ignorer si aucune box
            if any([len(t['boxes'])==0 for t in targets]):
                continue

            loss_dict = model(imgs, targets)
            losses = sum(loss for loss in loss_dict.values())

            optimizer.zero_grad()
            losses.backward()
            optimizer.step()

            total_loss += losses.item()
        print("Train Loss:", total_loss / len(loader))
    
    # Évaluation
    metrics = evaluate(model, loader)
    print(f"Metrics fog_level{level}:", metrics)
    
    # Sauvegarde CSV
    df = pd.DataFrame([metrics])
    df.to_csv(f"./Data1/results/fog_level{level}/RCNN_metrics.csv", index=False)

print("\n Entraînement et évaluation pour tous les niveaux de fog terminés !")


DEVICE: cuda

=== Training and evaluating fog_level1 ===

Epoch 1/5 - Training fog_level1


Train Loss: 0.4012378125190735
Epoch 2/5 - Training fog_level1


Train Loss: 0.25721079754829407
Epoch 3/5 - Training fog_level1


Train Loss: 0.20118069499731064
Epoch 4/5 - Training fog_level1


Train Loss: 0.18202959042787553
Epoch 5/5 - Training fog_level1


...

Train Loss: 0.2524673546552658
Epoch 3/5 - Training fog_level5


Train Loss: 0.20303187465667724
Epoch 4/5 - Training fog_level5


Train Loss: 0.1864630776643753
Epoch 5/5 - Training fog_level5


Train Loss: 0.18969214475154877


Metrics fog_level5: {'Box-P': 0.7981856532208388, 'Box-R': 0.7963891971883094, 'Box-F1': 0.7972782143474845, 'mAP50': np.float64(0.6102095405459405), 'mAP50-95': np.float64(0.5285415291190148)}

 Entraînement et évaluation pour tous les niveaux de fog terminés !


# Glare_level1 a 5 RCNN

In [7]:
# Imports
import os
import torch
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms import functional as F
from tqdm import tqdm
from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd
from torchvision.ops import box_iou

# Dataset YOLO
class YoloDataset(Dataset):
    def __init__(self, img_dir, labels_dir, transform=None):
        self.img_dir = img_dir
        self.labels_dir = labels_dir
        self.transform = transform
        self.images = [f for f in os.listdir(img_dir) if f.endswith((".jpg",".png",".jpeg"))]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        file = self.images[idx]
        img_path = os.path.join(self.img_dir, file)
        label_path = os.path.join(self.labels_dir, file.replace(".jpg", ".txt").replace(".png", ".txt"))

        img = Image.open(img_path).convert("RGB")
        w, h = img.size

        boxes, labels = [], []

        if os.path.exists(label_path):
            with open(label_path, "r") as f:
                for line in f.readlines():
                    cls, x, y, bw, bh = map(float, line.split())
                    xmin = (x - bw/2) * w
                    ymin = (y - bh/2) * h
                    xmax = (x + bw/2) * w
                    ymax = (y + bh/2) * h
                    boxes.append([xmin, ymin, xmax, ymax])
                    labels.append(int(cls))

        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.int64)

        target = {"boxes": boxes, "labels": labels}
        img = F.to_tensor(img)
        return img, target

# Évaluation
def evaluate(model, loader, iou_threshold=0.5):
    model.eval()
    all_matched_gt_labels = []
    all_matched_pred_labels = []
    all_map50, all_map5095 = [], []

    with torch.no_grad():
        for imgs, targets in tqdm(loader, leave=False):
            imgs = [img.to(device) for img in imgs]
            preds = model(imgs)

            for pred, target in zip(preds, targets):
                gt_boxes = target["boxes"].to(device)
                gt_labels = target["labels"].to(device)
                pred_boxes = pred["boxes"]
                pred_labels = pred["labels"]
                pred_scores = pred["scores"]

                if len(gt_boxes) == 0 or len(pred_boxes) == 0:
                    continue

                score_mask = pred_scores >= 0.5
                pred_boxes = pred_boxes[score_mask]
                pred_labels = pred_labels[score_mask]

                if len(pred_boxes) == 0:
                    continue

                iou = box_iou(gt_boxes, pred_boxes)
                max_iou_per_gt, best_pred_idx = iou.max(dim=1)

                all_map50.append((max_iou_per_gt >= 0.5).float().mean().item())
                all_map5095.append(max_iou_per_gt.mean().item())

                matched_mask = max_iou_per_gt >= iou_threshold
                if matched_mask.sum() > 0:
                    matched_gt_labels = gt_labels[matched_mask]
                    matched_pred_labels = pred_labels[best_pred_idx[matched_mask]]
                    all_matched_gt_labels.extend(matched_gt_labels.cpu().numpy().tolist())
                    all_matched_pred_labels.extend(matched_pred_labels.cpu().numpy().tolist())

    if len(all_matched_gt_labels) > 0:
        precision = precision_score(all_matched_gt_labels, all_matched_pred_labels, average="macro", zero_division=0)
        recall = recall_score(all_matched_gt_labels, all_matched_pred_labels, average="macro", zero_division=0)
        f1 = f1_score(all_matched_gt_labels, all_matched_pred_labels, average="macro", zero_division=0)
    else:
        precision, recall, f1 = 0, 0, 0

    return {
        "Box-P": precision,
        "Box-R": recall,
        "Box-F1": f1,
        "mAP50": np.mean(all_map50) if all_map50 else 0,
        "mAP50-95": np.mean(all_map5095) if all_map5095 else 0
    }

# Device GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", device)

# Paramètres
nc = 5  # nombre de classes
epochs = 5
batch_size = 4
glare_levels = [1,2,3,4,5]

# Boucle sur les niveaux de glare
for level in glare_levels:
    print(f"\n=== Training and evaluating glare_level{level} ===\n")
    
    img_dir = f"./Data1/results/glare_level{level}/images"
    labels_dir = f"./Data1/results/glare_level{level}/labels_yolo"

    dataset = YoloDataset(img_dir, labels_dir)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))
    
    # Charger modèle RCNN
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, nc)
    model.to(device)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)
    
    all_metrics = []

    # Entraînement
    for epoch in range(epochs):
        print(f"Epoch {epoch+1}/{epochs} - Training glare_level{level}")
        model.train()
        total_loss = 0
        for imgs, targets in tqdm(loader, leave=False):
            imgs = [img.to(device) for img in imgs]
            targets = [{k: v.to(device) for k,v in t.items()} for t in targets]

            # Ignorer si aucune box
            if any([len(t['boxes'])==0 for t in targets]):
                continue

            loss_dict = model(imgs, targets)
            losses = sum(loss for loss in loss_dict.values())

            optimizer.zero_grad()
            losses.backward()
            optimizer.step()

            total_loss += losses.item()
        print("Train Loss:", total_loss / len(loader))
    
    # Évaluation
    metrics = evaluate(model, loader)
    print(f"Metrics glare_level{level}:", metrics)
    
    # Sauvegarde CSV
    df = pd.DataFrame([metrics])
    df.to_csv(f"./Data1/results/glare_level{level}/RCNN_metrics.csv", index=False)

print("\n Entraînement et évaluation pour tous les niveaux de glare terminés !")


DEVICE: cuda

=== Training and evaluating glare_level1 ===

Epoch 1/5 - Training glare_level1


Train Loss: 0.3993015214204788
Epoch 2/5 - Training glare_level1


Train Loss: 0.21660871946811677
Epoch 3/5 - Training glare_level1


Train Loss: 0.18911255615949632
Epoch 4/5 - Training glare_level1


Train Loss: 0.17295955109596253
Epoch 5/5 - Training glare_level1


...

Train Loss: 0.2306016947031021
Epoch 3/5 - Training glare_level5


Train Loss: 0.21026985347270966
Epoch 4/5 - Training glare_level5


Train Loss: 0.19315709972381592
Epoch 5/5 - Training glare_level5


Train Loss: 0.1804869151711464


Metrics glare_level5: {'Box-P': 0.9979338842975207, 'Box-R': 0.9939024390243902, 'Box-F1': 0.9958762358485733, 'mAP50': np.float64(0.6239428744912148), 'mAP50-95': np.float64(0.5489461993873119)}

 Entraînement et évaluation pour tous les niveaux de glare terminés !


# Night_level 1 a 5 RCNN

In [8]:
# Imports
import os
import torch
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms import functional as F
from tqdm import tqdm
from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd
from torchvision.ops import box_iou

# Dataset YOLO
class YoloDataset(Dataset):
    def __init__(self, img_dir, labels_dir, transform=None):
        self.img_dir = img_dir
        self.labels_dir = labels_dir
        self.transform = transform
        self.images = [f for f in os.listdir(img_dir) if f.endswith((".jpg",".png",".jpeg"))]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        file = self.images[idx]
        img_path = os.path.join(self.img_dir, file)
        label_path = os.path.join(self.labels_dir, file.replace(".jpg", ".txt").replace(".png", ".txt"))

        img = Image.open(img_path).convert("RGB")
        w, h = img.size

        boxes, labels = [], []

        if os.path.exists(label_path):
            with open(label_path, "r") as f:
                for line in f.readlines():
                    cls, x, y, bw, bh = map(float, line.split())
                    xmin = (x - bw/2) * w
                    ymin = (y - bh/2) * h
                    xmax = (x + bw/2) * w
                    ymax = (y + bh/2) * h
                    boxes.append([xmin, ymin, xmax, ymax])
                    labels.append(int(cls))

        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.int64)

        target = {"boxes": boxes, "labels": labels}
        img = F.to_tensor(img)
        return img, target

# Évaluation
def evaluate(model, loader, iou_threshold=0.5):
    model.eval()
    all_matched_gt_labels = []
    all_matched_pred_labels = []
    all_map50, all_map5095 = [], []

    with torch.no_grad():
        for imgs, targets in tqdm(loader, leave=False):
            imgs = [img.to(device) for img in imgs]
            preds = model(imgs)

            for pred, target in zip(preds, targets):
                gt_boxes = target["boxes"].to(device)
                gt_labels = target["labels"].to(device)
                pred_boxes = pred["boxes"]
                pred_labels = pred["labels"]
                pred_scores = pred["scores"]

                if len(gt_boxes) == 0 or len(pred_boxes) == 0:
                    continue

                score_mask = pred_scores >= 0.5
                pred_boxes = pred_boxes[score_mask]
                pred_labels = pred_labels[score_mask]

                if len(pred_boxes) == 0:
                    continue

                iou = box_iou(gt_boxes, pred_boxes)
                max_iou_per_gt, best_pred_idx = iou.max(dim=1)

                all_map50.append((max_iou_per_gt >= 0.5).float().mean().item())
                all_map5095.append(max_iou_per_gt.mean().item())

                matched_mask = max_iou_per_gt >= iou_threshold
                if matched_mask.sum() > 0:
                    matched_gt_labels = gt_labels[matched_mask]
                    matched_pred_labels = pred_labels[best_pred_idx[matched_mask]]
                    all_matched_gt_labels.extend(matched_gt_labels.cpu().numpy().tolist())
                    all_matched_pred_labels.extend(matched_pred_labels.cpu().numpy().tolist())

    if len(all_matched_gt_labels) > 0:
        precision = precision_score(all_matched_gt_labels, all_matched_pred_labels, average="macro", zero_division=0)
        recall = recall_score(all_matched_gt_labels, all_matched_pred_labels, average="macro", zero_division=0)
        f1 = f1_score(all_matched_gt_labels, all_matched_pred_labels, average="macro", zero_division=0)
    else:
        precision, recall, f1 = 0, 0, 0

    return {
        "Box-P": precision,
        "Box-R": recall,
        "Box-F1": f1,
        "mAP50": np.mean(all_map50) if all_map50 else 0,
        "mAP50-95": np.mean(all_map5095) if all_map5095 else 0
    }

# Device GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", device)

# Paramètres
nc = 5  # nombre de classes
epochs = 5
batch_size = 4
night_levels = [1,2,3,4,5]

# Boucle sur les niveaux de night
for level in night_levels:
    print(f"\n=== Training and evaluating night_level{level} ===\n")
    
    img_dir = f"./Data1/results/night_level{level}/images"
    labels_dir = f"./Data1/results/night_level{level}/labels_yolo"

    dataset = YoloDataset(img_dir, labels_dir)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))
    
    # Charger modèle RCNN
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, nc)
    model.to(device)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)
    
    # Entraînement
    for epoch in range(epochs):
        print(f"Epoch {epoch+1}/{epochs} - Training night_level{level}")
        model.train()
        total_loss = 0
        for imgs, targets in tqdm(loader, leave=False):
            imgs = [img.to(device) for img in imgs]
            targets = [{k: v.to(device) for k,v in t.items()} for t in targets]

            # Ignorer si aucune box
            if any([len(t['boxes'])==0 for t in targets]):
                continue

            loss_dict = model(imgs, targets)
            losses = sum(loss for loss in loss_dict.values())

            optimizer.zero_grad()
            losses.backward()
            optimizer.step()

            total_loss += losses.item()
        print("Train Loss:", total_loss / len(loader))
    
    # Évaluation
    metrics = evaluate(model, loader)
    print(f"Metrics night_level{level}:", metrics)
    
    # Sauvegarde CSV
    df = pd.DataFrame([metrics])
    df.to_csv(f"./Data1/results/night_level{level}/RCNN_metrics.csv", index=False)

print("\n Entraînement et évaluation pour tous les niveaux de night terminés !")


DEVICE: cuda

=== Training and evaluating night_level1 ===

Epoch 1/5 - Training night_level1


Train Loss: 0.3857389898300171
Epoch 2/5 - Training night_level1


Train Loss: 0.22904340839385987
Epoch 3/5 - Training night_level1


Train Loss: 0.20469602417945862
Epoch 4/5 - Training night_level1


Train Loss: 0.18187534493207932
Epoch 5/5 - Training night_level1


...

Train Loss: 0.2803371514081955
Epoch 3/5 - Training night_level5


Train Loss: 0.23468715113401412
Epoch 4/5 - Training night_level5


Train Loss: 0.21093188667297363
Epoch 5/5 - Training night_level5


Train Loss: 0.18997198456525802


Metrics night_level5: {'Box-P': 0.9995039682539683, 'Box-R': 0.9991554054054054, 'Box-F1': 0.9993287259908223, 'mAP50': np.float64(0.6006428738832473), 'mAP50-95': np.float64(0.5263762805163861)}

 Entraînement et évaluation pour tous les niveaux de night terminés !


# Rain_level1 a 5 RCNN 

In [9]:
# Imports
import os
import torch
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms import functional as F
from tqdm import tqdm
from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd
from torchvision.ops import box_iou

# Dataset YOLO
class YoloDataset(Dataset):
    def __init__(self, img_dir, labels_dir, transform=None):
        self.img_dir = img_dir
        self.labels_dir = labels_dir
        self.transform = transform
        self.images = [f for f in os.listdir(img_dir) if f.endswith((".jpg",".png",".jpeg"))]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        file = self.images[idx]
        img_path = os.path.join(self.img_dir, file)
        label_path = os.path.join(self.labels_dir, file.replace(".jpg", ".txt").replace(".png", ".txt"))

        img = Image.open(img_path).convert("RGB")
        w, h = img.size

        boxes, labels = [], []

        if os.path.exists(label_path):
            with open(label_path, "r") as f:
                for line in f.readlines():
                    cls, x, y, bw, bh = map(float, line.split())
                    xmin = (x - bw/2) * w
                    ymin = (y - bh/2) * h
                    xmax = (x + bw/2) * w
                    ymax = (y + bh/2) * h
                    boxes.append([xmin, ymin, xmax, ymax])
                    labels.append(int(cls))

        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.int64)

        target = {"boxes": boxes, "labels": labels}
        img = F.to_tensor(img)
        return img, target

# Évaluation
def evaluate(model, loader, iou_threshold=0.5):
    model.eval()
    all_matched_gt_labels = []
    all_matched_pred_labels = []
    all_map50, all_map5095 = [], []

    with torch.no_grad():
        for imgs, targets in tqdm(loader, leave=False):
            imgs = [img.to(device) for img in imgs]
            preds = model(imgs)

            for pred, target in zip(preds, targets):
                gt_boxes = target["boxes"].to(device)
                gt_labels = target["labels"].to(device)
                pred_boxes = pred["boxes"]
                pred_labels = pred["labels"]
                pred_scores = pred["scores"]

                if len(gt_boxes) == 0 or len(pred_boxes) == 0:
                    continue

                score_mask = pred_scores >= 0.5
                pred_boxes = pred_boxes[score_mask]
                pred_labels = pred_labels[score_mask]

                if len(pred_boxes) == 0:
                    continue

                iou = box_iou(gt_boxes, pred_boxes)
                max_iou_per_gt, best_pred_idx = iou.max(dim=1)

                all_map50.append((max_iou_per_gt >= 0.5).float().mean().item())
                all_map5095.append(max_iou_per_gt.mean().item())

                matched_mask = max_iou_per_gt >= iou_threshold
                if matched_mask.sum() > 0:
                    matched_gt_labels = gt_labels[matched_mask]
                    matched_pred_labels = pred_labels[best_pred_idx[matched_mask]]
                    all_matched_gt_labels.extend(matched_gt_labels.cpu().numpy().tolist())
                    all_matched_pred_labels.extend(matched_pred_labels.cpu().numpy().tolist())

    if len(all_matched_gt_labels) > 0:
        precision = precision_score(all_matched_gt_labels, all_matched_pred_labels, average="macro", zero_division=0)
        recall = recall_score(all_matched_gt_labels, all_matched_pred_labels, average="macro", zero_division=0)
        f1 = f1_score(all_matched_gt_labels, all_matched_pred_labels, average="macro", zero_division=0)
    else:
        precision, recall, f1 = 0, 0, 0

    return {
        "Box-P": precision,
        "Box-R": recall,
        "Box-F1": f1,
        "mAP50": np.mean(all_map50) if all_map50 else 0,
        "mAP50-95": np.mean(all_map5095) if all_map5095 else 0
    }

# Device GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", device)

# Paramètres
nc = 5  # nombre de classes
epochs = 5
batch_size = 4
rain_levels = [1,2,3,4,5]

# Boucle sur les niveaux de rain
for level in rain_levels:
    print(f"\n=== Training and evaluating rain_level{level} ===\n")
    
    img_dir = f"./Data1/results/rain_level{level}/images"
    labels_dir = f"./Data1/results/rain_level{level}/labels_yolo"

    dataset = YoloDataset(img_dir, labels_dir)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))
    
    # Charger modèle RCNN
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, nc)
    model.to(device)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)
    
    # Entraînement
    for epoch in range(epochs):
        print(f"Epoch {epoch+1}/{epochs} - Training rain_level{level}")
        model.train()
        total_loss = 0
        for imgs, targets in tqdm(loader, leave=False):
            imgs = [img.to(device) for img in imgs]
            targets = [{k: v.to(device) for k,v in t.items()} for t in targets]

            if any([len(t['boxes'])==0 for t in targets]):
                continue

            loss_dict = model(imgs, targets)
            losses = sum(loss for loss in loss_dict.values())

            optimizer.zero_grad()
            losses.backward()
            optimizer.step()

            total_loss += losses.item()
        print("Train Loss:", total_loss / len(loader))
    
    # Évaluation
    metrics = evaluate(model, loader)
    print(f"Metrics rain_level{level}:", metrics)
    
    # Sauvegarde CSV
    df = pd.DataFrame([metrics])
    df.to_csv(f"./Data1/results/rain_level{level}/RCNN_metrics.csv", index=False)

print("\n Entraînement et évaluation pour tous les niveaux de rain terminés !")


DEVICE: cuda

=== Training and evaluating rain_level1 ===

Epoch 1/5 - Training rain_level1


Train Loss: 0.3949243296384811
Epoch 2/5 - Training rain_level1


Train Loss: 0.24347158193588256
Epoch 3/5 - Training rain_level1


Train Loss: 0.2145597783923149
Epoch 4/5 - Training rain_level1


Train Loss: 0.1807329915165901
Epoch 5/5 - Training rain_level1


...

Train Loss: 0.27637521231174467
Epoch 3/5 - Training rain_level5


Train Loss: 0.22692359846830368
Epoch 4/5 - Training rain_level5


Train Loss: 0.21306935316324235
Epoch 5/5 - Training rain_level5


Train Loss: 0.19863576740026473


Metrics rain_level5: {'Box-P': 0.7859623695288784, 'Box-R': 0.7969880066009174, 'Box-F1': 0.7913459776743579, 'mAP50': np.float64(0.5920762069225312), 'mAP50-95': np.float64(0.5068768773376942)}

 Entraînement et évaluation pour tous les niveaux de rain terminés !


# Snow_level1 a 5 RCNN 

In [1]:
# Imports
import os
import torch
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms import functional as F
from tqdm import tqdm
from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd
from torchvision.ops import box_iou

# Dataset YOLO
class YoloDataset(Dataset):
    def __init__(self, img_dir, labels_dir, transform=None):
        self.img_dir = img_dir
        self.labels_dir = labels_dir
        self.transform = transform
        self.images = [f for f in os.listdir(img_dir) if f.endswith((".jpg",".png",".jpeg"))]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        file = self.images[idx]
        img_path = os.path.join(self.img_dir, file)
        label_path = os.path.join(self.labels_dir, file.replace(".jpg", ".txt").replace(".png", ".txt"))

        img = Image.open(img_path).convert("RGB")
        w, h = img.size

        boxes, labels = [], []

        if os.path.exists(label_path):
            with open(label_path, "r") as f:
                for line in f.readlines():
                    cls, x, y, bw, bh = map(float, line.split())
                    xmin = (x - bw/2) * w
                    ymin = (y - bh/2) * h
                    xmax = (x + bw/2) * w
                    ymax = (y + bh/2) * h
                    boxes.append([xmin, ymin, xmax, ymax])
                    labels.append(int(cls))

        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.int64)

        target = {"boxes": boxes, "labels": labels}
        img = F.to_tensor(img)
        return img, target

# Évaluation
def evaluate(model, loader, iou_threshold=0.5):
    model.eval()
    all_matched_gt_labels = []
    all_matched_pred_labels = []
    all_map50, all_map5095 = [], []

    with torch.no_grad():
        for imgs, targets in tqdm(loader, leave=False):
            imgs = [img.to(device) for img in imgs]
            preds = model(imgs)

            for pred, target in zip(preds, targets):
                gt_boxes = target["boxes"].to(device)
                gt_labels = target["labels"].to(device)
                pred_boxes = pred["boxes"]
                pred_labels = pred["labels"]
                pred_scores = pred["scores"]

                if len(gt_boxes) == 0 or len(pred_boxes) == 0:
                    continue

                score_mask = pred_scores >= 0.5
                pred_boxes = pred_boxes[score_mask]
                pred_labels = pred_labels[score_mask]

                if len(pred_boxes) == 0:
                    continue

                iou = box_iou(gt_boxes, pred_boxes)
                max_iou_per_gt, best_pred_idx = iou.max(dim=1)

                all_map50.append((max_iou_per_gt >= 0.5).float().mean().item())
                all_map5095.append(max_iou_per_gt.mean().item())

                matched_mask = max_iou_per_gt >= iou_threshold
                if matched_mask.sum() > 0:
                    matched_gt_labels = gt_labels[matched_mask]
                    matched_pred_labels = pred_labels[best_pred_idx[matched_mask]]
                    all_matched_gt_labels.extend(matched_gt_labels.cpu().numpy().tolist())
                    all_matched_pred_labels.extend(matched_pred_labels.cpu().numpy().tolist())

    if len(all_matched_gt_labels) > 0:
        precision = precision_score(all_matched_gt_labels, all_matched_pred_labels, average="macro", zero_division=0)
        recall = recall_score(all_matched_gt_labels, all_matched_pred_labels, average="macro", zero_division=0)
        f1 = f1_score(all_matched_gt_labels, all_matched_pred_labels, average="macro", zero_division=0)
    else:
        precision, recall, f1 = 0, 0, 0

    return {
        "Box-P": precision,
        "Box-R": recall,
        "Box-F1": f1,
        "mAP50": np.mean(all_map50) if all_map50 else 0,
        "mAP50-95": np.mean(all_map5095) if all_map5095 else 0
    }

# Device GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", device)

# Paramètres
nc = 5  # nombre de classes
epochs = 5
batch_size = 4
snow_levels = [1,2,3,4,5]  # Changé de rain_levels à snow_levels

# Boucle sur les niveaux de snow
for level in snow_levels:  # Changé ici
    print(f"\n=== Training and evaluating snow_level{level} ===\n")  # Changé ici
    
    img_dir = f"./Data1/results/snow_level{level}/images"  # Changé ici
    labels_dir = f"./Data1/results/snow_level{level}/labels_yolo"  # Changé ici

    dataset = YoloDataset(img_dir, labels_dir)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))
    
    # Charger modèle RCNN
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, nc)
    model.to(device)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)
    
    # Entraînement
    for epoch in range(epochs):
        print(f"Epoch {epoch+1}/{epochs} - Training snow_level{level}")  # Changé ici
        model.train()
        total_loss = 0
        for imgs, targets in tqdm(loader, leave=False):
            imgs = [img.to(device) for img in imgs]
            targets = [{k: v.to(device) for k,v in t.items()} for t in targets]

            if any([len(t['boxes'])==0 for t in targets]):
                continue

            loss_dict = model(imgs, targets)
            losses = sum(loss for loss in loss_dict.values())

            optimizer.zero_grad()
            losses.backward()
            optimizer.step()

            total_loss += losses.item()
        print("Train Loss:", total_loss / len(loader))
    
    # Évaluation
    metrics = evaluate(model, loader)
    print(f"Metrics snow_level{level}:", metrics)  # Changé ici
    
    # Sauvegarde CSV
    df = pd.DataFrame([metrics])
    df.to_csv(f"./Data1/results/snow_level{level}/RCNN_metrics.csv", index=False)

print("\n Entraînement et évaluation pour tous les niveaux de snow terminés !")  # Changé ici

DEVICE: cuda

=== Training and evaluating snow_level1 ===

Epoch 1/5 - Training snow_level1


Train Loss: 0.3664986505508423
Epoch 2/5 - Training snow_level1


Train Loss: 0.21624299573898315
Epoch 3/5 - Training snow_level1


Train Loss: 0.20545065808296203
Epoch 4/5 - Training snow_level1


Train Loss: 0.17623081225156784
Epoch 5/5 - Training snow_level1


...

Train Loss: 0.3284932874441147
Epoch 3/5 - Training snow_level5


Train Loss: 0.264477424621582
Epoch 4/5 - Training snow_level5


Train Loss: 0.2416069267988205
Epoch 5/5 - Training snow_level5


Train Loss: 0.22556002306938172


Metrics snow_level5: {'Box-P': 0.9952550415183867, 'Box-R': 0.9973333333333333, 'Box-F1': 0.99627182565387, 'mAP50': np.float64(0.5162428717017173), 'mAP50-95': np.float64(0.4634569448828697)}

 Entraînement et évaluation pour tous les niveaux de snow terminés !


# DETR train et test  

In [11]:
# ======================================
#   RCNN - Évaluation sur snow_level1-5
# ======================================

import os
import torch
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms import functional as F
from tqdm import tqdm
from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd
from torchvision.ops import box_iou

# === Dataset pour labels YOLO ===
class YoloDataset(Dataset):
    def __init__(self, img_dir, labels_dir, transform=None):
        self.img_dir = img_dir
        self.labels_dir = labels_dir
        self.transform = transform
        self.images = [f for f in os.listdir(img_dir) if f.endswith((".jpg",".png",".jpeg"))]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        file = self.images[idx]
        img_path = os.path.join(self.img_dir, file)
        label_path = os.path.join(self.labels_dir, file.replace(".jpg", ".txt").replace(".png", ".txt"))

        img = Image.open(img_path).convert("RGB")
        w, h = img.size

        boxes, labels = [], []
        if os.path.exists(label_path):
            with open(label_path, "r") as f:
                for line in f.readlines():
                    cls, x, y, bw, bh = map(float, line.split())
                    xmin = (x - bw/2) * w
                    ymin = (y - bh/2) * h
                    xmax = (x + bw/2) * w
                    ymax = (y + bh/2) * h
                    boxes.append([xmin, ymin, xmax, ymax])
                    labels.append(int(cls))

        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.int64)

        target = {"boxes": boxes, "labels": labels}
        img = F.to_tensor(img)
        return img, target

# === Fonction d'évaluation ===
def evaluate(model, loader, iou_threshold=0.5):
    model.eval()
    all_matched_gt_labels = []
    all_matched_pred_labels = []
    all_map50, all_map5095 = [], []

    with torch.no_grad():
        for imgs, targets in tqdm(loader, leave=False):
            imgs = [img.to(device) for img in imgs]
            preds = model(imgs)

            for pred, target in zip(preds, targets):
                gt_boxes = target["boxes"].to(device)
                gt_labels = target["labels"].to(device)
                pred_boxes = pred["boxes"]
                pred_labels = pred["labels"]
                pred_scores = pred["scores"]

                if len(gt_boxes) == 0 or len(pred_boxes) == 0:
                    continue

                score_mask = pred_scores >= 0.5
                pred_boxes = pred_boxes[score_mask]
                pred_labels = pred_labels[score_mask]

                if len(pred_boxes) == 0:
                    continue

                iou = box_iou(gt_boxes, pred_boxes)
                max_iou_per_gt, best_pred_idx = iou.max(dim=1)

                all_map50.append((max_iou_per_gt >= 0.5).float().mean().item())
                all_map5095.append(max_iou_per_gt.mean().item())

                matched_mask = max_iou_per_gt >= iou_threshold
                if matched_mask.sum() > 0:
                    matched_gt_labels = gt_labels[matched_mask]
                    matched_pred_labels = pred_labels[best_pred_idx[matched_mask]]
                    all_matched_gt_labels.extend(matched_gt_labels.cpu().numpy().tolist())
                    all_matched_pred_labels.extend(matched_pred_labels.cpu().numpy().tolist())

    if len(all_matched_gt_labels) > 0:
        precision = precision_score(all_matched_gt_labels, all_matched_pred_labels, average="macro", zero_division=0)
        recall = recall_score(all_matched_gt_labels, all_matched_pred_labels, average="macro", zero_division=0)
        f1 = f1_score(all_matched_gt_labels, all_matched_pred_labels, average="macro", zero_division=0)
    else:
        precision, recall, f1 = 0, 0, 0

    return {
        "Box-P": precision,
        "Box-R": recall,
        "Box-F1": f1,
        "mAP50": np.mean(all_map50) if all_map50 else 0,
        "mAP50-95": np.mean(all_map5095) if all_map5095 else 0
    }

# === GPU ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", device)

# === Paramètres ===
nc = 5
epochs = 5
batch_size = 4
snow_levels = [1, 2, 3, 4, 5]

# === Boucle sur snow_level1 → snow_level5 ===
for level in snow_levels:
    print(f"\n=== Training and evaluating snow_level{level} ===\n")
    
    img_dir = f"./Data1/results/snow_level{level}/images"
    labels_dir = f"./Data1/results/snow_level{level}/labels_yolo"

    dataset = YoloDataset(img_dir, labels_dir)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))
    
    # Modèle RCNN
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, nc)
    model.to(device)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)
    
    # === Entraînement ===
    for epoch in range(epochs):
        print(f"Epoch {epoch+1}/{epochs} - Training snow_level{level}")
        model.train()
        total_loss = 0
        for imgs, targets in tqdm(loader, leave=False):
            imgs = [img.to(device) for img in imgs]
            targets = [{k: v.to(device) for k,v in t.items()} for t in targets]

            # Ignorer les images sans annotations
            if any([len(t['boxes'])==0 for t in targets]):
                continue

            loss_dict = model(imgs, targets)
            losses = sum(loss for loss in loss_dict.values())

            optimizer.zero_grad()
            losses.backward()
            optimizer.step()

            total_loss += losses.item()
        print("Train Loss:", total_loss / len(loader))
    
    # === Évaluation ===
    metrics = evaluate(model, loader)
    print(f"Metrics snow_level{level}:", metrics)
    
    # === Sauvegarde CSV ===
    df = pd.DataFrame([metrics])
    output_path = f"./Data1/results/snow_level{level}/RCNN_metrics.csv"
    df.to_csv(output_path, index=False)
    print(f" Résultats sauvegardés dans : {output_path}")

print("\n Entraînement et évaluation terminés pour tous les niveaux de neige (snow_level1 → snow_level5) !")


DEVICE: cuda

=== Training and evaluating snow_level1 ===

Epoch 1/5 - Training snow_level1


Train Loss: 0.39234311509132386
Epoch 2/5 - Training snow_level1


Train Loss: 0.24565544074773787
Epoch 3/5 - Training snow_level1


Train Loss: 0.21592006891965865
Epoch 4/5 - Training snow_level1


Train Loss: 0.19407511961460114
Epoch 5/5 - Training snow_level1


...

Train Loss: 0.32996144127845767
Epoch 3/5 - Training snow_level5


Train Loss: 0.27241386222839353
Epoch 4/5 - Training snow_level5


Train Loss: 0.252778227686882
Epoch 5/5 - Training snow_level5


Train Loss: 0.22727279150485993


Metrics snow_level5: {'Box-P': 0.9968382195092357, 'Box-R': 0.9958381605088192, 'Box-F1': 0.9963327974651103, 'mAP50': np.float64(0.5150396174980787), 'mAP50-95': np.float64(0.4641782995724009)}
 Résultats sauvegardés dans : ./Data1/results/snow_level5/RCNN_metrics.csv

 Entraînement et évaluation terminés pour tous les niveaux de neige (snow_level1 → snow_level5) !


In [15]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Données des métriques pour chaque condition météorologique
data = {
    'Condition': ['Fog', 'Fog', 'Fog', 'Fog', 'Fog',
                 'Glare', 'Glare', 'Glare', 'Glare', 'Glare',
                 'Night', 'Night', 'Night', 'Night', 'Night',
                 'Rain', 'Rain', 'Rain', 'Rain', 'Rain',
                 'Snow', 'Snow', 'Snow', 'Snow', 'Snow'],
    
    'Level': [1, 2, 3, 4, 5] * 5,
    
    'Precision': [0.946570, 0.935887, 0.860123, 0.864934, 0.853431,
                  0.930744, 0.927456, 0.937931, 0.936548, 0.947938,
                  0.917523, 0.913384, 0.837516, 0.852761, 0.813608,
                  0.882714, 0.796077, 0.765278, 0.705104, 0.658812,
                  0.923900, 0.874040, 0.784358, 0.641015, 0.565648],
    
    'Recall': [0.888444, 0.879188, 0.826829, 0.699716, 0.650729,
               0.892045, 0.899366, 0.883712, 0.878461, 0.878158,
               0.835939, 0.794528, 0.791854, 0.739568, 0.702203,
               0.779667, 0.731895, 0.635464, 0.478233, 0.352694,
               0.881409, 0.837809, 0.665188, 0.470768, 0.184825],
    
    'mAP50': [0.917434, 0.905792, 0.875295, 0.820859, 0.788426,
              0.923961, 0.925595, 0.921220, 0.924839, 0.925628,
              0.879698, 0.847937, 0.818365, 0.770859, 0.729463,
              0.831654, 0.765348, 0.702344, 0.574838, 0.469511,
              0.906978, 0.867632, 0.749166, 0.529694, 0.263719],
    
    'mAP50-95': [0.739336, 0.714160, 0.667098, 0.606611, 0.573878,
                  0.747775, 0.748017, 0.746060, 0.745944, 0.743987,
                  0.699469, 0.669769, 0.641441, 0.606034, 0.569764,
                  0.617434, 0.565173, 0.502573, 0.405390, 0.329272,
                  0.728555, 0.671967, 0.529519, 0.339280, 0.145198]
}

# Création du DataFrame
df = pd.DataFrame(data)

# Fonction pour créer un tableau stylisé
def create_styled_table(df):
    # Pivoter les données pour avoir les conditions en colonnes
    metrics = ['Precision', 'Recall', 'mAP50', 'mAP50-95']
    
    # Créer un tableau pour chaque métrique
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Performance Metrics by Weather Condition and Level', fontsize=16, fontweight='bold')
    
    colors = {'Fog': 'lightblue', 'Glare': 'lightyellow', 'Night': 'lightgray', 
              'Rain': 'lightgreen', 'Snow': 'lightcoral'}
    
    for idx, metric in enumerate(metrics):
        ax = axes[idx//2, idx%2]
        
        # Préparer les données pour cette métrique
        metric_data = []
        conditions = []
        levels = [1, 2, 3, 4, 5]
        
        for condition in ['Fog', 'Glare', 'Night', 'Rain', 'Snow']:
            condition_data = df[df['Condition'] == condition][metric].values
            metric_data.append(condition_data)
            conditions.append(condition)
        
        # Créer le graphique
        x_pos = np.arange(len(levels))
        width = 0.15
        
        for i, (condition, data_vals) in enumerate(zip(conditions, metric_data)):
            ax.bar(x_pos + i*width, data_vals, width, label=condition, 
                   color=colors[condition], alpha=0.8, edgecolor='black')
        
        ax.set_xlabel('Level')
        ax.set_ylabel(metric)
        ax.set_title(f'{metric} by Condition and Level')
        ax.set_xticks(x_pos + 2*width)
        ax.set_xticklabels(levels)
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        # Ajouter les valeurs sur les barres
        for i, condition_data in enumerate(metric_data):
            for j, value in enumerate(condition_data):
                ax.text(x_pos[j] + i*width, value + 0.01, f'{value:.3f}', 
                       ha='center', va='bottom', fontsize=8, rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    # Afficher aussi un tableau numérique
    print("TABLEAU COMPLET DES MÉTRIQUES")
    print("="*80)
    
    # Tableau pour chaque condition
    for condition in ['Fog', 'Glare', 'Night', 'Rain', 'Snow']:
        print(f"\n{condition.upper()}")
        print("-"*50)
        condition_df = df[df['Condition'] == condition]
        print(f"{'Level':<6} {'Precision':<10} {'Recall':<10} {'mAP50':<10} {'mAP50-95':<10}")
        for _, row in condition_df.iterrows():
            print(f"{row['Level']:<6} {row['Precision']:<10.3f} {row['Recall']:<10.3f} "
                  f"{row['mAP50']:<10.3f} {row['mAP50-95']:<10.3f}")

# Créer et afficher le tableau
create_styled_table(df)

# Tableau récapitulatif avec moyennes
print("\n" + "="*80)
print("RÉCAPITULATIF PAR CONDITION (Moyennes)")
print("="*80)
summary_df = df.groupby('Condition').agg({
    'Precision': 'mean',
    'Recall': 'mean', 
    'mAP50': 'mean',
    'mAP50-95': 'mean'
}).round(4)

print(summary_df)

# Graphique de tendance par niveau
plt.figure(figsize=(14, 10))

metrics = ['Precision', 'Recall', 'mAP50', 'mAP50-95']
colors = {'Fog': 'blue', 'Glare': 'orange', 'Night': 'black', 
          'Rain': 'green', 'Snow': 'red'}

for idx, metric in enumerate(metrics):
    plt.subplot(2, 2, idx+1)
    
    for condition in ['Fog', 'Glare', 'Night', 'Rain', 'Snow']:
        condition_data = df[df['Condition'] == condition]
        plt.plot(condition_data['Level'], condition_data[metric], 
                marker='o', linewidth=2, label=condition, color=colors[condition])
    
    plt.xlabel('Level')
    plt.ylabel(metric)
    plt.title(f'Tendance {metric} par niveau')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.xticks([1, 2, 3, 4, 5])

plt.tight_layout()
plt.show()

<Figure size 1600x1200 with 4 Axes>

TABLEAU COMPLET DES MÉTRIQUES

FOG
--------------------------------------------------
Level  Precision  Recall     mAP50      mAP50-95  
1      0.947      0.888      0.917      0.739     
2      0.936      0.879      0.906      0.714     
3      0.860      0.827      0.875      0.667     
4      0.865      0.700      0.821      0.607     
5      0.853      0.651      0.788      0.574     

GLARE
--------------------------------------------------
Level  Precision  Recall     mAP50      mAP50-95  
1      0.931      0.892      0.924      0.748     
2      0.927      0.899      0.926      0.748     
3      0.938      0.884      0.921      0.746     
4      0.937      0.878      0.925      0.746     
5      0.948      0.878      0.926      0.744     

NIGHT
--------------------------------------------------
Level  Precision  Recall     mAP50      mAP50-95  
1      0.918      0.836      0.880      0.699     
2      0.913      0.795      0.848      0.670     
3      0.838      0.792      0.81

<Figure size 1400x1000 with 4 Axes>

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Données des métriques pour chaque condition météorologique
data = {
    'Condition': ['Fog', 'Fog', 'Fog', 'Fog', 'Fog',
                 'Glare', 'Glare', 'Glare', 'Glare', 'Glare',
                 'Night', 'Night', 'Night', 'Night', 'Night',
                 'Rain', 'Rain', 'Rain', 'Rain', 'Rain',
                 'Snow', 'Snow', 'Snow', 'Snow', 'Snow'],
    
    'Level': [1, 2, 3, 4, 5] * 5,
    
    'Precision': [0.946570, 0.935887, 0.860123, 0.864934, 0.853431,
                  0.930744, 0.927456, 0.937931, 0.936548, 0.947938,
                  0.917523, 0.913384, 0.837516, 0.852761, 0.813608,
                  0.882714, 0.796077, 0.765278, 0.705104, 0.658812,
                  0.923900, 0.874040, 0.784358, 0.641015, 0.565648],
    
    'Recall': [0.888444, 0.879188, 0.826829, 0.699716, 0.650729,
               0.892045, 0.899366, 0.883712, 0.878461, 0.878158,
               0.835939, 0.794528, 0.791854, 0.739568, 0.702203,
               0.779667, 0.731895, 0.635464, 0.478233, 0.352694,
               0.881409, 0.837809, 0.665188, 0.470768, 0.184825],
    
    'mAP50': [0.917434, 0.905792, 0.875295, 0.820859, 0.788426,
              0.923961, 0.925595, 0.921220, 0.924839, 0.925628,
              0.879698, 0.847937, 0.818365, 0.770859, 0.729463,
              0.831654, 0.765348, 0.702344, 0.574838, 0.469511,
              0.906978, 0.867632, 0.749166, 0.529694, 0.263719],
    
    'mAP50-95': [0.739336, 0.714160, 0.667098, 0.606611, 0.573878,
                  0.747775, 0.748017, 0.746060, 0.745944, 0.743987,
                  0.699469, 0.669769, 0.641441, 0.606034, 0.569764,
                  0.617434, 0.565173, 0.502573, 0.405390, 0.329272,
                  0.728555, 0.671967, 0.529519, 0.339280, 0.145198]
}

# Création du DataFrame
df = pd.DataFrame(data)

# =====================================================
# GRAPHIQUE EN BARRES GROUPÉES AMÉLIORÉ
# =====================================================
def create_improved_bar_chart(df):
    # Couleurs vives et contrastées
    colors = {
        'Fog': '#4A90E2',      # Bleu vif
        'Glare': '#F5A623',    # Orange vif
        'Night': '#7B68EE',    # Violet moyen
        'Rain': '#50E3C2',     # Turquoise
        'Snow': '#E74C3C'      # Rouge vif
    }
    
    metrics = ['Precision', 'Recall', 'mAP50', 'mAP50-95']
    
    fig, axes = plt.subplots(2, 2, figsize=(20, 14))
    fig.suptitle('Performance YOLOv8 par Condition Météo et Niveau', 
                 fontsize=20, fontweight='bold', y=0.995)
    
    for idx, metric in enumerate(metrics):
        ax = axes[idx//2, idx%2]
        
        # Préparer les données
        conditions = ['Fog', 'Glare', 'Night', 'Rain', 'Snow']
        levels = [1, 2, 3, 4, 5]
        
        # Position des barres
        x = np.arange(len(levels))
        width = 0.16  # Largeur des barres
        
        # Dessiner les barres pour chaque condition
        for i, condition in enumerate(conditions):
            condition_data = df[df['Condition'] == condition][metric].values
            bars = ax.bar(x + i*width, condition_data, width, 
                         label=condition, 
                         color=colors[condition], 
                         alpha=0.9,
                         edgecolor='black',
                         linewidth=1.5)
            
            # Ajouter les valeurs AU-DESSUS des barres (VERTICAL)
            for j, (bar, value) in enumerate(zip(bars, condition_data)):
                height = bar.get_height()
                ax.text(bar.get_x() + bar.get_width()/2., height + 0.015,
                       f'{value:.2f}',
                       ha='center', va='bottom', 
                       fontsize=9, 
                       fontweight='bold',
                       rotation=0)  # Pas de rotation
        
        # Configuration de l'axe
        ax.set_xlabel('Niveau de Perturbation', fontsize=14, fontweight='bold')
        ax.set_ylabel(metric, fontsize=14, fontweight='bold')
        ax.set_title(f'{metric}', fontsize=16, fontweight='bold', pad=15)
        ax.set_xticks(x + 2*width)
        ax.set_xticklabels(levels, fontsize=12)
        ax.legend(loc='upper right', fontsize=11, framealpha=0.95, 
                 edgecolor='black', fancybox=True)
        ax.grid(axis='y', alpha=0.4, linestyle='--', linewidth=0.8)
        ax.set_ylim([0, 1.15])  # Espace pour les valeurs
        
        # Style de la grille
        ax.set_axisbelow(True)
        
        # Bordure du graphique
        for spine in ax.spines.values():
            spine.set_edgecolor('black')
            spine.set_linewidth(1.5)
    
    plt.tight_layout()
    plt.savefig('performance_yolov8_barres.png', dpi=300, bbox_inches='tight')
    print(" Graphique barres sauvegardé : performance_yolov8_barres.png")
    plt.show()

# =====================================================
# GRAPHIQUE DE TENDANCE AMÉLIORÉ
# =====================================================
def create_improved_line_chart(df):
    # Couleurs vives et contrastées
    colors = {
        'Fog': '#4A90E2',      # Bleu vif
        'Glare': '#F5A623',    # Orange vif
        'Night': '#7B68EE',    # Violet moyen
        'Rain': '#50E3C2',     # Turquoise
        'Snow': '#E74C3C'      # Rouge vif
    }
    
    # Marqueurs différents
    markers = {
        'Fog': 'o',
        'Glare': 's',
        'Night': '^',
        'Rain': 'D',
        'Snow': 'v'
    }
    
    metrics = ['Precision', 'Recall', 'mAP50', 'mAP50-95']
    
    fig, axes = plt.subplots(2, 2, figsize=(20, 14))
    fig.suptitle('Évolution des Performances YOLOv8 par Niveau', 
                 fontsize=20, fontweight='bold', y=0.995)
    
    for idx, metric in enumerate(metrics):
        ax = axes[idx//2, idx%2]
        
        for condition in ['Fog', 'Glare', 'Night', 'Rain', 'Snow']:
            condition_data = df[df['Condition'] == condition]
            ax.plot(condition_data['Level'], condition_data[metric], 
                   marker=markers[condition], 
                   linewidth=3.5, 
                   markersize=12,
                   label=condition, 
                   color=colors[condition],
                   markeredgecolor='black',
                   markeredgewidth=1.5)
            
            # Ajouter les valeurs sur les points
            for x, y in zip(condition_data['Level'], condition_data[metric]):
                ax.annotate(f'{y:.2f}', 
                           xy=(x, y), 
                           xytext=(0, 10), 
                           textcoords='offset points',
                           fontsize=8, 
                           ha='center',
                           bbox=dict(boxstyle='round,pad=0.3', 
                                   facecolor=colors[condition], 
                                   alpha=0.3,
                                   edgecolor='none'))
        
        ax.set_xlabel('Niveau de Perturbation', fontsize=14, fontweight='bold')
        ax.set_ylabel(metric, fontsize=14, fontweight='bold')
        ax.set_title(f'Tendance {metric}', fontsize=16, fontweight='bold', pad=15)
        ax.legend(loc='best', fontsize=12, framealpha=0.95,
                 edgecolor='black', fancybox=True)
        ax.grid(True, alpha=0.4, linestyle='--', linewidth=0.8)
        ax.set_xticks([1, 2, 3, 4, 5])
        ax.set_ylim([0, 1.05])
        
        # Bordure du graphique
        for spine in ax.spines.values():
            spine.set_edgecolor('black')
            spine.set_linewidth(1.5)
    
    plt.tight_layout()
    plt.savefig('performance_yolov8_tendance.png', dpi=300, bbox_inches='tight')
    print(" Graphique tendance sauvegardé : performance_yolov8_tendance.png")
    plt.show()

# =====================================================
# TABLEAUX NUMÉRIQUES STYLISÉS
# =====================================================
def print_styled_tables(df):
    print("\n" + "="*100)
    print("TABLEAUX DÉTAILLÉS DES MÉTRIQUES PAR CONDITION")
    print("="*100)
    
    for condition in ['Fog', 'Glare', 'Night', 'Rain', 'Snow']:
        print(f"\n{'='*100}")
        print(f" {condition.upper()}")
        print(f"{'='*100}")
        condition_df = df[df['Condition'] == condition].copy()
        
        # Calculer les variations par rapport au level 1
        ref_row = condition_df[condition_df['Level'] == 1].iloc[0]
        
        print(f"{'Level':<8} {'Precision':<12} {'Recall':<12} {'mAP50':<12} {'mAP50-95':<12}")
        print("-" * 100)
        
        for _, row in condition_df.iterrows():
            delta_p = row['Precision'] - ref_row['Precision']
            delta_r = row['Recall'] - ref_row['Recall']
            delta_map50 = row['mAP50'] - ref_row['mAP50']
            delta_map5095 = row['mAP50-95'] - ref_row['mAP50-95']
            
            print(f"{row['Level']:<8} "
                  f"{row['Precision']:.3f}({delta_p:+.3f}){'':<2} "
                  f"{row['Recall']:.3f}({delta_r:+.3f}){'':<2} "
                  f"{row['mAP50']:.3f}({delta_map50:+.3f}){'':<2} "
                  f"{row['mAP50-95']:.3f}({delta_map5095:+.3f})")

# =====================================================
# TABLEAU RÉCAPITULATIF AVEC DÉGRADATION
# =====================================================
def print_summary_table(df):
    print("\n" + "="*100)
    print(" RÉCAPITULATIF PAR CONDITION (Moyennes + Dégradation Level 1→5)")
    print("="*100)
    
    summary_data = []
    for condition in ['Fog', 'Glare', 'Night', 'Rain', 'Snow']:
        condition_df = df[df['Condition'] == condition]
        level1 = condition_df[condition_df['Level'] == 1].iloc[0]
        level5 = condition_df[condition_df['Level'] == 5].iloc[0]
        
        avg_precision = condition_df['Precision'].mean()
        avg_recall = condition_df['Recall'].mean()
        avg_map50 = condition_df['mAP50'].mean()
        avg_map5095 = condition_df['mAP50-95'].mean()
        
        degradation = (level1['mAP50'] - level5['mAP50']) / level1['mAP50'] * 100
        
        summary_data.append({
            'Condition': condition,
            'Avg P': f"{avg_precision:.3f}",
            'Avg R': f"{avg_recall:.3f}",
            'Avg mAP50': f"{avg_map50:.3f}",
            'Avg mAP50-95': f"{avg_map5095:.3f}",
            'Dégradation (%)': f"{degradation:.1f}%"
        })
    
    summary_df = pd.DataFrame(summary_data)
    print(summary_df.to_string(index=False))

# =====================================================
# EXÉCUTION
# =====================================================
print(" Génération des graphiques améliorés...")

# Graphiques en barres
create_improved_bar_chart(df)

# Graphiques de tendance
create_improved_line_chart(df)

# Tableaux numériques
print_styled_tables(df)

# Résumé
print_summary_table(df)

print("\n" + "="*100)
print(" GÉNÉRATION TERMINÉE!")
print("="*100)

 Génération des graphiques améliorés...
 Graphique barres sauvegardé : performance_yolov8_barres.png


<Figure size 2000x1400 with 4 Axes>

 Graphique tendance sauvegardé : performance_yolov8_tendance.png


<Figure size 2000x1400 with 4 Axes>


TABLEAUX DÉTAILLÉS DES MÉTRIQUES PAR CONDITION

 FOG
Level    Precision    Recall       mAP50        mAP50-95    
----------------------------------------------------------------------------------------------------
1        0.947(+0.000)   0.888(+0.000)   0.917(+0.000)   0.739(+0.000)
2        0.936(-0.011)   0.879(-0.009)   0.906(-0.012)   0.714(-0.025)
3        0.860(-0.086)   0.827(-0.062)   0.875(-0.042)   0.667(-0.072)
4        0.865(-0.082)   0.700(-0.189)   0.821(-0.097)   0.607(-0.133)
5        0.853(-0.093)   0.651(-0.238)   0.788(-0.129)   0.574(-0.165)

 GLARE
Level    Precision    Recall       mAP50        mAP50-95    
----------------------------------------------------------------------------------------------------
1        0.931(+0.000)   0.892(+0.000)   0.924(+0.000)   0.748(+0.000)
2        0.927(-0.003)   0.899(+0.007)   0.926(+0.002)   0.748(+0.000)
3        0.938(+0.007)   0.884(-0.008)   0.921(-0.003)   0.746(-0.002)
4        0.937(+0.006)   0.878(-0.014)   0.925

In [ ]:
import os
import json
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import DetrForObjectDetection, DetrImageProcessor
from tqdm import tqdm
import torch.optim as optim
from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd
from torchvision.ops import box_iou

# =========================
# CONFIGURATION
# =========================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"DEVICE: {DEVICE}")

IMAGES_DIR = "./Data2/train/images"
LABELS_YOLO_DIR = "./Data2/train/labels"
LABELS_COCO_DIR = "./Data2/train/labels_coco"

TEST_IMAGES_DIR = "./Data2/test/images"
TEST_LABELS_YOLO_DIR = "./Data2/test/labels"
TEST_LABELS_COCO_DIR = "./Data2/test/labels_coco"

SAVE_DIR = "./models_detr"

BATCH_SIZE = 2
EPOCHS = 10
LR = 1e-4
NUM_CLASSES = 5  #  5 classes comme dans tes données

os.makedirs(LABELS_COCO_DIR, exist_ok=True)
os.makedirs(TEST_LABELS_COCO_DIR, exist_ok=True)
os.makedirs(SAVE_DIR, exist_ok=True)

# =========================
# CONVERSION YOLO -> COCO
# =========================
def convert_yolo_to_coco(images_dir, labels_yolo_dir, labels_coco_dir):
    print(f"Conversion YOLO -> COCO pour {images_dir}...")
    converted = 0
    
    for label_file in os.listdir(labels_yolo_dir):
        if not label_file.endswith(".txt"):
            continue
        
        # Essayer différentes extensions d'image
        base_name = label_file.replace(".txt", "")
        image_file = None
        for ext in [".jpg", ".jpeg", ".png"]:
            test_path = os.path.join(images_dir, base_name + ext)
            if os.path.exists(test_path):
                image_file = base_name + ext
                break
        
        if image_file is None:
            continue
        
        image_path = os.path.join(images_dir, image_file)
        img = Image.open(image_path)
        width, height = img.size

        yolo_path = os.path.join(labels_yolo_dir, label_file)
        coco_path = os.path.join(labels_coco_dir, label_file.replace(".txt", ".json"))
        
        boxes, labels, areas, iscrowd = [], [], [], []

        with open(yolo_path, "r") as f:
            lines = f.readlines()
            for line in lines:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                
                cls, x_center, y_center, w, h = map(float, parts)
                
                #  Validation
                if not (0 <= x_center <= 1 and 0 <= y_center <= 1 and 
                        0 < w <= 1 and 0 < h <= 1):
                    continue
                
                # Conversion coordonnées COCO (xyxy absolu)
                x_min = (x_center - w/2) * width
                y_min = (y_center - h/2) * height
                x_max = (x_center + w/2) * width
                y_max = (y_center + h/2) * height
                
                #  Vérifier que la box est valide
                if x_max <= x_min or y_max <= y_min:
                    continue
                
                boxes.append([x_min, y_min, x_max, y_max])
                labels.append(int(cls))
                areas.append((x_max - x_min) * (y_max - y_min))
                iscrowd.append(0)

        target = {
            "boxes": boxes,
            "class_labels": labels,  #  DETR utilise "class_labels"
            "area": areas,
            "iscrowd": iscrowd
        }

        with open(coco_path, "w") as f:
            json.dump(target, f)
        
        converted += 1
    
    print(f" {converted} fichiers convertis")

# Convertir train et test
convert_yolo_to_coco(IMAGES_DIR, LABELS_YOLO_DIR, LABELS_COCO_DIR)
convert_yolo_to_coco(TEST_IMAGES_DIR, TEST_LABELS_YOLO_DIR, TEST_LABELS_COCO_DIR)

# =========================
# DATASET
# =========================
class DetrDataset(Dataset):
    def __init__(self, images_dir, labels_dir):
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        
        # Lister les images
        all_images = []
        for ext in [".jpg", ".jpeg", ".png"]:
            all_images.extend([f for f in os.listdir(images_dir) if f.lower().endswith(ext)])
        
        self.images = sorted(list(set(all_images)))
        print(f"Dataset: {len(self.images)} images trouvées")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.images_dir, img_name)
        
        base_name = os.path.splitext(img_name)[0]
        label_path = os.path.join(self.labels_dir, base_name + ".json")

        image = Image.open(img_path).convert("RGB")

        with open(label_path, "r") as f:
            target = json.load(f)

        # Convertir en tensors
        target = {
            "boxes": torch.tensor(target["boxes"], dtype=torch.float32) if target["boxes"] else torch.zeros((0, 4), dtype=torch.float32),
            "class_labels": torch.tensor(target["class_labels"], dtype=torch.int64) if target["class_labels"] else torch.zeros((0,), dtype=torch.int64)
        }

        return image, target

# =========================
# DATALOADER
# =========================
def collate_fn(batch):
    images, targets = zip(*batch)
    return list(images), list(targets)

train_dataset = DetrDataset(IMAGES_DIR, LABELS_COCO_DIR)
test_dataset = DetrDataset(TEST_IMAGES_DIR, TEST_LABELS_COCO_DIR)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

# =========================
# MODELE
# =========================
processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")

#  CORRECTION: utiliser id2label et label2id + ignore_mismatched_sizes
id2label = {i: f"class{i}" for i in range(NUM_CLASSES)}
label2id = {f"class{i}": i for i in range(NUM_CLASSES)}

model = DetrForObjectDetection.from_pretrained(
    "facebook/detr-resnet-50",
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True  #  IMPORTANT
)
model.to(DEVICE)

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

# =========================
# HELPER FUNCTIONS
# =========================
def cxcywh_to_xyxy(boxes, img_w, img_h):
    if boxes.numel() == 0:
        return torch.zeros((0, 4), dtype=torch.float32)
    cx = boxes[:, 0] * img_w
    cy = boxes[:, 1] * img_h
    w = boxes[:, 2] * img_w
    h = boxes[:, 3] * img_h
    x1 = cx - w/2
    y1 = cy - h/2
    x2 = cx + w/2
    y2 = cy + h/2
    return torch.stack([x1, y1, x2, y2], dim=1)

# =========================
# TRAIN
# =========================
def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0
    n_batches = 0

    for images, targets in tqdm(loader, desc="Train"):
        encoding = processor(images=images, return_tensors="pt")
        pixel_values = encoding["pixel_values"].to(DEVICE)
        pixel_mask = encoding["pixel_mask"].to(DEVICE)

        # Filtrer annotations vides
        labels = []
        for t in targets:
            if t["class_labels"].numel() > 0:
                labels.append({
                    "class_labels": t["class_labels"].to(DEVICE),
                    "boxes": t["boxes"].to(DEVICE)
                })
        
        if len(labels) == 0:
            continue
        
        if len(labels) < len(images):
            pixel_values = pixel_values[:len(labels)]
            pixel_mask = pixel_mask[:len(labels)]

        optimizer.zero_grad()
        outputs = model(pixel_values=pixel_values, pixel_mask=pixel_mask, labels=labels)
        loss = outputs.loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.1)
        optimizer.step()

        total_loss += loss.item()
        n_batches += 1

    return total_loss / max(n_batches, 1)

# =========================
# EVAL
# =========================
def evaluate(model, loader, conf_threshold=0.1):
    model.eval()
    all_gt = []
    all_pred = []
    all_ious = []
    total_det = 0
    total_gt = 0

    with torch.no_grad():
        for images, targets in tqdm(loader, desc="Eval"):
            encoding = processor(images=images, return_tensors="pt")
            pixel_values = encoding["pixel_values"].to(DEVICE)
            pixel_mask = encoding["pixel_mask"].to(DEVICE)

            outputs = model(pixel_values=pixel_values, pixel_mask=pixel_mask)
            
            target_sizes = torch.tensor([img.size[::-1] for img in images])
            results = processor.post_process_object_detection(
                outputs, target_sizes=target_sizes, threshold=conf_threshold
            )

            for i, res in enumerate(results):
                pred_boxes = torch.tensor(res["boxes"], dtype=torch.float32) if len(res["boxes"]) > 0 else torch.zeros((0, 4), dtype=torch.float32)
                pred_labels = torch.tensor(res["labels"], dtype=torch.int64) if len(res["labels"]) > 0 else torch.zeros((0,), dtype=torch.int64)
                
                gt_boxes = targets[i]["boxes"]
                gt_labels = targets[i]["class_labels"]
                
                total_det += len(pred_labels)
                total_gt += len(gt_labels)
                
                if len(pred_boxes) == 0 or len(gt_boxes) == 0:
                    continue
                
                iou = box_iou(gt_boxes, pred_boxes)
                max_iou, best_idx = iou.max(dim=1)
                all_ious.extend(max_iou.cpu().numpy().tolist())
                
                matched = max_iou >= 0.5
                if matched.sum() > 0:
                    all_gt.extend(gt_labels[matched].cpu().numpy().tolist())
                    all_pred.extend(pred_labels[best_idx[matched]].cpu().numpy().tolist())

    if len(all_gt) > 0 and len(all_pred) > 0:
        prec = precision_score(all_gt, all_pred, average="macro", zero_division=0)
        rec = recall_score(all_gt, all_pred, average="macro", zero_division=0)
        f1 = f1_score(all_gt, all_pred, average="macro", zero_division=0)
    else:
        prec, rec, f1 = 0, 0, 0

    map50 = sum(1 for iou in all_ious if iou >= 0.5) / len(all_ious) if all_ious else 0
    map5095 = sum(all_ious) / len(all_ious) if all_ious else 0

    print(f"  -> Det: {total_det}, GT: {total_gt}, Matched: {len(all_gt)}")
    
    return {
        "Box-P": round(prec, 5),
        "Box-R": round(rec, 5),
        "Box-F1": round(f1, 5),
        "mAP50": round(map50, 5),
        "mAP50-95": round(map5095, 5)
    }

# =========================
# MAIN LOOP
# =========================
train_history = []
test_history = []

for epoch in range(1, EPOCHS + 1):
    print(f"\n{'='*50}")
    print(f"Epoch {epoch}/{EPOCHS}")
    print('='*50)
    
    loss = train_one_epoch(model, train_loader, optimizer)
    print(f"Loss: {loss:.4f}")

    print("\nTrain eval:")
    train_metrics = evaluate(model, train_loader)
    print(f"Train: {train_metrics}")

    print("\nTest eval:")
    test_metrics = evaluate(model, test_loader)
    print(f"Test: {test_metrics}")

    torch.save(model.state_dict(), os.path.join(SAVE_DIR, f"detr_epoch{epoch}.pt"))

    train_history.append({"epoch": epoch, "loss": loss, **train_metrics})
    test_history.append({"epoch": epoch, **test_metrics})

pd.DataFrame(train_history).to_csv(os.path.join(SAVE_DIR, "train_history.csv"), index=False)
pd.DataFrame(test_history).to_csv(os.path.join(SAVE_DIR, "test_history.csv"), index=False)

print(f"\n TERMINÉ! Modèles: {SAVE_DIR}")

DEVICE: cpu
Conversion YOLO -> COCO pour ./Data2/train/images...
 677 fichiers convertis
Conversion YOLO -> COCO pour ./Data2/test/images...
 100 fichiers convertis
Dataset: 677 images trouvées
Dataset: 100 images trouvées


Some weights of the model checkpoint at facebook/detr-resnet-50 were not used when initializing DetrForObjectDetection: ['model.backbone.conv_encoder.model.layer1.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer2.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer3.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer4.0.downsample.1.num_batches_tracked']
- This IS expected if you are initializing DetrForObjectDetection from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DetrForObjectDetection from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of DetrForObjectDetection were not initialized from the model checkpoin


Epoch 1/10


Train: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████| 339/339 [30:30<00:00,  5.40s/it]


Loss: 6497.0602

Train eval:


...

Eval: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████| 339/339 [12:10<00:00,  2.16s/it]


  -> Det: 2708, GT: 2100, Matched: 0
Train: {'Box-P': 0, 'Box-R': 0, 'Box-F1': 0, 'mAP50': 0.0, 'mAP50-95': 0.00836}

Test eval:


Eval: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [01:47<00:00,  2.14s/it]


  -> Det: 400, GT: 317, Matched: 0
Test: {'Box-P': 0, 'Box-R': 0, 'Box-F1': 0, 'mAP50': 0.0, 'mAP50-95': 0.00924}

Epoch 3/10


Train:  44%|████████████████████████████████████████████████▏                                                            | 150/339 [13:40<18:17,  5.81s/it]

In [1]:
# Installer PyTorch avec CUDA 12.8 dans l'environnement actuel
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128 --upgrade


Looking in indexes: https://download.pytorch.org/whl/cu128


In [2]:
import os
import json
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import DetrForObjectDetection, DetrImageProcessor
from tqdm import tqdm
import torch.optim as optim
from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd
from torchvision.ops import box_iou

# =========================
# CONFIGURATION
# =========================
#  FORCER CPU POUR ÉVITER L'ERREUR CUDA
DEVICE = "cpu"
print(f"DEVICE: {DEVICE}")

IMAGES_DIR = "./Data2/train/images"
LABELS_YOLO_DIR = "./Data2/train/labels"
LABELS_COCO_DIR = "./Data2/train/labels_coco"

TEST_IMAGES_DIR = "./Data2/test/images"
TEST_LABELS_YOLO_DIR = "./Data2/test/labels"
TEST_LABELS_COCO_DIR = "./Data2/test/labels_coco"

SAVE_DIR = "./models_detr"

BATCH_SIZE = 2
EPOCHS = 10
LR = 1e-4
NUM_CLASSES = 5  #  5 classes comme dans tes données

os.makedirs(LABELS_COCO_DIR, exist_ok=True)
os.makedirs(TEST_LABELS_COCO_DIR, exist_ok=True)
os.makedirs(SAVE_DIR, exist_ok=True)

# =========================
# CONVERSION YOLO -> COCO (FORMAT DETR)
# =========================
def convert_yolo_to_coco(images_dir, labels_yolo_dir, labels_coco_dir):
    print(f"Conversion YOLO -> COCO pour {images_dir}...")
    converted = 0
    
    for label_file in os.listdir(labels_yolo_dir):
        if not label_file.endswith(".txt"):
            continue
        
        # Essayer différentes extensions d'image
        base_name = label_file.replace(".txt", "")
        image_file = None
        for ext in [".jpg", ".jpeg", ".png"]:
            test_path = os.path.join(images_dir, base_name + ext)
            if os.path.exists(test_path):
                image_file = base_name + ext
                break
        
        if image_file is None:
            continue
        
        image_path = os.path.join(images_dir, image_file)
        img = Image.open(image_path)
        width, height = img.size

        yolo_path = os.path.join(labels_yolo_dir, label_file)
        coco_path = os.path.join(labels_coco_dir, label_file.replace(".txt", ".json"))
        
        boxes, labels = [], []

        with open(yolo_path, "r") as f:
            lines = f.readlines()
            for line in lines:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                
                cls, x_center, y_center, w, h = map(float, parts)
                
                #  Validation
                if not (0 <= x_center <= 1 and 0 <= y_center <= 1 and 
                        0 < w <= 1 and 0 < h <= 1):
                    continue
                
                if not (0 <= int(cls) < NUM_CLASSES):
                    continue
                
                #  GARDER FORMAT CXCYWH NORMALISÉ (comme YOLO)
                boxes.append([x_center, y_center, w, h])
                labels.append(int(cls))

        target = {
            "boxes": boxes,
            "class_labels": labels
        }

        with open(coco_path, "w") as f:
            json.dump(target, f)
        
        converted += 1
    
    print(f" {converted} fichiers convertis")

# Convertir train et test
convert_yolo_to_coco(IMAGES_DIR, LABELS_YOLO_DIR, LABELS_COCO_DIR)
convert_yolo_to_coco(TEST_IMAGES_DIR, TEST_LABELS_YOLO_DIR, TEST_LABELS_COCO_DIR)

# =========================
# DATASET
# =========================
class DetrDataset(Dataset):
    def __init__(self, images_dir, labels_dir):
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        
        # Lister les images
        all_images = []
        for ext in [".jpg", ".jpeg", ".png"]:
            all_images.extend([f for f in os.listdir(images_dir) if f.lower().endswith(ext)])
        
        self.images = sorted(list(set(all_images)))
        print(f"Dataset: {len(self.images)} images trouvées")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.images_dir, img_name)
        
        base_name = os.path.splitext(img_name)[0]
        label_path = os.path.join(self.labels_dir, base_name + ".json")

        image = Image.open(img_path).convert("RGB")

        with open(label_path, "r") as f:
            target = json.load(f)

        # Convertir en tensors
        target = {
            "boxes": torch.tensor(target["boxes"], dtype=torch.float32) if target["boxes"] else torch.zeros((0, 4), dtype=torch.float32),
            "class_labels": torch.tensor(target["class_labels"], dtype=torch.int64) if target["class_labels"] else torch.zeros((0,), dtype=torch.int64)
        }

        return image, target

# =========================
# DATALOADER
# =========================
def collate_fn(batch):
    images, targets = zip(*batch)
    return list(images), list(targets)

train_dataset = DetrDataset(IMAGES_DIR, LABELS_COCO_DIR)
test_dataset = DetrDataset(TEST_IMAGES_DIR, TEST_LABELS_COCO_DIR)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

# =========================
# MODELE
# =========================
processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")

#  CORRECTION: utiliser id2label et label2id + ignore_mismatched_sizes
id2label = {i: f"class{i}" for i in range(NUM_CLASSES)}
label2id = {f"class{i}": i for i in range(NUM_CLASSES)}

model = DetrForObjectDetection.from_pretrained(
    "facebook/detr-resnet-50",
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True  #  IMPORTANT
)
model.to(DEVICE)

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

# =========================
# HELPER FUNCTIONS
# =========================
def cxcywh_to_xyxy(boxes, img_w, img_h):
    if boxes.numel() == 0:
        return torch.zeros((0, 4), dtype=torch.float32)
    cx = boxes[:, 0] * img_w
    cy = boxes[:, 1] * img_h
    w = boxes[:, 2] * img_w
    h = boxes[:, 3] * img_h
    x1 = cx - w/2
    y1 = cy - h/2
    x2 = cx + w/2
    y2 = cy + h/2
    return torch.stack([x1, y1, x2, y2], dim=1)

# =========================
# TRAIN
# =========================
def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0
    n_batches = 0

    for images, targets in tqdm(loader, desc="Train"):
        encoding = processor(images=images, return_tensors="pt")
        pixel_values = encoding["pixel_values"].to(DEVICE)
        pixel_mask = encoding["pixel_mask"].to(DEVICE)

        # Filtrer annotations vides
        labels = []
        for t in targets:
            if t["class_labels"].numel() > 0:
                labels.append({
                    "class_labels": t["class_labels"].to(DEVICE),
                    "boxes": t["boxes"].to(DEVICE)
                })
        
        if len(labels) == 0:
            continue
        
        if len(labels) < len(images):
            pixel_values = pixel_values[:len(labels)]
            pixel_mask = pixel_mask[:len(labels)]

        optimizer.zero_grad()
        outputs = model(pixel_values=pixel_values, pixel_mask=pixel_mask, labels=labels)
        loss = outputs.loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.1)
        optimizer.step()

        total_loss += loss.item()
        n_batches += 1

    return total_loss / max(n_batches, 1)

# =========================
# EVAL (CORRECTION: boxes déjà en format xyxy)
# =========================
def evaluate(model, loader, conf_threshold=0.1):
    model.eval()
    all_gt = []
    all_pred = []
    all_ious = []
    total_det = 0
    total_gt = 0

    with torch.no_grad():
        for images, targets in tqdm(loader, desc="Eval"):
            encoding = processor(images=images, return_tensors="pt")
            pixel_values = encoding["pixel_values"].to(DEVICE)
            pixel_mask = encoding["pixel_mask"].to(DEVICE)

            outputs = model(pixel_values=pixel_values, pixel_mask=pixel_mask)
            
            target_sizes = torch.tensor([img.size[::-1] for img in images])
            results = processor.post_process_object_detection(
                outputs, target_sizes=target_sizes, threshold=conf_threshold
            )

            for i, res in enumerate(results):
                #  Les prédictions sont déjà en format xyxy absolu
                pred_boxes = res["boxes"].clone() if len(res["boxes"]) > 0 else torch.zeros((0, 4), dtype=torch.float32)
                pred_labels = res["labels"].clone() if len(res["labels"]) > 0 else torch.zeros((0,), dtype=torch.int64)
                
                #  Convertir GT de cxcywh normalisé vers xyxy absolu
                gt_boxes_norm = targets[i]["boxes"]
                gt_labels = targets[i]["class_labels"]
                
                total_det += len(pred_labels)
                total_gt += len(gt_labels)
                
                if len(pred_boxes) == 0 or len(gt_boxes_norm) == 0:
                    continue
                
                # Convertir GT vers xyxy absolu
                img_w, img_h = images[i].size
                gt_boxes = cxcywh_to_xyxy(gt_boxes_norm, img_w, img_h)
                
                iou = box_iou(gt_boxes, pred_boxes)
                max_iou, best_idx = iou.max(dim=1)
                all_ious.extend(max_iou.cpu().numpy().tolist())
                
                matched = max_iou >= 0.5
                if matched.sum() > 0:
                    all_gt.extend(gt_labels[matched].cpu().numpy().tolist())
                    all_pred.extend(pred_labels[best_idx[matched]].cpu().numpy().tolist())

    if len(all_gt) > 0 and len(all_pred) > 0:
        prec = precision_score(all_gt, all_pred, average="macro", zero_division=0)
        rec = recall_score(all_gt, all_pred, average="macro", zero_division=0)
        f1 = f1_score(all_gt, all_pred, average="macro", zero_division=0)
    else:
        prec, rec, f1 = 0, 0, 0

    map50 = sum(1 for iou in all_ious if iou >= 0.5) / len(all_ious) if all_ious else 0
    map5095 = sum(all_ious) / len(all_ious) if all_ious else 0

    print(f"  -> Det: {total_det}, GT: {total_gt}, Matched: {len(all_gt)}")
    
    return {
        "Box-P": round(prec, 5),
        "Box-R": round(rec, 5),
        "Box-F1": round(f1, 5),
        "mAP50": round(map50, 5),
        "mAP50-95": round(map5095, 5)
    }

# =========================
# MAIN LOOP
# =========================
train_history = []
test_history = []

for epoch in range(1, EPOCHS + 1):
    print(f"\n{'='*50}")
    print(f"Epoch {epoch}/{EPOCHS}")
    print('='*50)
    
    loss = train_one_epoch(model, train_loader, optimizer)
    print(f"Loss: {loss:.4f}")

    print("\nTrain eval:")
    train_metrics = evaluate(model, train_loader)
    print(f"Train: {train_metrics}")

    print("\nTest eval:")
    test_metrics = evaluate(model, test_loader)
    print(f"Test: {test_metrics}")

    torch.save(model.state_dict(), os.path.join(SAVE_DIR, f"detr_epoch{epoch}.pt"))

    train_history.append({"epoch": epoch, "loss": loss, **train_metrics})
    test_history.append({"epoch": epoch, **test_metrics})

pd.DataFrame(train_history).to_csv(os.path.join(SAVE_DIR, "train_history.csv"), index=False)
pd.DataFrame(test_history).to_csv(os.path.join(SAVE_DIR, "test_history.csv"), index=False)

print(f"\n TERMINÉ! Modèles: {SAVE_DIR}")

DEVICE: cpu
Conversion YOLO -> COCO pour ./Data2/train/images...
 677 fichiers convertis
Conversion YOLO -> COCO pour ./Data2/test/images...
 100 fichiers convertis
Dataset: 677 images trouvées
Dataset: 100 images trouvées


Some weights of the model checkpoint at facebook/detr-resnet-50 were not used when initializing DetrForObjectDetection: ['model.backbone.conv_encoder.model.layer1.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer2.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer3.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer4.0.downsample.1.num_batches_tracked']
- This IS expected if you are initializing DetrForObjectDetection from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DetrForObjectDetection from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of DetrForObjectDetection were not initialized from the model checkpoin


Epoch 1/10


Train: 100%|█████████████████████████████████████████████████████████████████████████| 339/339 [33:48<00:00,  5.98s/it]


Loss: 3.7665

Train eval:


...

Loss: 1.8468

Train eval:


Eval: 100%|██████████████████████████████████████████████████████████████████████████| 339/339 [12:31<00:00,  2.22s/it]


  -> Det: 28327, GT: 2100, Matched: 1383
Train: {'Box-P': 0.90757, 'Box-R': 0.88833, 'Box-F1': 0.8952, 'mAP50': 0.65857, 'mAP50-95': 0.54532}

Test eval:


Eval: 100%|████████████████████████████████████████████████████████████████████████████| 50/50 [01:52<00:00,  2.25s/it]


  -> Det: 4208, GT: 317, Matched: 189
Test: {'Box-P': 0.97774, 'Box-R': 0.85727, 'Box-F1': 0.90486, 'mAP50': 0.59621, 'mAP50-95': 0.51799}

 TERMINÉ! Modèles: ./models_detr


# test Detr

In [3]:
import os
import json
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import DetrForObjectDetection, DetrImageProcessor
from tqdm import tqdm
import torch.optim as optim
from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd
from torchvision.ops import box_iou
import numpy as np

# =========================
# CONFIGURATION
# =========================
DEVICE = "cpu"
print(f"DEVICE: {DEVICE}")

TEST_IMAGES_DIR = "./Data2/test/images"
TEST_LABELS_YOLO_DIR = "./Data2/test/labels"
TEST_LABELS_COCO_DIR = "./Data2/test/labels_coco"
MODELS_DIR = "./models_detr"

BATCH_SIZE = 2
NUM_CLASSES = 5

os.makedirs(TEST_LABELS_COCO_DIR, exist_ok=True)

# =========================
# CONVERSION YOLO -> COCO
# =========================
def convert_yolo_to_coco(images_dir, labels_yolo_dir, labels_coco_dir):
    print(f"Conversion YOLO -> COCO pour {images_dir}...")
    converted = 0
    
    for label_file in os.listdir(labels_yolo_dir):
        if not label_file.endswith(".txt"):
            continue
        
        base_name = label_file.replace(".txt", "")
        image_file = None
        for ext in [".jpg", ".jpeg", ".png"]:
            test_path = os.path.join(images_dir, base_name + ext)
            if os.path.exists(test_path):
                image_file = base_name + ext
                break
        
        if image_file is None:
            continue
        
        image_path = os.path.join(images_dir, image_file)
        img = Image.open(image_path)
        width, height = img.size

        yolo_path = os.path.join(labels_yolo_dir, label_file)
        coco_path = os.path.join(labels_coco_dir, label_file.replace(".txt", ".json"))
        
        boxes, labels = [], []

        with open(yolo_path, "r") as f:
            lines = f.readlines()
            for line in lines:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                
                cls, x_center, y_center, w, h = map(float, parts)
                
                if not (0 <= x_center <= 1 and 0 <= y_center <= 1 and 
                        0 < w <= 1 and 0 < h <= 1):
                    continue
                
                if not (0 <= int(cls) < NUM_CLASSES):
                    continue
                
                boxes.append([x_center, y_center, w, h])
                labels.append(int(cls))

        target = {"boxes": boxes, "class_labels": labels}

        with open(coco_path, "w") as f:
            json.dump(target, f)
        
        converted += 1
    
    print(f" {converted} fichiers convertis")

# Convertir test si pas déjà fait
if not os.path.exists(TEST_LABELS_COCO_DIR) or len(os.listdir(TEST_LABELS_COCO_DIR)) == 0:
    convert_yolo_to_coco(TEST_IMAGES_DIR, TEST_LABELS_YOLO_DIR, TEST_LABELS_COCO_DIR)
else:
    print("Labels COCO déjà convertis")

# =========================
# DATASET
# =========================
class DetrDataset(Dataset):
    def __init__(self, images_dir, labels_dir):
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        
        all_images = []
        for ext in [".jpg", ".jpeg", ".png"]:
            all_images.extend([f for f in os.listdir(images_dir) if f.lower().endswith(ext)])
        
        self.images = sorted(list(set(all_images)))
        print(f"Dataset: {len(self.images)} images")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.images_dir, img_name)
        
        base_name = os.path.splitext(img_name)[0]
        label_path = os.path.join(self.labels_dir, base_name + ".json")

        image = Image.open(img_path).convert("RGB")

        with open(label_path, "r") as f:
            target = json.load(f)

        target = {
            "boxes": torch.tensor(target["boxes"], dtype=torch.float32) if target["boxes"] else torch.zeros((0, 4), dtype=torch.float32),
            "class_labels": torch.tensor(target["class_labels"], dtype=torch.int64) if target["class_labels"] else torch.zeros((0,), dtype=torch.int64)
        }

        return image, target

def collate_fn(batch):
    images, targets = zip(*batch)
    return list(images), list(targets)

test_dataset = DetrDataset(TEST_IMAGES_DIR, TEST_LABELS_COCO_DIR)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

# =========================
# CHARGER LE MEILLEUR MODÈLE
# =========================
print("\n Analyse des modèles sauvegardés...")

# Lire l'historique pour trouver le meilleur epoch
history_path = os.path.join(MODELS_DIR, "test_history.csv")
if os.path.exists(history_path):
    df_history = pd.read_csv(history_path)
    print("\nHistorique des époques:")
    print(df_history[['epoch', 'Box-F1', 'mAP50', 'mAP50-95']])
    
    # Trouver meilleur epoch basé sur F1 score
    best_epoch = df_history.loc[df_history['Box-F1'].idxmax(), 'epoch']
    best_f1 = df_history.loc[df_history['Box-F1'].idxmax(), 'Box-F1']
    
    print(f"\n Meilleur epoch: {int(best_epoch)} (F1={best_f1:.4f})")
else:
    print(" Historique non trouvé, utilisation du dernier modèle")
    # Trouver le dernier epoch
    model_files = [f for f in os.listdir(MODELS_DIR) if f.startswith("detr_epoch") and f.endswith(".pt")]
    epochs = [int(f.split("epoch")[1].split(".pt")[0]) for f in model_files]
    best_epoch = max(epochs) if epochs else 1

best_model_path = os.path.join(MODELS_DIR, f"detr_epoch{int(best_epoch)}.pt")

# Charger le modèle
processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")
id2label = {i: f"class{i}" for i in range(NUM_CLASSES)}
label2id = {f"class{i}": i for i in range(NUM_CLASSES)}

model = DetrForObjectDetection.from_pretrained(
    "facebook/detr-resnet-50",
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
model.to(DEVICE)
model.eval()

print(f" Modèle chargé: {best_model_path}")

# =========================
# HELPER
# =========================
def cxcywh_to_xyxy(boxes, img_w, img_h):
    if boxes.numel() == 0:
        return torch.zeros((0, 4), dtype=torch.float32)
    cx = boxes[:, 0] * img_w
    cy = boxes[:, 1] * img_h
    w = boxes[:, 2] * img_w
    h = boxes[:, 3] * img_h
    x1 = cx - w/2
    y1 = cy - h/2
    x2 = cx + w/2
    y2 = cy + h/2
    return torch.stack([x1, y1, x2, y2], dim=1)

# =========================
# ÉVALUATION DÉTAILLÉE
# =========================
def evaluate_detailed(model, loader, conf_threshold=0.1):
    model.eval()
    
    # Métriques globales
    all_gt = []
    all_pred = []
    all_ious = []
    total_det = 0
    total_gt = 0
    
    # Métriques par classe
    class_tp = {i: 0 for i in range(NUM_CLASSES)}
    class_fp = {i: 0 for i in range(NUM_CLASSES)}
    class_fn = {i: 0 for i in range(NUM_CLASSES)}
    class_gt_count = {i: 0 for i in range(NUM_CLASSES)}

    with torch.no_grad():
        for images, targets in tqdm(loader, desc="Test"):
            encoding = processor(images=images, return_tensors="pt")
            pixel_values = encoding["pixel_values"].to(DEVICE)
            pixel_mask = encoding["pixel_mask"].to(DEVICE)

            outputs = model(pixel_values=pixel_values, pixel_mask=pixel_mask)
            
            target_sizes = torch.tensor([img.size[::-1] for img in images])
            results = processor.post_process_object_detection(
                outputs, target_sizes=target_sizes, threshold=conf_threshold
            )

            for i, res in enumerate(results):
                pred_boxes = res["boxes"].clone() if len(res["boxes"]) > 0 else torch.zeros((0, 4), dtype=torch.float32)
                pred_labels = res["labels"].clone() if len(res["labels"]) > 0 else torch.zeros((0,), dtype=torch.int64)
                
                gt_boxes_norm = targets[i]["boxes"]
                gt_labels = targets[i]["class_labels"]
                
                # Compter GT par classe
                for cls in gt_labels.tolist():
                    class_gt_count[cls] += 1
                
                total_det += len(pred_labels)
                total_gt += len(gt_labels)
                
                if len(pred_boxes) == 0 or len(gt_boxes_norm) == 0:
                    # Compter FN si pas de prédictions
                    if len(pred_boxes) == 0 and len(gt_boxes_norm) > 0:
                        for cls in gt_labels.tolist():
                            class_fn[cls] += 1
                    # Compter FP si pas de GT
                    if len(gt_boxes_norm) == 0 and len(pred_boxes) > 0:
                        for cls in pred_labels.tolist():
                            class_fp[cls] += 1
                    continue
                
                img_w, img_h = images[i].size
                gt_boxes = cxcywh_to_xyxy(gt_boxes_norm, img_w, img_h)
                
                iou = box_iou(gt_boxes, pred_boxes)
                max_iou, best_idx = iou.max(dim=1)
                all_ious.extend(max_iou.cpu().numpy().tolist())
                
                # Matching avec IoU >= 0.5
                matched_gt = set()
                matched_pred = set()
                
                for gt_idx, (iou_val, pred_idx) in enumerate(zip(max_iou, best_idx)):
                    if iou_val >= 0.5:
                        gt_cls = gt_labels[gt_idx].item()
                        pred_cls = pred_labels[pred_idx].item()
                        
                        if gt_cls == pred_cls:
                            class_tp[gt_cls] += 1
                            matched_gt.add(gt_idx)
                            matched_pred.add(pred_idx.item())
                            
                            all_gt.append(gt_cls)
                            all_pred.append(pred_cls)
                
                # Compter FN (GT non matchés)
                for gt_idx, gt_cls in enumerate(gt_labels.tolist()):
                    if gt_idx not in matched_gt:
                        class_fn[gt_cls] += 1
                
                # Compter FP (prédictions non matchées)
                for pred_idx, pred_cls in enumerate(pred_labels.tolist()):
                    if pred_idx not in matched_pred:
                        class_fp[pred_cls] += 1

    # Calcul métriques globales
    if len(all_gt) > 0 and len(all_pred) > 0:
        prec = precision_score(all_gt, all_pred, average="macro", zero_division=0)
        rec = recall_score(all_gt, all_pred, average="macro", zero_division=0)
        f1 = f1_score(all_gt, all_pred, average="macro", zero_division=0)
    else:
        prec, rec, f1 = 0, 0, 0

    map50 = sum(1 for iou in all_ious if iou >= 0.5) / len(all_ious) if all_ious else 0
    map5095 = sum(all_ious) / len(all_ious) if all_ious else 0

    # Métriques par classe
    class_metrics = []
    for cls in range(NUM_CLASSES):
        tp = class_tp[cls]
        fp = class_fp[cls]
        fn = class_fn[cls]
        
        prec_cls = tp / (tp + fp) if (tp + fp) > 0 else 0
        rec_cls = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1_cls = 2 * prec_cls * rec_cls / (prec_cls + rec_cls) if (prec_cls + rec_cls) > 0 else 0
        
        class_metrics.append({
            "Class": f"class{cls}",
            "GT Count": class_gt_count[cls],
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "Precision": round(prec_cls, 4),
            "Recall": round(rec_cls, 4),
            "F1": round(f1_cls, 4)
        })

    print(f"\n Résultats globaux:")
    print(f"  Total détections: {total_det}")
    print(f"  Total GT: {total_gt}")
    print(f"  Matchés (TP): {len(all_gt)}")
    
    return {
        "Box-P": round(prec, 5),
        "Box-R": round(rec, 5),
        "Box-F1": round(f1, 5),
        "mAP50": round(map50, 5),
        "mAP50-95": round(map5095, 5)
    }, class_metrics

# =========================
# TEST FINAL
# =========================
print("\n" + "="*50)
print("ÉVALUATION FINALE SUR TEST SET")
print("="*50)

test_metrics, class_metrics = evaluate_detailed(model, test_loader, conf_threshold=0.1)

print(f"\n Métriques globales:")
print(f"  Precision: {test_metrics['Box-P']:.4f}")
print(f"  Recall:    {test_metrics['Box-R']:.4f}")
print(f"  F1-Score:  {test_metrics['Box-F1']:.4f}")
print(f"  mAP50:     {test_metrics['mAP50']:.4f}")
print(f"  mAP50-95:  {test_metrics['mAP50-95']:.4f}")

print(f"\n Métriques par classe:")
df_class = pd.DataFrame(class_metrics)
print(df_class.to_string(index=False))

# Sauvegarder les résultats
results_path = os.path.join(MODELS_DIR, "test_final_results.csv")
df_class.to_csv(results_path, index=False)

summary_path = os.path.join(MODELS_DIR, "test_final_summary.csv")
pd.DataFrame([test_metrics]).to_csv(summary_path, index=False)

print(f"\n Résultats sauvegardés:")
print(f"  - {results_path}")
print(f"  - {summary_path}")
print("="*50)

DEVICE: cpu
Labels COCO déjà convertis
Dataset: 100 images

 Analyse des modèles sauvegardés...

Historique des époques:
   epoch   Box-F1    mAP50  mAP50-95
0      1  0.39254  0.30599   0.35781
1      2  0.51280  0.29022   0.36797
2      3  0.72632  0.41640   0.41762
3      4  0.56910  0.34069   0.36039
4      5  0.57302  0.24921   0.29131
5      6  0.62473  0.21136   0.28175
6      7  0.72233  0.22713   0.30755
7      8  0.78117  0.47950   0.46503
8      9  0.72987  0.63407   0.55190
9     10  0.90486  0.59621   0.51799

 Meilleur epoch: 10 (F1=0.9049)


<env>\lib\site-packages\torch\nn\modules\module.py:2446: UserWarning: for conv1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  warnings.warn(
<env>\lib\site-packages\torch\nn\modules\module.py:2446: UserWarning: for bn1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  warnings.warn(
<env>\lib\site-packages\torch\nn\modules\module.py:2446: UserWarning: for bn1.bias: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dic

 Modèle chargé: ./models_detr\detr_epoch10.pt

ÉVALUATION FINALE SUR TEST SET


Test: 100%|████████████████████████████████████████████████████████████████████████████| 50/50 [01:47<00:00,  2.15s/it]


 Résultats globaux:
  Total détections: 4208
  Total GT: 317
  Matchés (TP): 181

 Métriques globales:
  Precision: 1.0000
  Recall:    1.0000
  F1-Score:  1.0000
  mAP50:     0.5962
  mAP50-95:  0.5180

 Métriques par classe:
 Class  GT Count  TP   FP  FN  Precision  Recall     F1
class0       100  45 1735  55     0.0253  0.4500 0.0479
class1        22   3  186  19     0.0159  0.1364 0.0284
class2        59  39  430  20     0.0832  0.6610 0.1477
class3        36   4  267  32     0.0148  0.1111 0.0261
class4       100  90 1409  10     0.0600  0.9000 0.1126

 Résultats sauvegardés:
  - ./models_detr\test_final_results.csv
  - ./models_detr\test_final_summary.csv


In [22]:
import pandas as pd
import numpy as np

# =========================
# FONCTIONS UTILITAIRES
# =========================
def format_with_delta(test_value, train_value):
    """Formate une valeur avec son delta par rapport au train"""
    delta = test_value - train_value
    if abs(delta) < 0.001:
        return f"{test_value:.3f}"
    elif delta > 0:
        return f"{test_value:.3f}(+{delta:.3f})"
    else:
        return f"{test_value:.3f}({delta:.3f})"

def get_color_for_delta(delta):
    """Retourne la couleur selon le delta"""
    if delta > 0.01:
        return ""  # Vert pour amélioration
    elif delta < -0.01:
        return ""  # Rouge pour détérioration
    else:
        return ""  # Blanc pour stable

# =========================
# DONNÉES RÉELLES DETR
# =========================

# DONNÉES GLOBALES TRAIN RÉELLES
global_metrics_train = {
    'Precision': 0.90757,
    'Recall': 0.88833,
    'F1-Score': 0.89520,
    'mAP50': 0.65857,
    'mAP50-95': 0.54532
}

# DONNÉES GLOBALES TEST RÉELLES  
global_metrics_test = {
    'Precision': 0.97774,
    'Recall': 0.85727,
    'F1-Score': 0.90486,
    'mAP50': 0.59621,
    'mAP50-95': 0.51799
}

# =========================
# TABLEAU GLOBAL COMPARATIF
# =========================
print("\n" + "="*80)
print(" DETR - MÉTRIQUES GLOBALES TRAIN vs TEST")
print("="*80)

global_comparison_data = []
for metric in global_metrics_train.keys():
    train_value = global_metrics_train[metric]
    test_value = global_metrics_test[metric]
    delta = test_value - train_value
    
    global_comparison_data.append({
        'Métrique': metric,
        'Train': f"{train_value:.5f}",
        'Test': format_with_delta(test_value, train_value),
        'Δ': f"{delta:+.5f}",
        'Variation': get_color_for_delta(delta),
        'Tendance': "" if delta > 0.01 else "" if delta < -0.01 else ""
    })

df_global_comparison = pd.DataFrame(global_comparison_data)
print(df_global_comparison.to_string(index=False))

# =========================
# TABLEAU SYNTHÈSE SIMPLIFIÉ
# =========================
print("\n" + "="*60)
print(" DETR - SYNTHÈSE DES VARIATIONS")
print("="*60)

synthesis_data = []
for metric in global_metrics_train.keys():
    train_value = global_metrics_train[metric]
    test_value = global_metrics_test[metric]
    delta = test_value - train_value
    
    synthesis_data.append({
        'Métrique': metric,
        'Train → Test': f"{train_value:.3f} → {test_value:.3f}",
        'Variation': format_with_delta(test_value, train_value),
        'Statut': get_color_for_delta(delta)
    })

df_synthesis = pd.DataFrame(synthesis_data)
print(df_synthesis.to_string(index=False))

# =========================
# ANALYSE RAPIDE
# =========================
print("\n" + "="*50)
print(" ANALYSE RAPIDE")
print("="*50)

print(" Améliorations:")
for metric in global_metrics_train.keys():
    delta = global_metrics_test[metric] - global_metrics_train[metric]
    if delta > 0.01:
        print(f"    {metric}: +{delta:.3f}")

print("\n Détériorations:")
for metric in global_metrics_train.keys():
    delta = global_metrics_test[metric] - global_metrics_train[metric]
    if delta < -0.01:
        print(f"    {metric}: {delta:.3f}")

print("\n Stable:")
for metric in global_metrics_train.keys():
    delta = global_metrics_test[metric] - global_metrics_train[metric]
    if -0.01 <= delta <= 0.01:
        print(f"    {metric}: {delta:+.3f}")


 DETR - MÉTRIQUES GLOBALES TRAIN vs TEST
 Métrique   Train          Test        Δ Variation Tendance
Precision 0.90757 0.978(+0.070) +0.07017                 
   Recall 0.88833 0.857(-0.031) -0.03106                 
 F1-Score 0.89520 0.905(+0.010) +0.00966                
    mAP50 0.65857 0.596(-0.062) -0.06236                 
 mAP50-95 0.54532 0.518(-0.027) -0.02733                 

 DETR - SYNTHÈSE DES VARIATIONS
 Métrique  Train → Test     Variation Statut
Precision 0.908 → 0.978 0.978(+0.070)      
   Recall 0.888 → 0.857 0.857(-0.031)      
 F1-Score 0.895 → 0.905 0.905(+0.010)      
    mAP50 0.659 → 0.596 0.596(-0.062)      
 mAP50-95 0.545 → 0.518 0.518(-0.027)      

 ANALYSE RAPIDE
 Améliorations:
    Precision: +0.070

 Détériorations:
    Recall: -0.031
    mAP50: -0.062
    mAP50-95: -0.027

 Stable:
    F1-Score: +0.010


In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =========================
# FONCTIONS UTILITAIRES
# =========================
def format_with_delta(test_value, train_value):
    """Formate une valeur avec son delta par rapport au train"""
    delta = test_value - train_value
    if abs(delta) < 0.001:
        return f"{test_value:.3f}"
    elif delta > 0:
        return f"{test_value:.3f}(+{delta:.3f})"
    else:
        return f"{test_value:.3f}({delta:.3f})"

def get_color_for_delta(delta):
    """Retourne la couleur selon le delta"""
    if delta > 0.01:
        return ""  # Vert pour amélioration
    elif delta < -0.01:
        return ""  # Rouge pour détérioration
    else:
        return ""  # Blanc pour stable

# =========================
# DONNÉES RÉELLES DETR
# =========================

# DONNÉES GLOBALES TRAIN RÉELLES
global_metrics_train = {
    'Precision': 0.90757,
    'Recall': 0.88833,
    'F1-Score': 0.89520,
    'mAP50': 0.65857,
    'mAP50-95': 0.54532
}

# DONNÉES GLOBALES TEST RÉELLES  
global_metrics_test = {
    'Precision': 0.97774,
    'Recall': 0.85727,
    'F1-Score': 0.90486,
    'mAP50': 0.59621,
    'mAP50-95': 0.51799
}

# =========================
# TABLEAU GLOBAL COMPARATIF
# =========================
print("\n" + "="*80)
print(" DETR - MÉTRIQUES GLOBALES TRAIN vs TEST")
print("="*80)

global_comparison_data = []
for metric in global_metrics_train.keys():
    train_value = global_metrics_train[metric]
    test_value = global_metrics_test[metric]
    delta = test_value - train_value
    
    global_comparison_data.append({
        'Métrique': metric,
        'Train': f"{train_value:.5f}",
        'Test': format_with_delta(test_value, train_value),
        'Δ': f"{delta:+.5f}",
        'Variation': get_color_for_delta(delta),
        'Tendance': "" if delta > 0.01 else "" if delta < -0.01 else ""
    })

df_global_comparison = pd.DataFrame(global_comparison_data)
print(df_global_comparison.to_string(index=False))

# =========================
# SCHÉMA VISUEL - GRAPHIQUE COMPARATIF
# =========================
plt.figure(figsize=(14, 10))

# Données pour le graphique
metrics = list(global_metrics_train.keys())
train_values = [global_metrics_train[m] for m in metrics]
test_values = [global_metrics_test[m] for m in metrics]
deltas = [test_values[i] - train_values[i] for i in range(len(metrics))]

# Créer le graphique
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 12))

# Graphique 1: Barres comparatives Train vs Test
x = np.arange(len(metrics))
width = 0.35

bars1 = ax1.bar(x - width/2, train_values, width, label='TRAIN', 
                color='#1f77b4', alpha=0.8, edgecolor='black')
bars2 = ax1.bar(x + width/2, test_values, width, label='TEST', 
                color='#ff7f0e', alpha=0.8, edgecolor='black')

ax1.set_xlabel('Métriques', fontsize=14, fontweight='bold')
ax1.set_ylabel('Score', fontsize=14, fontweight='bold')
ax1.set_title('DETR - Comparaison Globale TRAIN vs TEST', fontsize=16, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(metrics, rotation=45, ha='right')
ax1.legend(fontsize=12)
ax1.grid(axis='y', alpha=0.3)
ax1.set_ylim(0, 1.1)

# Ajouter les valeurs sur les barres
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{height:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Graphique 2: Deltas (variations)
colors = ['green' if delta > 0.01 else 'red' if delta < -0.01 else 'gray' for delta in deltas]
bars3 = ax2.bar(metrics, deltas, color=colors, alpha=0.8, edgecolor='black')

ax2.axhline(y=0, color='black', linestyle='-', alpha=0.8, linewidth=1)
ax2.set_xlabel('Métriques', fontsize=14, fontweight='bold')
ax2.set_ylabel('Δ (Test - Train)', fontsize=14, fontweight='bold')
ax2.set_title('Variations des Métriques (TEST vs TRAIN)', fontsize=16, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(metrics, rotation=45, ha='right')
ax2.grid(axis='y', alpha=0.3)

# Ajouter les valeurs sur les barres de delta
for i, bar in enumerate(bars3):
    height = bar.get_height()
    va = 'bottom' if height >= 0 else 'top'
    y_pos = height + 0.005 if height >= 0 else height - 0.005
    color = 'darkgreen' if height > 0.01 else 'darkred' if height < -0.01 else 'black'
    
    ax2.text(bar.get_x() + bar.get_width()/2., y_pos,
            f'{height:+.3f}', ha='center', va=va, fontsize=11, 
            fontweight='bold', color=color)

# Ajuster l'espacement
plt.tight_layout()

# Sauvegarder le schéma
plt.savefig('detr_global_comparison_chart.png', dpi=300, bbox_inches='tight')
plt.show()

# =========================
# TABLEAU SYNTHÈSE SIMPLIFIÉ
# =========================
print("\n" + "="*60)
print(" DETR - SYNTHÈSE DES VARIATIONS")
print("="*60)

synthesis_data = []
for metric in global_metrics_train.keys():
    train_value = global_metrics_train[metric]
    test_value = global_metrics_test[metric]
    delta = test_value - train_value
    
    synthesis_data.append({
        'Métrique': metric,
        'Train → Test': f"{train_value:.3f} → {test_value:.3f}",
        'Variation': format_with_delta(test_value, train_value),
        'Statut': get_color_for_delta(delta)
    })

df_synthesis = pd.DataFrame(synthesis_data)
print(df_synthesis.to_string(index=False))

# =========================
# SCHÉMA 2: GRAPHIQUE EN RADAR
# =========================
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, polar=True)

# Préparer les données pour le radar
categories = metrics + [metrics[0]]  # Fermer le cercle
train_radar = train_values + [train_values[0]]
test_radar = test_values + [test_values[0]]

# Angles pour chaque métrique
angles = np.linspace(0, 2*np.pi, len(categories), endpoint=False).tolist()
angles += angles[:1]  # Fermer le cercle

# Tracer le radar
ax.plot(angles, train_radar, 'o-', linewidth=2, label='TRAIN', color='#1f77b4')
ax.fill(angles, train_radar, alpha=0.25, color='#1f77b4')
ax.plot(angles, test_radar, 'o-', linewidth=2, label='TEST', color='#ff7f0e')
ax.fill(angles, test_radar, alpha=0.25, color='#ff7f0e')

# Configuration du radar
ax.set_xticks(angles[:-1])
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.0)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.grid(True)
plt.title('DETR - Comparaison Radar TRAIN vs TEST', size=16, fontweight='bold', y=1.08)
plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))

plt.tight_layout()
plt.savefig('detr_radar_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

# =========================
# RÉSUMÉ FINAL
# =========================
print("\n" + "="*50)
print(" RÉSUMÉ DES VARIATIONS")
print("="*50)

print(" AMÉLIORATIONS:")
for metric in global_metrics_train.keys():
    delta = global_metrics_test[metric] - global_metrics_train[metric]
    if delta > 0.01:
        print(f"   {metric}: {format_with_delta(global_metrics_test[metric], global_metrics_train[metric])}")

print("\n DÉTÉRIORATIONS:")
for metric in global_metrics_train.keys():
    delta = global_metrics_test[metric] - global_metrics_train[metric]
    if delta < -0.01:
        print(f"   {metric}: {format_with_delta(global_metrics_test[metric], global_metrics_train[metric])}")

print("\n STABLE:")
for metric in global_metrics_train.keys():
    delta = global_metrics_test[metric] - global_metrics_train[metric]
    if -0.01 <= delta <= 0.01:
        print(f"   {metric}: {format_with_delta(global_metrics_test[metric], global_metrics_train[metric])}")

print(f"\n Schémas générés:")
print(f"    detr_global_comparison_chart.png")
print(f"    detr_radar_comparison.png")


 DETR - MÉTRIQUES GLOBALES TRAIN vs TEST
 Métrique   Train          Test        Δ Variation Tendance
Precision 0.90757 0.978(+0.070) +0.07017                 
   Recall 0.88833 0.857(-0.031) -0.03106                 
 F1-Score 0.89520 0.905(+0.010) +0.00966                
    mAP50 0.65857 0.596(-0.062) -0.06236                 
 mAP50-95 0.54532 0.518(-0.027) -0.02733                 


<Figure size 1400x1000 with 0 Axes>

<Figure size 1400x1200 with 2 Axes>


 DETR - SYNTHÈSE DES VARIATIONS
 Métrique  Train → Test     Variation Statut
Precision 0.908 → 0.978 0.978(+0.070)      
   Recall 0.888 → 0.857 0.857(-0.031)      
 F1-Score 0.895 → 0.905 0.905(+0.010)      
    mAP50 0.659 → 0.596 0.596(-0.062)      
 mAP50-95 0.545 → 0.518 0.518(-0.027)      


ValueError: x and y must have same first dimension, but have shapes (7,) and (6,)

<Figure size 1000x800 with 1 Axes>

# DETR TRAON TEST TABLEAUX 

In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =========================
# FONCTIONS UTILITAIRES
# =========================
def format_with_delta(test_value, train_value):
    """Formate une valeur avec son delta par rapport au train"""
    delta = test_value - train_value
    if abs(delta) < 0.001:
        return f"{test_value:.3f}"
    elif delta > 0:
        return f"{test_value:.3f}(+{delta:.3f})"
    else:
        return f"{test_value:.3f}({delta:.3f})"

def get_color_for_delta(delta):
    """Retourne la couleur selon le delta"""
    if delta > 0.01:
        return ""  # Vert pour amélioration
    elif delta < -0.01:
        return ""  # Rouge pour détérioration
    else:
        return ""  # Blanc pour stable

# =========================
# DONNÉES RÉELLES DETR
# =========================

# DONNÉES GLOBALES TRAIN RÉELLES
global_metrics_train = {
    'Precision': 0.90757,
    'Recall': 0.88833,
    'F1-Score': 0.89520,
    'mAP50': 0.65857,
    'mAP50-95': 0.54532
}

# DONNÉES GLOBALES TEST RÉELLES  
global_metrics_test = {
    'Precision': 0.97774,
    'Recall': 0.85727,
    'F1-Score': 0.90486,
    'mAP50': 0.59621,
    'mAP50-95': 0.51799
}

# =========================
# TABLEAU GLOBAL COMPARATIF
# =========================
print("\n" + "="*80)
print(" DETR - MÉTRIQUES GLOBALES TRAIN vs TEST")
print("="*80)

global_comparison_data = []
for metric in global_metrics_train.keys():
    train_value = global_metrics_train[metric]
    test_value = global_metrics_test[metric]
    delta = test_value - train_value
    
    global_comparison_data.append({
        'Métrique': metric,
        'Train': f"{train_value:.5f}",
        'Test': format_with_delta(test_value, train_value),
        'Δ': f"{delta:+.5f}",
        'Variation': get_color_for_delta(delta),
        'Tendance': "" if delta > 0.01 else "" if delta < -0.01 else ""
    })

df_global_comparison = pd.DataFrame(global_comparison_data)
print(df_global_comparison.to_string(index=False))

# =========================
# SCHÉMA VISUEL - GRAPHIQUE COMPARATIF
# =========================
plt.figure(figsize=(14, 10))

# Données pour le graphique
metrics = list(global_metrics_train.keys())
train_values = [global_metrics_train[m] for m in metrics]
test_values = [global_metrics_test[m] for m in metrics]
deltas = [test_values[i] - train_values[i] for i in range(len(metrics))]

# Créer le graphique
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 12))

# Graphique 1: Barres comparatives Train vs Test
x = np.arange(len(metrics))
width = 0.35

bars1 = ax1.bar(x - width/2, train_values, width, label='TRAIN', 
                color='#1f77b4', alpha=0.8, edgecolor='black')
bars2 = ax1.bar(x + width/2, test_values, width, label='TEST', 
                color='#ff7f0e', alpha=0.8, edgecolor='black')

ax1.set_xlabel('Métriques', fontsize=14, fontweight='bold')
ax1.set_ylabel('Score', fontsize=14, fontweight='bold')
ax1.set_title('DETR - Comparaison Globale TRAIN vs TEST', fontsize=16, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(metrics, rotation=45, ha='right')
ax1.legend(fontsize=12)
ax1.grid(axis='y', alpha=0.3)
ax1.set_ylim(0, 1.1)

# Ajouter les valeurs sur les barres
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{height:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Graphique 2: Deltas (variations)
colors = ['green' if delta > 0.01 else 'red' if delta < -0.01 else 'gray' for delta in deltas]
bars3 = ax2.bar(metrics, deltas, color=colors, alpha=0.8, edgecolor='black')

ax2.axhline(y=0, color='black', linestyle='-', alpha=0.8, linewidth=1)
ax2.set_xlabel('Métriques', fontsize=14, fontweight='bold')
ax2.set_ylabel('Δ (Test - Train)', fontsize=14, fontweight='bold')
ax2.set_title('Variations des Métriques (TEST vs TRAIN)', fontsize=16, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(metrics, rotation=45, ha='right')
ax2.grid(axis='y', alpha=0.3)

# Ajouter les valeurs sur les barres de delta
for i, bar in enumerate(bars3):
    height = bar.get_height()
    va = 'bottom' if height >= 0 else 'top'
    y_pos = height + 0.005 if height >= 0 else height - 0.005
    color = 'darkgreen' if height > 0.01 else 'darkred' if height < -0.01 else 'black'
    
    ax2.text(bar.get_x() + bar.get_width()/2., y_pos,
            f'{height:+.3f}', ha='center', va=va, fontsize=11, 
            fontweight='bold', color=color)

# Ajuster l'espacement
plt.tight_layout()

# Sauvegarder le schéma
plt.savefig('detr_global_comparison_chart.png', dpi=300, bbox_inches='tight')
plt.show()

# =========================
# TABLEAU SYNTHÈSE SIMPLIFIÉ
# =========================
print("\n" + "="*60)
print(" DETR - SYNTHÈSE DES VARIATIONS")
print("="*60)

synthesis_data = []
for metric in global_metrics_train.keys():
    train_value = global_metrics_train[metric]
    test_value = global_metrics_test[metric]
    delta = test_value - train_value
    
    synthesis_data.append({
        'Métrique': metric,
        'Train → Test': f"{train_value:.3f} → {test_value:.3f}",
        'Variation': format_with_delta(test_value, train_value),
        'Statut': get_color_for_delta(delta)
    })

df_synthesis = pd.DataFrame(synthesis_data)
print(df_synthesis.to_string(index=False))

# =========================
# SCHÉMA 2: GRAPHIQUE EN LIGNE
# =========================
plt.figure(figsize=(12, 8))

# Graphique en ligne pour montrer l'évolution
x = np.arange(len(metrics))
plt.plot(x, train_values, 'o-', linewidth=3, label='TRAIN', color='#1f77b4', markersize=8)
plt.plot(x, test_values, 's-', linewidth=3, label='TEST', color='#ff7f0e', markersize=8)

plt.xlabel('Métriques', fontsize=14, fontweight='bold')
plt.ylabel('Score', fontsize=14, fontweight='bold')
plt.title('DETR - Évolution des Métriques TRAIN → TEST', fontsize=16, fontweight='bold')
plt.xticks(x, metrics, rotation=45, ha='right')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.ylim(0.4, 1.05)

# Ajouter les valeurs sur les points
for i, (train_val, test_val) in enumerate(zip(train_values, test_values)):
    plt.text(i, train_val + 0.02, f'{train_val:.3f}', ha='center', va='bottom', 
             fontsize=9, fontweight='bold', color='#1f77b4')
    plt.text(i, test_val - 0.03, f'{test_val:.3f}', ha='center', va='top', 
             fontsize=9, fontweight='bold', color='#ff7f0e')

plt.tight_layout()
plt.savefig('detr_line_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

# =========================
# SCHÉMA 3: GRAPHIQUE EN BARRES HORIZONTALES
# =========================
plt.figure(figsize=(12, 8))

y_pos = np.arange(len(metrics))
height = 0.35

plt.barh(y_pos - height/2, train_values, height, label='TRAIN', 
         color='#1f77b4', alpha=0.8, edgecolor='black')
plt.barh(y_pos + height/2, test_values, height, label='TEST', 
         color='#ff7f0e', alpha=0.8, edgecolor='black')

plt.xlabel('Score', fontsize=14, fontweight='bold')
plt.ylabel('Métriques', fontsize=14, fontweight='bold')
plt.title('DETR - Comparaison Horizontale TRAIN vs TEST', fontsize=16, fontweight='bold')
plt.yticks(y_pos, metrics)
plt.legend(fontsize=12)
plt.grid(axis='x', alpha=0.3)
plt.xlim(0, 1.1)

# Ajouter les valeurs sur les barres horizontales
for i, (train_val, test_val) in enumerate(zip(train_values, test_values)):
    plt.text(train_val + 0.01, i - height/2, f'{train_val:.3f}', 
             va='center', fontsize=9, fontweight='bold', color='#1f77b4')
    plt.text(test_val + 0.01, i + height/2, f'{test_val:.3f}', 
             va='center', fontsize=9, fontweight='bold', color='#ff7f0e')

plt.tight_layout()
plt.savefig('detr_horizontal_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

# =========================
# RÉSUMÉ FINAL
# =========================
print("\n" + "="*50)
print(" RÉSUMÉ DES VARIATIONS")
print("="*50)

print(" AMÉLIORATIONS:")
for metric in global_metrics_train.keys():
    delta = global_metrics_test[metric] - global_metrics_train[metric]
    if delta > 0.01:
        print(f"   {metric}: {format_with_delta(global_metrics_test[metric], global_metrics_train[metric])}")

print("\n DÉTÉRIORATIONS:")
for metric in global_metrics_train.keys():
    delta = global_metrics_test[metric] - global_metrics_train[metric]
    if delta < -0.01:
        print(f"   {metric}: {format_with_delta(global_metrics_test[metric], global_metrics_train[metric])}")

print("\n STABLE:")
for metric in global_metrics_train.keys():
    delta = global_metrics_test[metric] - global_metrics_train[metric]
    if -0.01 <= delta <= 0.01:
        print(f"   {metric}: {format_with_delta(global_metrics_test[metric], global_metrics_train[metric])}")

print(f"\n Schémas générés:")
print(f"    detr_global_comparison_chart.png")
print(f"    detr_line_comparison.png")
print(f"    detr_horizontal_comparison.png")


 DETR - MÉTRIQUES GLOBALES TRAIN vs TEST
 Métrique   Train          Test        Δ Variation Tendance
Precision 0.90757 0.978(+0.070) +0.07017                 
   Recall 0.88833 0.857(-0.031) -0.03106                 
 F1-Score 0.89520 0.905(+0.010) +0.00966                
    mAP50 0.65857 0.596(-0.062) -0.06236                 
 mAP50-95 0.54532 0.518(-0.027) -0.02733                 


<Figure size 1400x1000 with 0 Axes>

<Figure size 1400x1200 with 2 Axes>


 DETR - SYNTHÈSE DES VARIATIONS
 Métrique  Train → Test     Variation Statut
Precision 0.908 → 0.978 0.978(+0.070)      
   Recall 0.888 → 0.857 0.857(-0.031)      
 F1-Score 0.895 → 0.905 0.905(+0.010)      
    mAP50 0.659 → 0.596 0.596(-0.062)      
 mAP50-95 0.545 → 0.518 0.518(-0.027)      


<Figure size 1200x800 with 1 Axes>

<Figure size 1200x800 with 1 Axes>


 RÉSUMÉ DES VARIATIONS
 AMÉLIORATIONS:
   Precision: 0.978(+0.070)

 DÉTÉRIORATIONS:
   Recall: 0.857(-0.031)
   mAP50: 0.596(-0.062)
   mAP50-95: 0.518(-0.027)

 STABLE:
   F1-Score: 0.905(+0.010)

 Schémas générés:
    detr_global_comparison_chart.png
    detr_line_comparison.png
    detr_horizontal_comparison.png


# DETR TEST SUR LA METEO(glare,night,rain...........)

In [1]:
import shutil
import os

RESULTS_BASE_DIR = "./Data2/results"
WEATHER_CONDITIONS = ["fog", "glare", "night", "rain", "snow"]
LEVELS = [1, 2, 3, 4, 5]

for weather in WEATHER_CONDITIONS:
    for level in LEVELS:
        coco_dir = os.path.join(RESULTS_BASE_DIR, f"{weather}_level{level}", "labels_coco")
        if os.path.exists(coco_dir):
            shutil.rmtree(coco_dir)
            print(f" Supprimé: {coco_dir}")

print("\n Relance maintenant le code d'évaluation!")

 Supprimé: ./Data2/results\fog_level1\labels_coco
 Supprimé: ./Data2/results\fog_level2\labels_coco
 Supprimé: ./Data2/results\fog_level3\labels_coco
 Supprimé: ./Data2/results\fog_level4\labels_coco

 Relance maintenant le code d'évaluation!


In [ ]:
import os
import json
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import DetrForObjectDetection, DetrImageProcessor
from tqdm import tqdm
from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd
from torchvision.ops import box_iou
import glob

# =========================
# CONFIGURATION
# =========================
DEVICE = "cpu"
print(f"DEVICE: {DEVICE}")

RESULTS_BASE_DIR = "./Data2/results"
MODELS_DIR = "./models_detr"
OUTPUT_DIR = "./weather_evaluation"

BATCH_SIZE = 2
NUM_CLASSES = 5

# Conditions météo et leurs niveaux
WEATHER_CONDITIONS = ["fog", "glare", "night", "rain", "snow"]
LEVELS = [1, 2, 3, 4, 5]

os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================
# CONVERSION CSV -> COCO FORMAT
# =========================
def convert_csv_to_coco(images_dir, labels_csv_dir, labels_coco_dir):
    """
    Convertit les labels CSV en format COCO pour DETR
    Détecte automatiquement le format du CSV
    """
    print(f"Conversion CSV -> COCO pour {images_dir}...")
    os.makedirs(labels_coco_dir, exist_ok=True)
    
    # Lire tous les fichiers CSV
    csv_files = glob.glob(os.path.join(labels_csv_dir, "*.csv"))
    
    if len(csv_files) == 0:
        print(f" Aucun fichier CSV trouvé dans {labels_csv_dir}")
        return 0
    
    converted = 0
    
    # Lire le premier CSV pour détecter le format
    first_csv = csv_files[0]
    df_sample = pd.read_csv(first_csv, nrows=5)
    print(f"  Colonnes détectées: {list(df_sample.columns)}")
    
    # Détecter le format
    has_filename = 'filename' in df_sample.columns
    has_image = 'image' in df_sample.columns
    
    for csv_file in csv_files:
        try:
            df = pd.read_csv(csv_file)
            
            # Adapter selon le format
            if has_filename:
                filename_col = 'filename'
            elif has_image:
                filename_col = 'image'
            else:
                # Si pas de colonne filename, le nom du CSV est le nom de l'image
                base_name = os.path.splitext(os.path.basename(csv_file))[0]
                # Chercher l'image correspondante
                img_file = None
                for ext in [".jpg", ".jpeg", ".png"]:
                    test_path = os.path.join(images_dir, base_name + ext)
                    if os.path.exists(test_path):
                        img_file = base_name + ext
                        break
                
                if img_file is None:
                    continue
                
                # Traiter ce fichier comme une seule image
                img_path = os.path.join(images_dir, img_file)
                img = Image.open(img_path)
                img_width, img_height = img.size
                
                boxes = []
                labels = []
                
                for _, row in df.iterrows():
                    try:
                        # Essayer différents formats de colonnes
                        if 'xmin' in df.columns:
                            xmin = float(row['xmin'])
                            ymin = float(row['ymin'])
                            xmax = float(row['xmax'])
                            ymax = float(row['ymax'])
                            # Gérer différents noms de colonnes pour la classe
                            if 'class' in df.columns:
                                cls = int(row['class'])
                            elif 'class_id' in df.columns:
                                cls = int(row['class_id'])
                            elif 'label' in df.columns:
                                cls = int(row['label'])
                            else:
                                continue
                        elif 'x1' in df.columns:
                            xmin = float(row['x1'])
                            ymin = float(row['y1'])
                            xmax = float(row['x2'])
                            ymax = float(row['y2'])
                            if 'class' in df.columns:
                                cls = int(row['class'])
                            elif 'class_id' in df.columns:
                                cls = int(row['class_id'])
                            elif 'label' in df.columns:
                                cls = int(row['label'])
                            else:
                                continue
                        else:
                            continue
                        
                        # Validation
                        if xmax <= xmin or ymax <= ymin:
                            continue
                        
                        if not (0 <= cls < NUM_CLASSES):
                            continue
                        
                        # Convertir en cxcywh normalisé
                        cx = ((xmin + xmax) / 2) / img_width
                        cy = ((ymin + ymax) / 2) / img_height
                        w = (xmax - xmin) / img_width
                        h = (ymax - ymin) / img_height
                        
                        if not (0 <= cx <= 1 and 0 <= cy <= 1 and 0 < w <= 1 and 0 < h <= 1):
                            continue
                        
                        boxes.append([cx, cy, w, h])
                        labels.append(cls)
                    except Exception as e:
                        continue
                
                # Sauvegarder
                json_path = os.path.join(labels_coco_dir, base_name + ".json")
                target = {"boxes": boxes, "class_labels": labels}
                with open(json_path, "w") as f:
                    json.dump(target, f)
                
                converted += 1
                continue
            
            # Grouper par image (si colonne filename existe)
            for filename in df[filename_col].unique():
                img_data = df[df[filename_col] == filename]
                
                # Vérifier si l'image existe
                img_path = os.path.join(images_dir, filename)
                if not os.path.exists(img_path):
                    continue
                
                img = Image.open(img_path)
                img_width, img_height = img.size
                
                boxes = []
                labels = []
                
                for _, row in img_data.iterrows():
                    try:
                        # Récupérer les coordonnées
                        if 'xmin' in df.columns:
                            xmin = float(row['xmin'])
                            ymin = float(row['ymin'])
                            xmax = float(row['xmax'])
                            ymax = float(row['ymax'])
                        elif 'x1' in df.columns:
                            xmin = float(row['x1'])
                            ymin = float(row['y1'])
                            xmax = float(row['x2'])
                            ymax = float(row['y2'])
                        else:
                            continue
                        
                        # Gérer différents noms de colonnes pour la classe
                        if 'class' in df.columns:
                            cls = int(row['class'])
                        elif 'class_id' in df.columns:
                            cls = int(row['class_id'])
                        elif 'label' in df.columns:
                            cls = int(row['label'])
                        else:
                            continue
                        
                        # Validation
                        if xmax <= xmin or ymax <= ymin:
                            continue
                        
                        if not (0 <= cls < NUM_CLASSES):
                            continue
                        
                        # Convertir en cxcywh normalisé
                        cx = ((xmin + xmax) / 2) / img_width
                        cy = ((ymin + ymax) / 2) / img_height
                        w = (xmax - xmin) / img_width
                        h = (ymax - ymin) / img_height
                        
                        if not (0 <= cx <= 1 and 0 <= cy <= 1 and 0 < w <= 1 and 0 < h <= 1):
                            continue
                        
                        boxes.append([cx, cy, w, h])
                        labels.append(cls)
                    except Exception as e:
                        continue
                
                # Sauvegarder en JSON
                base_name = os.path.splitext(filename)[0]
                json_path = os.path.join(labels_coco_dir, base_name + ".json")
                
                target = {"boxes": boxes, "class_labels": labels}
                
                with open(json_path, "w") as f:
                    json.dump(target, f)
                
                converted += 1
        
        except Exception as e:
            print(f"   Erreur avec {csv_file}: {e}")
            continue
    
    print(f" {converted} fichiers convertis")
    return converted

# =========================
# DATASET
# =========================
class DetrDataset(Dataset):
    def __init__(self, images_dir, labels_dir):
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        
        # Lister les images
        all_images = []
        for ext in [".jpg", ".jpeg", ".png"]:
            all_images.extend([f for f in os.listdir(images_dir) if f.lower().endswith(ext)])
        
        self.images = sorted(list(set(all_images)))
        
        # Filtrer les images qui ont des labels
        self.images = [img for img in self.images 
                      if os.path.exists(os.path.join(labels_dir, os.path.splitext(img)[0] + ".json"))]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.images_dir, img_name)
        
        base_name = os.path.splitext(img_name)[0]
        label_path = os.path.join(self.labels_dir, base_name + ".json")

        image = Image.open(img_path).convert("RGB")

        with open(label_path, "r") as f:
            target = json.load(f)

        target = {
            "boxes": torch.tensor(target["boxes"], dtype=torch.float32) if target["boxes"] else torch.zeros((0, 4), dtype=torch.float32),
            "class_labels": torch.tensor(target["class_labels"], dtype=torch.int64) if target["class_labels"] else torch.zeros((0,), dtype=torch.int64)
        }

        return image, target

def collate_fn(batch):
    images, targets = zip(*batch)
    return list(images), list(targets)

# =========================
# CHARGER LE MEILLEUR MODÈLE
# =========================
print("\n Chargement du meilleur modèle...")

history_path = os.path.join(MODELS_DIR, "test_history.csv")
if os.path.exists(history_path):
    df_history = pd.read_csv(history_path)
    best_epoch = df_history.loc[df_history['Box-F1'].idxmax(), 'epoch']
    best_f1 = df_history.loc[df_history['Box-F1'].idxmax(), 'Box-F1']
    print(f" Meilleur epoch: {int(best_epoch)} (F1={best_f1:.4f})")
else:
    model_files = [f for f in os.listdir(MODELS_DIR) if f.startswith("detr_epoch") and f.endswith(".pt")]
    epochs = [int(f.split("epoch")[1].split(".pt")[0]) for f in model_files]
    best_epoch = max(epochs) if epochs else 1
    print(f" Utilisation de l'epoch {int(best_epoch)}")

best_model_path = os.path.join(MODELS_DIR, f"detr_epoch{int(best_epoch)}.pt")

processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")
id2label = {i: f"class{i}" for i in range(NUM_CLASSES)}
label2id = {f"class{i}": i for i in range(NUM_CLASSES)}

model = DetrForObjectDetection.from_pretrained(
    "facebook/detr-resnet-50",
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
model.to(DEVICE)
model.eval()

print(f" Modèle chargé: {best_model_path}")

# =========================
# HELPER
# =========================
def cxcywh_to_xyxy(boxes, img_w, img_h):
    if boxes.numel() == 0:
        return torch.zeros((0, 4), dtype=torch.float32)
    cx = boxes[:, 0] * img_w
    cy = boxes[:, 1] * img_h
    w = boxes[:, 2] * img_w
    h = boxes[:, 3] * img_h
    x1 = cx - w/2
    y1 = cy - h/2
    x2 = cx + w/2
    y2 = cy + h/2
    return torch.stack([x1, y1, x2, y2], dim=1)

# =========================
# ÉVALUATION
# =========================
def evaluate(model, loader, conf_threshold=0.1):
    model.eval()
    all_gt = []
    all_pred = []
    all_ious = []
    total_det = 0
    total_gt = 0

    with torch.no_grad():
        #  DÉSACTIVER tqdm pour éviter trop de messages
        for images, targets in loader:
            encoding = processor(images=images, return_tensors="pt")
            pixel_values = encoding["pixel_values"].to(DEVICE)
            pixel_mask = encoding["pixel_mask"].to(DEVICE)

            outputs = model(pixel_values=pixel_values, pixel_mask=pixel_mask)
            
            target_sizes = torch.tensor([img.size[::-1] for img in images])
            results = processor.post_process_object_detection(
                outputs, target_sizes=target_sizes, threshold=conf_threshold
            )

            for i, res in enumerate(results):
                pred_boxes = res["boxes"].clone() if len(res["boxes"]) > 0 else torch.zeros((0, 4), dtype=torch.float32)
                pred_labels = res["labels"].clone() if len(res["labels"]) > 0 else torch.zeros((0,), dtype=torch.int64)
                
                gt_boxes_norm = targets[i]["boxes"]
                gt_labels = targets[i]["class_labels"]
                
                total_det += len(pred_labels)
                total_gt += len(gt_labels)
                
                if len(pred_boxes) == 0 or len(gt_boxes_norm) == 0:
                    continue
                
                img_w, img_h = images[i].size
                gt_boxes = cxcywh_to_xyxy(gt_boxes_norm, img_w, img_h)
                
                iou = box_iou(gt_boxes, pred_boxes)
                max_iou, best_idx = iou.max(dim=1)
                all_ious.extend(max_iou.cpu().numpy().tolist())
                
                matched = max_iou >= 0.5
                if matched.sum() > 0:
                    all_gt.extend(gt_labels[matched].cpu().numpy().tolist())
                    all_pred.extend(pred_labels[best_idx[matched]].cpu().numpy().tolist())

    if len(all_gt) > 0 and len(all_pred) > 0:
        prec = precision_score(all_gt, all_pred, average="macro", zero_division=0)
        rec = recall_score(all_gt, all_pred, average="macro", zero_division=0)
        f1 = f1_score(all_gt, all_pred, average="macro", zero_division=0)
    else:
        prec, rec, f1 = 0, 0, 0

    map50 = sum(1 for iou in all_ious if iou >= 0.5) / len(all_ious) if all_ious else 0
    map5095 = sum(all_ious) / len(all_ious) if all_ious else 0

    return {
        "Detections": total_det,
        "Ground_Truth": total_gt,
        "Matched": len(all_gt),
        "Precision": round(prec, 5),
        "Recall": round(rec, 5),
        "F1": round(f1, 5),
        "mAP50": round(map50, 5),
        "mAP50-95": round(map5095, 5)
    }

# =========================
# TEST SUR TOUTES LES CONDITIONS
# =========================
all_results = []

print("\n" + "="*70)
print("ÉVALUATION SUR CONDITIONS MÉTÉO")
print("="*70)

for weather in WEATHER_CONDITIONS:
    print(f"\n{'='*70}")
    print(f"  CONDITION: {weather.upper()}")
    print(f"{'='*70}")
    
    for level in LEVELS:
        condition_name = f"{weather}_level{level}"
        images_dir = os.path.join(RESULTS_BASE_DIR, condition_name, "images")
        labels_csv_dir = os.path.join(RESULTS_BASE_DIR, condition_name, "labels_csv")
        labels_coco_dir = os.path.join(RESULTS_BASE_DIR, condition_name, "labels_coco")
        
        # Vérifier si le dossier existe
        if not os.path.exists(images_dir):
            print(f" {condition_name}: Dossier non trouvé, ignoré")
            continue
        
        print(f"\n {condition_name}:")
        
        # Convertir les labels CSV -> COCO si nécessaire
        if not os.path.exists(labels_coco_dir) or len(os.listdir(labels_coco_dir)) == 0:
            num_converted = convert_csv_to_coco(images_dir, labels_csv_dir, labels_coco_dir)
            if num_converted == 0:
                print(f" Aucun label converti, ignoré")
                continue
        else:
            print(f" Labels COCO déjà convertis")
        
        # Créer dataset et dataloader
        try:
            dataset = DetrDataset(images_dir, labels_coco_dir)
            if len(dataset) == 0:
                print(f" Dataset vide, ignoré")
                continue
            
            print(f" {len(dataset)} images à évaluer")
            
            loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
            
            # Évaluation (avec affichage de progression simple)
            print(f"   Évaluation en cours...", end="", flush=True)
            metrics = evaluate(model, loader)
            print("  Terminé!")
            
            # Ajouter les métadonnées
            metrics["Weather"] = weather
            metrics["Level"] = level
            metrics["Condition"] = condition_name
            metrics["Num_Images"] = len(dataset)
            
            all_results.append(metrics)
            
            # Afficher les résultats
            print(f"   Détections: {metrics['Detections']}, GT: {metrics['Ground_Truth']}, Matchés: {metrics['Matched']}")
            print(f"   P: {metrics['Precision']:.4f}, R: {metrics['Recall']:.4f}, F1: {metrics['F1']:.4f}")
            print(f"   mAP50: {metrics['mAP50']:.4f}, mAP50-95: {metrics['mAP50-95']:.4f}")
            
        except Exception as e:
            print(f" Erreur: {e}")
            continue

# =========================
# SAUVEGARDER LES RÉSULTATS
# =========================
print("\n" + "="*70)
print(" SAUVEGARDE DES RÉSULTATS")
print("="*70)

if len(all_results) > 0:
    df_results = pd.DataFrame(all_results)
    
    # Réorganiser les colonnes
    cols = ["Weather", "Level", "Condition", "Num_Images", "Detections", "Ground_Truth", "Matched",
            "Precision", "Recall", "F1", "mAP50", "mAP50-95"]
    df_results = df_results[cols]
    
    # Sauvegarder résultats complets
    results_path = os.path.join(OUTPUT_DIR, "weather_conditions_results.csv")
    df_results.to_csv(results_path, index=False)
    print(f" Résultats complets: {results_path}")
    
    # Créer un résumé par condition météo
    summary = df_results.groupby("Weather")[["Precision", "Recall", "F1", "mAP50", "mAP50-95"]].mean()
    summary_path = os.path.join(OUTPUT_DIR, "weather_summary.csv")
    summary.to_csv(summary_path)
    print(f" Résumé par météo: {summary_path}")
    
    # Afficher le tableau complet
    print("\n RÉSULTATS COMPLETS:")
    print(df_results.to_string(index=False))
    
    print("\n RÉSUMÉ PAR CONDITION MÉTÉO:")
    print(summary.round(4))
    
else:
    print(" Aucun résultat à sauvegarder")

print("\n" + "="*70)
print(" ÉVALUATION TERMINÉE!")
print(f" Résultats dans: {OUTPUT_DIR}")
print("="*70)

DEVICE: cpu

 Chargement du meilleur modèle...
 Meilleur epoch: 10 (F1=0.9049)


<env>\lib\site-packages\torch\nn\modules\module.py:2446: UserWarning: for conv1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  warnings.warn(
<env>\lib\site-packages\torch\nn\modules\module.py:2446: UserWarning: for bn1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  warnings.warn(
<env>\lib\site-packages\torch\nn\modules\module.py:2446: UserWarning: for bn1.bias: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dic

 Modèle chargé: ./models_detr\detr_epoch10.pt

ÉVALUATION SUR CONDITIONS MÉTÉO

  CONDITION: FOG

 fog_level1:
 Labels COCO déjà convertis
 500 images à évaluer
   Évaluation en cours...

In [4]:
import os
import json
import cv2
import random
from torch.utils.data import Dataset

# =========================
# CONFIGURATION (les mêmes chemins que pour le train)
# =========================
IMAGES_DIR = "./Data2/train/images"
LABELS_COCO_DIR = "./Data2/train/labels_coco"
SAMPLE_DIR = "./Data2/sample_visual"
os.makedirs(SAMPLE_DIR, exist_ok=True)

# =========================
# DATASET SIMPLE
# =========================
class SimpleDataset(Dataset):
    def __init__(self, images_dir):
        self.images_dir = images_dir
        self.images = sorted([f for f in os.listdir(images_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))])

train_dataset = SimpleDataset(IMAGES_DIR)

# =========================
# PRÉLÈVEMENT DES 30 IMAGES
# =========================
sample_images = random.sample(train_dataset.images, min(30, len(train_dataset.images)))

for img_name in sample_images:
    img_path = os.path.join(IMAGES_DIR, img_name)
    base_name = os.path.splitext(img_name)[0]
    label_path = os.path.join(LABELS_COCO_DIR, base_name + ".json")

    image = cv2.imread(img_path)
    if image is None:
        continue

    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Charger les boxes
    if os.path.exists(label_path):
        with open(label_path, "r") as f:
            target = json.load(f)
        boxes = target.get("boxes", [])
        labels = target.get("class_labels", [])

        for box, label in zip(boxes, labels):
            x_min, y_min, x_max, y_max = map(int, box)
            color = (0, 255, 0)
            cv2.rectangle(image_rgb, (x_min, y_min), (x_max, y_max), color, 2)
            cv2.putText(image_rgb, str(label), (x_min, y_min-5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    save_path = os.path.join(SAMPLE_DIR, img_name)
    cv2.imwrite(save_path, cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR))

print(f" 30 images annotées enregistrées dans : {SAMPLE_DIR}")


 30 images annotées enregistrées dans : ./Data2/sample_visual


In [6]:
import os
import random
from PIL import Image, ImageDraw

# =========================
# CONFIG
# =========================
SAMPLE_DIR_YOLO = "./Data2/sample_visual1"
os.makedirs(SAMPLE_DIR_YOLO, exist_ok=True)

IMAGES_DIR = "./Data2/train/images"
LABELS_YOLO_DIR = "./Data2/train/labels"

# =========================
# Prendre un échantillon de 30 images
# =========================
all_images = [f for f in os.listdir(IMAGES_DIR) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
sample_images = random.sample(all_images, min(30, len(all_images)))

# =========================
# Dessiner les boxes YOLO
# =========================
for img_name in sample_images:
    img_path = os.path.join(IMAGES_DIR, img_name)
    base_name = os.path.splitext(img_name)[0]
    label_path = os.path.join(LABELS_YOLO_DIR, base_name + ".txt")

    image = Image.open(img_path).convert("RGB")
    draw = ImageDraw.Draw(image)  #  Correct ici
    width, height = image.size

    if os.path.exists(label_path):
        with open(label_path, "r") as f:
            lines = f.readlines()
            for line in lines:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                cls, x_center, y_center, w, h = map(float, parts)
                
                # Conversion YOLO -> xyxy absolu
                x_min = (x_center - w/2) * width
                y_min = (y_center - h/2) * height
                x_max = (x_center + w/2) * width
                y_max = (y_center + h/2) * height
                
                # Dessiner la box
                draw.rectangle([x_min, y_min, x_max, y_max], outline="red", width=2)
                draw.text((x_min, y_min), str(int(cls)), fill="red")

    # Sauvegarder l'image annotée
    save_path = os.path.join(SAMPLE_DIR_YOLO, img_name)
    image.save(save_path)

print(f" 30 images annotées avec les labels YOLO sauvegardées dans {SAMPLE_DIR_YOLO}")


 30 images annotées avec les labels YOLO sauvegardées dans ./Data2/sample_visual1


# DETR train et test  

# Flog_level1 a 5 DETR

In [1]:
import os
import json
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import DetrForObjectDetection, DetrImageProcessor
from tqdm import tqdm
from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd
from torchvision.ops import box_iou
import glob

# =========================
# CONFIGURATION
# =========================
DEVICE = "cpu"
print(f"DEVICE: {DEVICE}")

RESULTS_BASE_DIR = "./Data2/results"
MODELS_DIR = "./models_detr"
OUTPUT_DIR = "./weather_evaluation"

BATCH_SIZE = 2
NUM_CLASSES = 5

# Conditions météo et leurs niveaux
WEATHER_CONDITIONS = ["fog", "glare", "night", "rain", "snow"]
LEVELS = [1, 2, 3, 4, 5]

os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================
# CONVERSION CSV -> COCO FORMAT
# =========================
def convert_csv_to_coco(images_dir, labels_csv_dir, labels_coco_dir):
    """
    Convertit les labels CSV en format COCO pour DETR
    Détecte automatiquement le format du CSV
    """
    print(f"Conversion CSV -> COCO pour {images_dir}...")
    os.makedirs(labels_coco_dir, exist_ok=True)
    
    # Lire tous les fichiers CSV
    csv_files = glob.glob(os.path.join(labels_csv_dir, "*.csv"))
    
    if len(csv_files) == 0:
        print(f" Aucun fichier CSV trouvé dans {labels_csv_dir}")
        return 0
    
    converted = 0
    
    # Lire le premier CSV pour détecter le format
    first_csv = csv_files[0]
    df_sample = pd.read_csv(first_csv, nrows=5)
    print(f"  Colonnes détectées: {list(df_sample.columns)}")
    
    # Détecter le format
    has_filename = 'filename' in df_sample.columns
    has_image = 'image' in df_sample.columns
    
    for csv_file in csv_files:
        try:
            df = pd.read_csv(csv_file)
            
            # Adapter selon le format
            if has_filename:
                filename_col = 'filename'
            elif has_image:
                filename_col = 'image'
            else:
                # Si pas de colonne filename, le nom du CSV est le nom de l'image
                base_name = os.path.splitext(os.path.basename(csv_file))[0]
                # Chercher l'image correspondante
                img_file = None
                for ext in [".jpg", ".jpeg", ".png"]:
                    test_path = os.path.join(images_dir, base_name + ext)
                    if os.path.exists(test_path):
                        img_file = base_name + ext
                        break
                
                if img_file is None:
                    continue
                
                # Traiter ce fichier comme une seule image
                img_path = os.path.join(images_dir, img_file)
                img = Image.open(img_path)
                img_width, img_height = img.size
                
                boxes = []
                labels = []
                
                for _, row in df.iterrows():
                    try:
                        # Essayer différents formats de colonnes
                        if 'xmin' in df.columns:
                            xmin = float(row['xmin'])
                            ymin = float(row['ymin'])
                            xmax = float(row['xmax'])
                            ymax = float(row['ymax'])
                            # Gérer différents noms de colonnes pour la classe
                            if 'class' in df.columns:
                                cls = int(row['class'])
                            elif 'class_id' in df.columns:
                                cls = int(row['class_id'])
                            elif 'label' in df.columns:
                                cls = int(row['label'])
                            else:
                                continue
                        elif 'x1' in df.columns:
                            xmin = float(row['x1'])
                            ymin = float(row['y1'])
                            xmax = float(row['x2'])
                            ymax = float(row['y2'])
                            if 'class' in df.columns:
                                cls = int(row['class'])
                            elif 'class_id' in df.columns:
                                cls = int(row['class_id'])
                            elif 'label' in df.columns:
                                cls = int(row['label'])
                            else:
                                continue
                        else:
                            continue
                        
                        # Validation
                        if xmax <= xmin or ymax <= ymin:
                            continue
                        
                        if not (0 <= cls < NUM_CLASSES):
                            continue
                        
                        # Convertir en cxcywh normalisé
                        cx = ((xmin + xmax) / 2) / img_width
                        cy = ((ymin + ymax) / 2) / img_height
                        w = (xmax - xmin) / img_width
                        h = (ymax - ymin) / img_height
                        
                        if not (0 <= cx <= 1 and 0 <= cy <= 1 and 0 < w <= 1 and 0 < h <= 1):
                            continue
                        
                        boxes.append([cx, cy, w, h])
                        labels.append(cls)
                    except Exception as e:
                        continue
                
                # Sauvegarder
                json_path = os.path.join(labels_coco_dir, base_name + ".json")
                target = {"boxes": boxes, "class_labels": labels}
                with open(json_path, "w") as f:
                    json.dump(target, f)
                
                converted += 1
                continue
            
            # Grouper par image (si colonne filename existe)
            for filename in df[filename_col].unique():
                img_data = df[df[filename_col] == filename]
                
                # Vérifier si l'image existe
                img_path = os.path.join(images_dir, filename)
                if not os.path.exists(img_path):
                    continue
                
                img = Image.open(img_path)
                img_width, img_height = img.size
                
                boxes = []
                labels = []
                
                for _, row in img_data.iterrows():
                    try:
                        # Récupérer les coordonnées
                        if 'xmin' in df.columns:
                            xmin = float(row['xmin'])
                            ymin = float(row['ymin'])
                            xmax = float(row['xmax'])
                            ymax = float(row['ymax'])
                        elif 'x1' in df.columns:
                            xmin = float(row['x1'])
                            ymin = float(row['y1'])
                            xmax = float(row['x2'])
                            ymax = float(row['y2'])
                        else:
                            continue
                        
                        # Gérer différents noms de colonnes pour la classe
                        if 'class' in df.columns:
                            cls = int(row['class'])
                        elif 'class_id' in df.columns:
                            cls = int(row['class_id'])
                        elif 'label' in df.columns:
                            cls = int(row['label'])
                        else:
                            continue
                        
                        # Validation
                        if xmax <= xmin or ymax <= ymin:
                            continue
                        
                        if not (0 <= cls < NUM_CLASSES):
                            continue
                        
                        # Convertir en cxcywh normalisé
                        cx = ((xmin + xmax) / 2) / img_width
                        cy = ((ymin + ymax) / 2) / img_height
                        w = (xmax - xmin) / img_width
                        h = (ymax - ymin) / img_height
                        
                        if not (0 <= cx <= 1 and 0 <= cy <= 1 and 0 < w <= 1 and 0 < h <= 1):
                            continue
                        
                        boxes.append([cx, cy, w, h])
                        labels.append(cls)
                    except Exception as e:
                        continue
                
                # Sauvegarder en JSON
                base_name = os.path.splitext(filename)[0]
                json_path = os.path.join(labels_coco_dir, base_name + ".json")
                
                target = {"boxes": boxes, "class_labels": labels}
                
                with open(json_path, "w") as f:
                    json.dump(target, f)
                
                converted += 1
        
        except Exception as e:
            print(f"   Erreur avec {csv_file}: {e}")
            continue
    
    print(f" {converted} fichiers convertis")
    return converted

# =========================
# DATASET
# =========================
class DetrDataset(Dataset):
    def __init__(self, images_dir, labels_dir):
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        
        # Lister les images
        all_images = []
        for ext in [".jpg", ".jpeg", ".png"]:
            all_images.extend([f for f in os.listdir(images_dir) if f.lower().endswith(ext)])
        
        self.images = sorted(list(set(all_images)))
        
        # Filtrer les images qui ont des labels
        self.images = [img for img in self.images 
                      if os.path.exists(os.path.join(labels_dir, os.path.splitext(img)[0] + ".json"))]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.images_dir, img_name)
        
        base_name = os.path.splitext(img_name)[0]
        label_path = os.path.join(self.labels_dir, base_name + ".json")

        image = Image.open(img_path).convert("RGB")

        with open(label_path, "r") as f:
            target = json.load(f)

        target = {
            "boxes": torch.tensor(target["boxes"], dtype=torch.float32) if target["boxes"] else torch.zeros((0, 4), dtype=torch.float32),
            "class_labels": torch.tensor(target["class_labels"], dtype=torch.int64) if target["class_labels"] else torch.zeros((0,), dtype=torch.int64)
        }

        return image, target

def collate_fn(batch):
    images, targets = zip(*batch)
    return list(images), list(targets)

# =========================
# CHARGER LE MEILLEUR MODÈLE
# =========================
print("\n Chargement du meilleur modèle...")

history_path = os.path.join(MODELS_DIR, "test_history.csv")
if os.path.exists(history_path):
    df_history = pd.read_csv(history_path)
    best_epoch = df_history.loc[df_history['Box-F1'].idxmax(), 'epoch']
    best_f1 = df_history.loc[df_history['Box-F1'].idxmax(), 'Box-F1']
    print(f" Meilleur epoch: {int(best_epoch)} (F1={best_f1:.4f})")
else:
    model_files = [f for f in os.listdir(MODELS_DIR) if f.startswith("detr_epoch") and f.endswith(".pt")]
    epochs = [int(f.split("epoch")[1].split(".pt")[0]) for f in model_files]
    best_epoch = max(epochs) if epochs else 1
    print(f" Utilisation de l'epoch {int(best_epoch)}")

best_model_path = os.path.join(MODELS_DIR, f"detr_epoch{int(best_epoch)}.pt")

processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")
id2label = {i: f"class{i}" for i in range(NUM_CLASSES)}
label2id = {f"class{i}": i for i in range(NUM_CLASSES)}

model = DetrForObjectDetection.from_pretrained(
    "facebook/detr-resnet-50",
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
model.to(DEVICE)
model.eval()

print(f" Modèle chargé: {best_model_path}")

# =========================
# HELPER
# =========================
def cxcywh_to_xyxy(boxes, img_w, img_h):
    if boxes.numel() == 0:
        return torch.zeros((0, 4), dtype=torch.float32)
    cx = boxes[:, 0] * img_w
    cy = boxes[:, 1] * img_h
    w = boxes[:, 2] * img_w
    h = boxes[:, 3] * img_h
    x1 = cx - w/2
    y1 = cy - h/2
    x2 = cx + w/2
    y2 = cy + h/2
    return torch.stack([x1, y1, x2, y2], dim=1)

# =========================
# ÉVALUATION
# =========================
def evaluate(model, loader, conf_threshold=0.1):
    model.eval()
    all_gt = []
    all_pred = []
    all_ious = []
    total_det = 0
    total_gt = 0

    with torch.no_grad():
        #  DÉSACTIVER tqdm pour éviter trop de messages
        for images, targets in loader:
            encoding = processor(images=images, return_tensors="pt")
            pixel_values = encoding["pixel_values"].to(DEVICE)
            pixel_mask = encoding["pixel_mask"].to(DEVICE)

            outputs = model(pixel_values=pixel_values, pixel_mask=pixel_mask)
            
            target_sizes = torch.tensor([img.size[::-1] for img in images])
            results = processor.post_process_object_detection(
                outputs, target_sizes=target_sizes, threshold=conf_threshold
            )

            for i, res in enumerate(results):
                pred_boxes = res["boxes"].clone() if len(res["boxes"]) > 0 else torch.zeros((0, 4), dtype=torch.float32)
                pred_labels = res["labels"].clone() if len(res["labels"]) > 0 else torch.zeros((0,), dtype=torch.int64)
                
                gt_boxes_norm = targets[i]["boxes"]
                gt_labels = targets[i]["class_labels"]
                
                total_det += len(pred_labels)
                total_gt += len(gt_labels)
                
                if len(pred_boxes) == 0 or len(gt_boxes_norm) == 0:
                    continue
                
                img_w, img_h = images[i].size
                gt_boxes = cxcywh_to_xyxy(gt_boxes_norm, img_w, img_h)
                
                iou = box_iou(gt_boxes, pred_boxes)
                max_iou, best_idx = iou.max(dim=1)
                all_ious.extend(max_iou.cpu().numpy().tolist())
                
                matched = max_iou >= 0.5
                if matched.sum() > 0:
                    all_gt.extend(gt_labels[matched].cpu().numpy().tolist())
                    all_pred.extend(pred_labels[best_idx[matched]].cpu().numpy().tolist())

    if len(all_gt) > 0 and len(all_pred) > 0:
        prec = precision_score(all_gt, all_pred, average="macro", zero_division=0)
        rec = recall_score(all_gt, all_pred, average="macro", zero_division=0)
        f1 = f1_score(all_gt, all_pred, average="macro", zero_division=0)
    else:
        prec, rec, f1 = 0, 0, 0

    map50 = sum(1 for iou in all_ious if iou >= 0.5) / len(all_ious) if all_ious else 0
    map5095 = sum(all_ious) / len(all_ious) if all_ious else 0

    return {
        "Detections": total_det,
        "Ground_Truth": total_gt,
        "Matched": len(all_gt),
        "Precision": round(prec, 5),
        "Recall": round(rec, 5),
        "F1": round(f1, 5),
        "mAP50": round(map50, 5),
        "mAP50-95": round(map5095, 5)
    }

# =========================
# TEST SUR TOUTES LES CONDITIONS
# =========================
all_results = []

print("\n" + "="*70)
print("ÉVALUATION SUR CONDITIONS MÉTÉO")
print("="*70)

for weather in WEATHER_CONDITIONS:
    print(f"\n{'='*70}")
    print(f"  CONDITION: {weather.upper()}")
    print(f"{'='*70}")
    
    for level in LEVELS:
        condition_name = f"{weather}_level{level}"
        images_dir = os.path.join(RESULTS_BASE_DIR, condition_name, "images")
        labels_csv_dir = os.path.join(RESULTS_BASE_DIR, condition_name, "labels_csv")
        labels_coco_dir = os.path.join(RESULTS_BASE_DIR, condition_name, "labels_coco")
        
        # Vérifier si le dossier existe
        if not os.path.exists(images_dir):
            print(f" {condition_name}: Dossier non trouvé, ignoré")
            continue
        
        print(f"\n {condition_name}:")
        
        # Convertir les labels CSV -> COCO si nécessaire
        if not os.path.exists(labels_coco_dir) or len(os.listdir(labels_coco_dir)) == 0:
            num_converted = convert_csv_to_coco(images_dir, labels_csv_dir, labels_coco_dir)
            if num_converted == 0:
                print(f" Aucun label converti, ignoré")
                continue
        else:
            print(f" Labels COCO déjà convertis")
        
        # Créer dataset et dataloader
        try:
            dataset = DetrDataset(images_dir, labels_coco_dir)
            if len(dataset) == 0:
                print(f" Dataset vide, ignoré")
                continue
            
            print(f" {len(dataset)} images à évaluer")
            
            loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
            
            # Évaluation
            metrics = evaluate(model, loader)
            
            # Ajouter les métadonnées
            metrics["Weather"] = weather
            metrics["Level"] = level
            metrics["Condition"] = condition_name
            metrics["Num_Images"] = len(dataset)
            
            all_results.append(metrics)
            
            # Afficher les résultats
            print(f"   Détections: {metrics['Detections']}, GT: {metrics['Ground_Truth']}, Matchés: {metrics['Matched']}")
            print(f"   P: {metrics['Precision']:.4f}, R: {metrics['Recall']:.4f}, F1: {metrics['F1']:.4f}")
            print(f"   mAP50: {metrics['mAP50']:.4f}, mAP50-95: {metrics['mAP50-95']:.4f}")
            
        except Exception as e:
            print(f" Erreur: {e}")
            continue

# =========================
# SAUVEGARDER LES RÉSULTATS
# =========================
print("\n" + "="*70)
print(" SAUVEGARDE DES RÉSULTATS")
print("="*70)

if len(all_results) > 0:
    df_results = pd.DataFrame(all_results)
    
    # Réorganiser les colonnes
    cols = ["Weather", "Level", "Condition", "Num_Images", "Detections", "Ground_Truth", "Matched",
            "Precision", "Recall", "F1", "mAP50", "mAP50-95"]
    df_results = df_results[cols]
    
    # Sauvegarder résultats complets
    results_path = os.path.join(OUTPUT_DIR, "weather_conditions_results.csv")
    df_results.to_csv(results_path, index=False)
    print(f" Résultats complets: {results_path}")
    
    # Créer un résumé par condition météo
    summary = df_results.groupby("Weather")[["Precision", "Recall", "F1", "mAP50", "mAP50-95"]].mean()
    summary_path = os.path.join(OUTPUT_DIR, "weather_summary.csv")
    summary.to_csv(summary_path)
    print(f" Résumé par météo: {summary_path}")
    
    # Afficher le tableau complet
    print("\n RÉSULTATS COMPLETS:")
    print(df_results.to_string(index=False))
    
    print("\n RÉSUMÉ PAR CONDITION MÉTÉO:")
    print(summary.round(4))
    
else:
    print(" Aucun résultat à sauvegarder")

print("\n" + "="*70)
print(" ÉVALUATION TERMINÉE!")
print(f" Résultats dans: {OUTPUT_DIR}")
print("="*70)

DEVICE: cpu

 Chargement du meilleur modèle...
 Meilleur epoch: 10 (F1=0.9049)


<env>\lib\site-packages\torch\nn\modules\module.py:2446: UserWarning: for conv1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  warnings.warn(
<env>\lib\site-packages\torch\nn\modules\module.py:2446: UserWarning: for bn1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  warnings.warn(
<env>\lib\site-packages\torch\nn\modules\module.py:2446: UserWarning: for bn1.bias: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dic

 Modèle chargé: ./models_detr\detr_epoch10.pt

ÉVALUATION SUR CONDITIONS MÉTÉO

  CONDITION: FOG

 fog_level1:
 Labels COCO déjà convertis
 500 images à évaluer
   Détections: 20460, GT: 1559, Matchés: 963
   P: 0.8698, R: 0.8649, F1: 0.8615
   mAP50: 0.6177, mAP50-95: 0.5162

 fog_level2:
 Labels COCO déjà convertis
 500 images à évaluer
   Détections: 20019, GT: 1559, Matchés: 931
   P: 0.8853, R: 0.9183, F1: 0.8978
   mAP50: 0.5972, mAP50-95: 0.4988

 fog_level3:
 Labels COCO déjà convertis
 500 images à évaluer
   Détections: 19224, GT: 1559, Matchés: 824
   P: 0.8729, R: 0.9179, F1: 0.8891
   mAP50: 0.5285, mAP50-95: 0.4599

 fog_level4:
 Labels COCO déjà convertis
 500 images à évaluer
   Détections: 18625, GT: 1559, Matchés: 691
   P: 0.9409, R: 0.9439, F1: 0.9409
   mAP50: 0.4432, mAP50-95: 0.4117

 fog_level5:
 Labels COCO déjà convertis
 500 images à évaluer
   Détections: 18352, GT: 1559, Matchés: 574
   P: 0.9196, R: 0.9043, F1: 0.9068
   mAP50: 0.3682, mAP50-95: 0.3657

  

In [26]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# =========================
# DONNÉES DETR COMPLÈTES
# =========================

data_detr = {
    'Condition': ['Fog', 'Fog', 'Fog', 'Fog', 'Fog',
                 'Glare', 'Glare', 'Glare', 'Glare', 'Glare',
                 'Night', 'Night', 'Night', 'Night', 'Night',
                 'Rain', 'Rain', 'Rain', 'Rain', 'Rain',
                 'Snow', 'Snow', 'Snow', 'Snow', 'Snow'],
    
    'Level': [1, 2, 3, 4, 5] * 5,
    
    'Precision': [0.86978, 0.88526, 0.87290, 0.94094, 0.91958,
                  0.88332, 0.89776, 0.82090, 0.90290, 0.89695,
                  0.87056, 0.87767, 0.85226, 0.86615, 0.87722,
                  0.88266, 0.88925, 0.87938, 0.89466, 0.88396,
                  0.88557, 0.90002, 0.91712, 0.87905, 0.91571],
    
    'Recall': [0.86492, 0.91831, 0.91789, 0.94387, 0.90433,
               0.89204, 0.90227, 0.85636, 0.91448, 0.90288,
               0.92738, 0.92431, 0.87911, 0.89334, 0.90877,
               0.90179, 0.89238, 0.89527, 0.89551, 0.87448,
               0.89094, 0.92161, 0.89601, 0.89209, 0.86560],
    
    'F1-Score': [0.86148, 0.89782, 0.88912, 0.94086, 0.90680,
                 0.87870, 0.89551, 0.83416, 0.90326, 0.89469,
                 0.89317, 0.89619, 0.86175, 0.87593, 0.88912,
                 0.88675, 0.88777, 0.88239, 0.88785, 0.86993,
                 0.87976, 0.90541, 0.89450, 0.87407, 0.87250],
    
    'mAP50': [0.61770, 0.59718, 0.52854, 0.44323, 0.36818,
              0.64015, 0.63246, 0.62027, 0.61642, 0.61450,
              0.62412, 0.62348, 0.61321, 0.59589, 0.58178,
              0.63310, 0.63566, 0.64080, 0.63117, 0.62348,
              0.61642, 0.61450, 0.54843, 0.40218, 0.36305],
    
    'mAP50-95': [0.51620, 0.49876, 0.45991, 0.41165, 0.36567,
                 0.53311, 0.52895, 0.52185, 0.51835, 0.51750,
                 0.52233, 0.51843, 0.51041, 0.50231, 0.49199,
                 0.52773, 0.53081, 0.53099, 0.52907, 0.52426,
                 0.52216, 0.51641, 0.48286, 0.40442, 0.37921]
}

df_detr = pd.DataFrame(data_detr)

# =========================
# 1. GRAPHIQUES GLOBAUX
# =========================
print(" GRAPHIQUES GLOBAUX DETR")
print("="*50)

# Calcul des moyennes globales par condition
summary_detr = df_detr.groupby('Condition').agg({
    'Precision': 'mean',
    'Recall': 'mean', 
    'F1-Score': 'mean',
    'mAP50': 'mean',
    'mAP50-95': 'mean'
}).round(4)

# Graphique 1: Barres des moyennes globales
plt.figure(figsize=(14, 10))

# Sous-graphique 1: Métriques de base
plt.subplot(2, 2, 1)
metrics_basic = ['Precision', 'Recall', 'F1-Score']
x = np.arange(len(summary_detr.index))
width = 0.25

for i, metric in enumerate(metrics_basic):
    values = [summary_detr.loc[cond, metric] for cond in summary_detr.index]
    plt.bar(x + i*width, values, width, label=metric, alpha=0.8)

plt.xlabel('Conditions Météorologiques')
plt.ylabel('Score')
plt.title('DETR - Métriques Globales par Condition')
plt.xticks(x + width, summary_detr.index)
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(0.8, 0.95)

# Sous-graphique 2: Métriques mAP
plt.subplot(2, 2, 2)
metrics_map = ['mAP50', 'mAP50-95']
x = np.arange(len(summary_detr.index))
width = 0.35

for i, metric in enumerate(metrics_map):
    values = [summary_detr.loc[cond, metric] for cond in summary_detr.index]
    plt.bar(x + i*width, values, width, label=metric, alpha=0.8)

plt.xlabel('Conditions Météorologiques')
plt.ylabel('Score mAP')
plt.title('DETR - Métriques mAP Globales par Condition')
plt.xticks(x + width/2, summary_detr.index)
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(0.4, 0.7)

# Sous-graphique 3: Radar chart des performances globales
plt.subplot(2, 2, 3, polar=True)
categories = ['Precision', 'Recall', 'F1-Score', 'mAP50', 'mAP50-95']
N = len(categories)

angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

colors = ['blue', 'orange', 'green', 'red', 'purple']

for i, condition in enumerate(summary_detr.index):
    values = [summary_detr.loc[condition, metric] for metric in categories]
    values += values[:1]
    plt.plot(angles, values, 'o-', linewidth=2, label=condition, color=colors[i])
    plt.fill(angles, values, alpha=0.1, color=colors[i])

plt.xticks(angles[:-1], categories)
plt.yticks([0.2, 0.4, 0.6, 0.8, 1.0], ["0.2", "0.4", "0.6", "0.8", "1.0"])
plt.ylim(0, 1)
plt.title('Performance Globale - Diagramme Radar', size=12, y=1.08)
plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))

# Sous-graphique 4: Heatmap des performances globales
plt.subplot(2, 2, 4)
metrics_heatmap = ['Precision', 'Recall', 'F1-Score', 'mAP50', 'mAP50-95']
heatmap_data = summary_detr[metrics_heatmap].values

im = plt.imshow(heatmap_data, cmap='RdYlGn', aspect='auto', vmin=0.4, vmax=0.9)
plt.xticks(np.arange(len(metrics_heatmap)), metrics_heatmap, rotation=45)
plt.yticks(np.arange(len(summary_detr.index)), summary_detr.index)
plt.title('Heatmap des Performances Globales')
plt.colorbar(im, shrink=0.8)

# Ajouter les valeurs dans la heatmap
for i in range(len(summary_detr.index)):
    for j in range(len(metrics_heatmap)):
        text = plt.text(j, i, f'{heatmap_data[i, j]:.3f}',
                       ha="center", va="center", color="black", fontsize=9)

plt.tight_layout()
plt.suptitle('DETR - VUE GLOBALE DES PERFORMANCES', fontsize=16, fontweight='bold', y=1.02)
plt.show()

# =========================
# 2. GRAPHIQUES PAR CLASSE/CONDITION
# =========================
print("\n GRAPHIQUES PAR CONDITION ET NIVEAU")
print("="*50)

# Graphique 2: Évolution par niveau pour chaque condition
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('DETR - Évolution des Performances par Niveau et Condition', fontsize=16, fontweight='bold')

conditions = ['Fog', 'Glare', 'Night', 'Rain', 'Snow']
colors = ['blue', 'orange', 'gray', 'green', 'red']
metrics_to_plot = ['Precision', 'Recall', 'F1-Score', 'mAP50']

for idx, condition in enumerate(conditions):
    ax = axes[idx//3, idx%3]
    condition_data = df_detr[df_detr['Condition'] == condition]
    
    for metric in metrics_to_plot:
        ax.plot(condition_data['Level'], condition_data[metric], 
                marker='o', linewidth=2, label=metric)
    
    ax.set_xlabel('Niveau')
    ax.set_ylabel('Score')
    ax.set_title(f'{condition} - Évolution par Niveau')
    ax.set_xticks([1, 2, 3, 4, 5])
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0.3, 1.0)

# Cacher le dernier subplot vide
axes[1, 2].set_visible(False)

plt.tight_layout()
plt.show()

# =========================
# 3. GRAPHIQUES EN BARRES DÉTAILLÉS
# =========================
print("\n GRAPHIQUES EN BARRES DÉTAILLÉS")
print("="*50)

# Graphique 3: Barres groupées par condition et niveau
fig, axes = plt.subplots(3, 2, figsize=(16, 15))
fig.suptitle('DETR - Performances Détaillées par Condition et Niveau', fontsize=16, fontweight='bold')

metrics_bar = ['Precision', 'Recall', 'F1-Score', 'mAP50', 'mAP50-95']

for idx, metric in enumerate(metrics_bar):
    ax = axes[idx//2, idx%2]
    
    # Préparer les données pour cette métrique
    x = np.arange(5)  # 5 niveaux
    width = 0.15
    colors = ['lightblue', 'gold', 'lightgray', 'lightgreen', 'lightcoral']
    
    for i, condition in enumerate(conditions):
        condition_data = df_detr[df_detr['Condition'] == condition][metric].values
        ax.bar(x + i*width, condition_data, width, label=condition, 
               color=colors[i], alpha=0.8, edgecolor='black')
    
    ax.set_xlabel('Niveau')
    ax.set_ylabel(metric)
    ax.set_title(f'{metric} par Condition et Niveau')
    ax.set_xticks(x + 2*width)
    ax.set_xticklabels([1, 2, 3, 4, 5])
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    
    # Ajouter les valeurs sur les barres
    for i, condition in enumerate(conditions):
        condition_data = df_detr[df_detr['Condition'] == condition][metric].values
        for j, value in enumerate(condition_data):
            if value > 0.1:  # Éviter d'afficher sur les très petites valeurs
                ax.text(x[j] + i*width, value + 0.01, f'{value:.3f}', 
                       ha='center', va='bottom', fontsize=6)

plt.tight_layout()
plt.show()

# =========================
# 4. GRAPHIQUE COMPARATIF DES DÉGRADATIONS
# =========================
print("\n ANALYSE DES DÉGRADATIONS")
print("="*50)

# Graphique 4: Dégradation des performances avec les niveaux
plt.figure(figsize=(14, 8))

degradation_data = []
for condition in conditions:
    condition_data = df_detr[df_detr['Condition'] == condition]
    level1_mAP = condition_data[condition_data['Level'] == 1]['mAP50'].values[0]
    level5_mAP = condition_data[condition_data['Level'] == 5]['mAP50'].values[0]
    degradation_pct = ((level1_mAP - level5_mAP) / level1_mAP) * 100
    
    degradation_data.append({
        'Condition': condition,
        'Level1_mAP50': level1_mAP,
        'Level5_mAP50': level5_mAP,
        'Degradation': level1_mAP - level5_mAP,
        'Degradation_pct': degradation_pct
    })

df_degradation = pd.DataFrame(degradation_data)

# Graphique des dégradations
plt.subplot(1, 2, 1)
plt.bar(df_degradation['Condition'], df_degradation['Degradation_pct'], 
        color=['lightblue', 'gold', 'lightgray', 'lightgreen', 'lightcoral'],
        alpha=0.8, edgecolor='black')
plt.xlabel('Condition Météorologique')
plt.ylabel('Dégradation (%)')
plt.title('Dégradation mAP50 du Niveau 1 au 5')
plt.grid(True, alpha=0.3)

# Ajouter les valeurs sur les barres
for i, row in df_degradation.iterrows():
    plt.text(i, row['Degradation_pct'] + 1, f'{row["Degradation_pct"]:.1f}%', 
             ha='center', va='bottom', fontweight='bold')

# Graphique de comparaison Level 1 vs Level 5
plt.subplot(1, 2, 2)
x = np.arange(len(conditions))
width = 0.35

plt.bar(x - width/2, df_degradation['Level1_mAP50'], width, 
        label='Niveau 1', color='green', alpha=0.7)
plt.bar(x + width/2, df_degradation['Level5_mAP50'], width, 
        label='Niveau 5', color='red', alpha=0.7)

plt.xlabel('Condition Météorologique')
plt.ylabel('mAP50')
plt.title('Comparaison mAP50: Niveau 1 vs Niveau 5')
plt.xticks(x, conditions)
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# =========================
# RÉSUMÉ FINAL
# =========================
print("\n" + "="*60)
print(" RÉSUMÉ DES PERFORMANCES DETR")
print("="*60)

print(" CLASSEMENT PAR CONDITION (mAP50):")
ranking = summary_detr['mAP50'].sort_values(ascending=False)
for i, (condition, score) in enumerate(ranking.items(), 1):
    print(f"  {i}. {condition}: {score:.3f}")

print(f"\n DÉGRADATION MAXIMALE: Snow ({df_degradation[df_degradation['Condition']=='Snow']['Degradation_pct'].values[0]:.1f}%)")
print(f" PERFORMANCE LA PLUS STABLE: Rain ({df_degradation[df_degradation['Condition']=='Rain']['Degradation_pct'].values[0]:.1f}%)")

print(f"\n Graphiques générés:")
print(f"    4 graphiques globaux")
print(f"    6 graphiques par condition") 
print(f"    5 graphiques en barres détaillés")
print(f"    2 graphiques d'analyse de dégradation")

 GRAPHIQUES GLOBAUX DETR


<Figure size 1400x1000 with 5 Axes>


 GRAPHIQUES PAR CONDITION ET NIVEAU


<Figure size 1800x1200 with 6 Axes>


 GRAPHIQUES EN BARRES DÉTAILLÉS


<Figure size 1600x1500 with 6 Axes>


 ANALYSE DES DÉGRADATIONS


<Figure size 1400x800 with 2 Axes>


 RÉSUMÉ DES PERFORMANCES DETR
 CLASSEMENT PAR CONDITION (mAP50):
  1. Rain: 0.633
  2. Glare: 0.625
  3. Night: 0.608
  4. Fog: 0.511
  5. Snow: 0.509

 DÉGRADATION MAXIMALE: Snow (41.1%)
 PERFORMANCE LA PLUS STABLE: Rain (1.5%)

 Graphiques générés:
    4 graphiques globaux
    6 graphiques par condition
    5 graphiques en barres détaillés
    2 graphiques d'analyse de dégradation


# DETR WEATRHER 

In [27]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# =========================
# DONNÉES DETR COMPLÈTES
# =========================

data_detr = {
    'Condition': ['Fog', 'Fog', 'Fog', 'Fog', 'Fog',
                 'Glare', 'Glare', 'Glare', 'Glare', 'Glare',
                 'Night', 'Night', 'Night', 'Night', 'Night',
                 'Rain', 'Rain', 'Rain', 'Rain', 'Rain',
                 'Snow', 'Snow', 'Snow', 'Snow', 'Snow'],
    
    'Level': [1, 2, 3, 4, 5] * 5,
    
    'Precision': [0.86978, 0.88526, 0.87290, 0.94094, 0.91958,
                  0.88332, 0.89776, 0.82090, 0.90290, 0.89695,
                  0.87056, 0.87767, 0.85226, 0.86615, 0.87722,
                  0.88266, 0.88925, 0.87938, 0.89466, 0.88396,
                  0.88557, 0.90002, 0.91712, 0.87905, 0.91571],
    
    'Recall': [0.86492, 0.91831, 0.91789, 0.94387, 0.90433,
               0.89204, 0.90227, 0.85636, 0.91448, 0.90288,
               0.92738, 0.92431, 0.87911, 0.89334, 0.90877,
               0.90179, 0.89238, 0.89527, 0.89551, 0.87448,
               0.89094, 0.92161, 0.89601, 0.89209, 0.86560],
    
    'F1-Score': [0.86148, 0.89782, 0.88912, 0.94086, 0.90680,
                 0.87870, 0.89551, 0.83416, 0.90326, 0.89469,
                 0.89317, 0.89619, 0.86175, 0.87593, 0.88912,
                 0.88675, 0.88777, 0.88239, 0.88785, 0.86993,
                 0.87976, 0.90541, 0.89450, 0.87407, 0.87250],
    
    'mAP50': [0.61770, 0.59718, 0.52854, 0.44323, 0.36818,
              0.64015, 0.63246, 0.62027, 0.61642, 0.61450,
              0.62412, 0.62348, 0.61321, 0.59589, 0.58178,
              0.63310, 0.63566, 0.64080, 0.63117, 0.62348,
              0.61642, 0.61450, 0.54843, 0.40218, 0.36305],
    
    'mAP50-95': [0.51620, 0.49876, 0.45991, 0.41165, 0.36567,
                 0.53311, 0.52895, 0.52185, 0.51835, 0.51750,
                 0.52233, 0.51843, 0.51041, 0.50231, 0.49199,
                 0.52773, 0.53081, 0.53099, 0.52907, 0.52426,
                 0.52216, 0.51641, 0.48286, 0.40442, 0.37921]
}

df_detr = pd.DataFrame(data_detr)

# =========================
# 1. GRAPHIQUES GLOBAUX - CORRIGÉS
# =========================
print(" GRAPHIQUES GLOBAUX DETR")
print("="*50)

# Calcul des moyennes globales par condition
summary_detr = df_detr.groupby('Condition').agg({
    'Precision': 'mean',
    'Recall': 'mean', 
    'F1-Score': 'mean',
    'mAP50': 'mean',
    'mAP50-95': 'mean'
}).round(4)

# Graphique 1: Barres des moyennes globales - LÉGENDE AVEC CONDITIONS
plt.figure(figsize=(14, 10))

# Sous-graphique 1: Precision par condition
plt.subplot(2, 2, 1)
conditions = ['Fog', 'Glare', 'Night', 'Rain', 'Snow']
precision_values = [summary_detr.loc[cond, 'Precision'] for cond in conditions]
colors = ['lightblue', 'gold', 'lightgray', 'lightgreen', 'lightcoral']

bars = plt.bar(conditions, precision_values, color=colors, alpha=0.8, edgecolor='black')
plt.xlabel('Conditions Météorologiques')
plt.ylabel('Precision')
plt.title('DETR - Precision Moyenne par Condition')
plt.grid(True, alpha=0.3)
plt.ylim(0.8, 0.95)

# Ajouter les valeurs sur les barres
for bar, value in zip(bars, precision_values):
    plt.text(bar.get_x() + bar.get_width()/2, value + 0.005, f'{value:.3f}', 
             ha='center', va='bottom', fontweight='bold')

# Sous-graphique 2: Recall par condition
plt.subplot(2, 2, 2)
recall_values = [summary_detr.loc[cond, 'Recall'] for cond in conditions]

bars = plt.bar(conditions, recall_values, color=colors, alpha=0.8, edgecolor='black')
plt.xlabel('Conditions Météorologiques')
plt.ylabel('Recall')
plt.title('DETR - Recall Moyen par Condition')
plt.grid(True, alpha=0.3)
plt.ylim(0.8, 0.95)

# Ajouter les valeurs sur les barres
for bar, value in zip(bars, recall_values):
    plt.text(bar.get_x() + bar.get_width()/2, value + 0.005, f'{value:.3f}', 
             ha='center', va='bottom', fontweight='bold')

# Sous-graphique 3: mAP50 par condition
plt.subplot(2, 2, 3)
map50_values = [summary_detr.loc[cond, 'mAP50'] for cond in conditions]

bars = plt.bar(conditions, map50_values, color=colors, alpha=0.8, edgecolor='black')
plt.xlabel('Conditions Météorologiques')
plt.ylabel('mAP50')
plt.title('DETR - mAP50 Moyen par Condition')
plt.grid(True, alpha=0.3)
plt.ylim(0.4, 0.7)

# Ajouter les valeurs sur les barres
for bar, value in zip(bars, map50_values):
    plt.text(bar.get_x() + bar.get_width()/2, value + 0.005, f'{value:.3f}', 
             ha='center', va='bottom', fontweight='bold')

# Sous-graphique 4: Comparaison Precision vs Recall
plt.subplot(2, 2, 4)
x = np.arange(len(conditions))
width = 0.35

plt.bar(x - width/2, precision_values, width, label='Precision', 
        color='blue', alpha=0.7)
plt.bar(x + width/2, recall_values, width, label='Recall', 
        color='orange', alpha=0.7)

plt.xlabel('Conditions Météorologiques')
plt.ylabel('Score')
plt.title('DETR - Comparaison Precision vs Recall')
plt.xticks(x, conditions)
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(0.8, 0.95)

plt.tight_layout()
plt.suptitle('DETR - VUE GLOBALE DES PERFORMANCES', fontsize=16, fontweight='bold', y=1.02)
plt.show()

# =========================
# 2. GRAPHIQUES PAR MÉTRIQUE - LÉGENDE AVEC CONDITIONS
# =========================
print("\n GRAPHIQUES PAR MÉTRIQUE")
print("="*50)

# Graphique 2: Évolution de chaque métrique avec légende des conditions
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('DETR - Évolution des Métriques par Condition', fontsize=16, fontweight='bold')

metrics_to_plot = ['Precision', 'Recall', 'F1-Score', 'mAP50', 'mAP50-95']
colors = ['blue', 'orange', 'green', 'red', 'purple']

for idx, metric in enumerate(metrics_to_plot):
    ax = axes[idx//3, idx%3]
    
    for i, condition in enumerate(conditions):
        condition_data = df_detr[df_detr['Condition'] == condition]
        ax.plot(condition_data['Level'], condition_data[metric], 
                marker='o', linewidth=2, label=condition, color=colors[i])
    
    ax.set_xlabel('Niveau')
    ax.set_ylabel(metric)  # Ici on met le nom de la métrique sur l'axe Y
    ax.set_title(f'Évolution de {metric} par Condition')
    ax.set_xticks([1, 2, 3, 4, 5])
    ax.legend()  # Légende avec les conditions météo
    ax.grid(True, alpha=0.3)
    
    # Ajuster les limites Y selon la métrique
    if metric in ['Precision', 'Recall', 'F1-Score']:
        ax.set_ylim(0.8, 1.0)
    else:
        ax.set_ylim(0.3, 0.7)

# Cacher le dernier subplot vide si nécessaire
if len(metrics_to_plot) < 6:
    axes[1, 2].set_visible(False)

plt.tight_layout()
plt.show()

# =========================
# 3. GRAPHIQUES EN BARRES PAR NIVEAU - LÉGENDE AVEC CONDITIONS
# =========================
print("\n GRAPHIQUES EN BARRES PAR NIVEAU")
print("="*50)

# Graphique 3: Barres groupées par niveau avec conditions dans la légende
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('DETR - Performances par Niveau et Condition', fontsize=16, fontweight='bold')

levels = [1, 2, 3, 4, 5]
colors = ['lightblue', 'gold', 'lightgray', 'lightgreen', 'lightcoral']

for idx, level in enumerate(levels):
    ax = axes[idx//3, idx%2] if idx < 5 else axes[1, 2]
    
    if idx < 5:  # Pour les 5 premiers niveaux
        level_data = df_detr[df_detr['Level'] == level]
        
        # Préparer les données pour ce niveau
        precision_vals = [level_data[level_data['Condition'] == cond]['Precision'].values[0] for cond in conditions]
        recall_vals = [level_data[level_data['Condition'] == cond]['Recall'].values[0] for cond in conditions]
        
        x = np.arange(len(conditions))
        width = 0.35
        
        bars1 = ax.bar(x - width/2, precision_vals, width, label='Precision', 
                      color='blue', alpha=0.7)
        bars2 = ax.bar(x + width/2, recall_vals, width, label='Recall', 
                      color='orange', alpha=0.7)
        
        ax.set_xlabel('Conditions Météorologiques')
        ax.set_ylabel('Score')
        ax.set_title(f'Niveau {level} - Precision vs Recall')
        ax.set_xticks(x)
        ax.set_xticklabels(conditions)
        ax.legend()
        ax.grid(True, alpha=0.3)
        ax.set_ylim(0.8, 1.0)
        
        # Ajouter les valeurs sur les barres
        for bars in [bars1, bars2]:
            for bar in bars:
                height = bar.get_height()
                ax.text(bar.get_x() + bar.get_width()/2, height + 0.005, f'{height:.3f}', 
                       ha='center', va='bottom', fontsize=8)

# Graphique final: mAP50 par niveau avec conditions dans la légende
plt.figure(figsize=(12, 8))

for i, condition in enumerate(conditions):
    condition_data = df_detr[df_detr['Condition'] == condition]
    plt.plot(condition_data['Level'], condition_data['mAP50'], 
             marker='o', linewidth=3, label=condition, color=colors[i], markersize=8)

plt.xlabel('Niveau')
plt.ylabel('mAP50')
plt.title('DETR - Évolution du mAP50 par Condition')
plt.xticks([1, 2, 3, 4, 5])
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(0.3, 0.7)

plt.tight_layout()
plt.show()

# =========================
# RÉSUMÉ FINAL
# =========================
print("\n" + "="*60)
print(" RÉSUMÉ DES PERFORMANCES DETR")
print("="*60)

print(" CLASSEMENT PAR CONDITION (mAP50):")
ranking = summary_detr['mAP50'].sort_values(ascending=False)
for i, (condition, score) in enumerate(ranking.items(), 1):
    print(f"  {i}. {condition}: {score:.3f}")

print(f"\n PERFORMANCE MOYENNE:")
print(f"  Precision: {summary_detr['Precision'].mean():.3f}")
print(f"  Recall: {summary_detr['Recall'].mean():.3f}")
print(f"  mAP50: {summary_detr['mAP50'].mean():.3f}")

print(f"\n Graphiques générés avec légende des conditions météo!")

 GRAPHIQUES GLOBAUX DETR


<Figure size 1400x1000 with 4 Axes>


 GRAPHIQUES PAR MÉTRIQUE


<Figure size 1800x1200 with 6 Axes>


 GRAPHIQUES EN BARRES PAR NIVEAU


<Figure size 1800x1200 with 6 Axes>

<Figure size 1200x800 with 1 Axes>


 RÉSUMÉ DES PERFORMANCES DETR
 CLASSEMENT PAR CONDITION (mAP50):
  1. Rain: 0.633
  2. Glare: 0.625
  3. Night: 0.608
  4. Fog: 0.511
  5. Snow: 0.509

 PERFORMANCE MOYENNE:
  Precision: 0.886
  Recall: 0.899
  mAP50: 0.577

 Graphiques générés avec légende des conditions météo!


In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# =========================
# DONNÉES DETR EXACTES (copiées de vos résultats)
# =========================

data_detr = {
    'Condition': ['Fog', 'Fog', 'Fog', 'Fog', 'Fog',
                 'Glare', 'Glare', 'Glare', 'Glare', 'Glare',
                 'Night', 'Night', 'Night', 'Night', 'Night',
                 'Rain', 'Rain', 'Rain', 'Rain', 'Rain',
                 'Snow', 'Snow', 'Snow', 'Snow', 'Snow'],
    
    'Level': [1, 2, 3, 4, 5] * 5,
    
    'Precision': [0.8698, 0.8853, 0.8729, 0.9409, 0.9196,
                  0.8833, 0.8978, 0.8209, 0.9029, 0.8970,
                  0.8706, 0.8777, 0.8523, 0.8661, 0.8772,
                  0.8827, 0.8892, 0.8794, 0.8947, 0.8840,
                  0.8856, 0.9000, 0.9171, 0.8790, 0.9157],
    
    'Recall': [0.8649, 0.9183, 0.9179, 0.9439, 0.9043,
               0.8920, 0.9023, 0.8564, 0.9145, 0.9029,
               0.9274, 0.9243, 0.8791, 0.8933, 0.9088,
               0.9018, 0.8924, 0.8953, 0.8955, 0.8745,
               0.8909, 0.9216, 0.8960, 0.8921, 0.8656],
    
    'F1-Score': [0.8615, 0.8978, 0.8891, 0.9409, 0.9068,
                 0.8787, 0.8955, 0.8342, 0.9033, 0.8947,
                 0.8932, 0.8962, 0.8618, 0.8759, 0.8891,
                 0.8868, 0.8878, 0.8824, 0.8879, 0.8699,
                 0.8798, 0.9054, 0.8945, 0.8741, 0.8725],
    
    'mAP50': [0.6177, 0.5972, 0.5285, 0.4432, 0.3682,
              0.6401, 0.6325, 0.6203, 0.6164, 0.6145,
              0.6241, 0.6235, 0.6132, 0.5959, 0.5818,
              0.6331, 0.6357, 0.6408, 0.6312, 0.6235,
              0.6164, 0.6145, 0.5484, 0.4022, 0.3630],
    
    'mAP50-95': [0.5162, 0.4988, 0.4599, 0.4117, 0.3657,
                 0.5331, 0.5290, 0.5219, 0.5183, 0.5175,
                 0.5223, 0.5184, 0.5104, 0.5023, 0.4920,
                 0.5277, 0.5308, 0.5310, 0.5291, 0.5243,
                 0.5222, 0.5164, 0.4829, 0.4044, 0.3792]
}

df_detr = pd.DataFrame(data_detr)

# =========================
# 1. GRAPHIQUES GLOBAUX - PRECISION ET RECALL
# =========================
print(" DETR - GRAPHIQUES GLOBAUX (Précision et Recall)")
print("="*60)

# Calcul des moyennes globales par condition
summary_detr = df_detr.groupby('Condition').agg({
    'Precision': 'mean',
    'Recall': 'mean', 
    'F1-Score': 'mean',
    'mAP50': 'mean',
    'mAP50-95': 'mean'
}).round(4)

# Graphique 1: Precision et Recall par condition
plt.figure(figsize=(15, 10))

# Sous-graphique 1: Precision par condition
plt.subplot(2, 2, 1)
conditions = ['Fog', 'Glare', 'Night', 'Rain', 'Snow']
precision_values = [summary_detr.loc[cond, 'Precision'] for cond in conditions]
colors = ['lightblue', 'gold', 'lightgray', 'lightgreen', 'lightcoral']

bars = plt.bar(conditions, precision_values, color=colors, alpha=0.8, edgecolor='black')
plt.xlabel('Conditions Météorologiques', fontweight='bold')
plt.ylabel('Precision', fontweight='bold')
plt.title('DETR - Precision Moyenne par Condition', fontweight='bold')
plt.grid(True, alpha=0.3)
plt.ylim(0.85, 0.95)

# Ajouter les valeurs sur les barres
for bar, value in zip(bars, precision_values):
    plt.text(bar.get_x() + bar.get_width()/2, value + 0.003, f'{value:.3f}', 
             ha='center', va='bottom', fontweight='bold', fontsize=10)

# Sous-graphique 2: Recall par condition
plt.subplot(2, 2, 2)
recall_values = [summary_detr.loc[cond, 'Recall'] for cond in conditions]

bars = plt.bar(conditions, recall_values, color=colors, alpha=0.8, edgecolor='black')
plt.xlabel('Conditions Météorologiques', fontweight='bold')
plt.ylabel('Recall', fontweight='bold')
plt.title('DETR - Recall Moyen par Condition', fontweight='bold')
plt.grid(True, alpha=0.3)
plt.ylim(0.85, 0.95)

# Ajouter les valeurs sur les barres
for bar, value in zip(bars, recall_values):
    plt.text(bar.get_x() + bar.get_width()/2, value + 0.003, f'{value:.3f}', 
             ha='center', va='bottom', fontweight='bold', fontsize=10)

# Sous-graphique 3: Comparaison Precision vs Recall
plt.subplot(2, 2, 3)
x = np.arange(len(conditions))
width = 0.35

bars1 = plt.bar(x - width/2, precision_values, width, label='Precision', 
                color='blue', alpha=0.7, edgecolor='black')
bars2 = plt.bar(x + width/2, recall_values, width, label='Recall', 
                color='orange', alpha=0.7, edgecolor='black')

plt.xlabel('Conditions Météorologiques', fontweight='bold')
plt.ylabel('Score', fontweight='bold')
plt.title('DETR - Comparaison Precision vs Recall', fontweight='bold')
plt.xticks(x, conditions)
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(0.85, 0.95)

# Sous-graphique 4: mAP50 par condition
plt.subplot(2, 2, 4)
map50_values = [summary_detr.loc[cond, 'mAP50'] for cond in conditions]

bars = plt.bar(conditions, map50_values, color=colors, alpha=0.8, edgecolor='black')
plt.xlabel('Conditions Météorologiques', fontweight='bold')
plt.ylabel('mAP50', fontweight='bold')
plt.title('DETR - mAP50 Moyen par Condition', fontweight='bold')
plt.grid(True, alpha=0.3)
plt.ylim(0.4, 0.7)

# Ajouter les valeurs sur les barres
for bar, value in zip(bars, map50_values):
    plt.text(bar.get_x() + bar.get_width()/2, value + 0.005, f'{value:.3f}', 
             ha='center', va='bottom', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()

# =========================
# 2. GRAPHIQUES D'ÉVOLUTION PAR CONDITION
# =========================
print("\n DETR - ÉVOLUTION PAR CONDITION MÉTÉO")
print("="*60)

# Graphique 2: Évolution de la Precision par condition
plt.figure(figsize=(14, 10))

# Couleurs pour chaque condition
colors = {'Fog': 'blue', 'Glare': 'orange', 'Night': 'black', 
          'Rain': 'green', 'Snow': 'red'}

# Sous-graphique 1: Évolution de la Precision
plt.subplot(2, 2, 1)
for condition in conditions:
    condition_data = df_detr[df_detr['Condition'] == condition]
    plt.plot(condition_data['Level'], condition_data['Precision'], 
             marker='o', linewidth=3, label=condition, color=colors[condition], markersize=8)

plt.xlabel('Niveau', fontweight='bold')
plt.ylabel('Precision', fontweight='bold')
plt.title('Évolution de la Precision par Condition', fontweight='bold')
plt.xticks([1, 2, 3, 4, 5])
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(0.8, 1.0)

# Sous-graphique 2: Évolution du Recall
plt.subplot(2, 2, 2)
for condition in conditions:
    condition_data = df_detr[df_detr['Condition'] == condition]
    plt.plot(condition_data['Level'], condition_data['Recall'], 
             marker='s', linewidth=3, label=condition, color=colors[condition], markersize=8)

plt.xlabel('Niveau', fontweight='bold')
plt.ylabel('Recall', fontweight='bold')
plt.title('Évolution du Recall par Condition', fontweight='bold')
plt.xticks([1, 2, 3, 4, 5])
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(0.8, 1.0)

# Sous-graphique 3: Évolution du mAP50
plt.subplot(2, 2, 3)
for condition in conditions:
    condition_data = df_detr[df_detr['Condition'] == condition]
    plt.plot(condition_data['Level'], condition_data['mAP50'], 
             marker='^', linewidth=3, label=condition, color=colors[condition], markersize=8)

plt.xlabel('Niveau', fontweight='bold')
plt.ylabel('mAP50', fontweight='bold')
plt.title('Évolution du mAP50 par Condition', fontweight='bold')
plt.xticks([1, 2, 3, 4, 5])
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(0.3, 0.7)

# Sous-graphique 4: Évolution du F1-Score
plt.subplot(2, 2, 4)
for condition in conditions:
    condition_data = df_detr[df_detr['Condition'] == condition]
    plt.plot(condition_data['Level'], condition_data['F1-Score'], 
             marker='D', linewidth=3, label=condition, color=colors[condition], markersize=8)

plt.xlabel('Niveau', fontweight='bold')
plt.ylabel('F1-Score', fontweight='bold')
plt.title('Évolution du F1-Score par Condition', fontweight='bold')
plt.xticks([1, 2, 3, 4, 5])
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(0.8, 1.0)

plt.tight_layout()
plt.show()

# =========================
# 3. GRAPHIQUES EN BARRES DÉTAILLÉS
# =========================
print("\n DETR - GRAPHIQUES EN BARRES DÉTAILLÉS")
print("="*60)

# Graphique 3: Barres groupées par niveau
fig, axes = plt.subplots(3, 2, figsize=(16, 12))
fig.suptitle('DETR - Performances Détaillées par Niveau', fontweight='bold', fontsize=16)

metrics_to_plot = ['Precision', 'Recall', 'F1-Score', 'mAP50', 'mAP50-95']

for idx, metric in enumerate(metrics_to_plot):
    ax = axes[idx//2, idx%2]
    
    x = np.arange(5)  # 5 niveaux
    width = 0.15
    
    for i, condition in enumerate(conditions):
        condition_data = df_detr[df_detr['Condition'] == condition][metric].values
        ax.bar(x + i*width, condition_data, width, label=condition, 
               color=colors[condition], alpha=0.8, edgecolor='black')
    
    ax.set_xlabel('Niveau', fontweight='bold')
    ax.set_ylabel(metric, fontweight='bold')
    ax.set_title(f'{metric} par Niveau et Condition', fontweight='bold')
    ax.set_xticks(x + 2*width)
    ax.set_xticklabels([1, 2, 3, 4, 5])
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    
    # Ajuster les limites Y selon la métrique
    if metric in ['Precision', 'Recall', 'F1-Score']:
        ax.set_ylim(0.8, 1.0)
    else:
        ax.set_ylim(0.3, 0.7)

# Cacher le dernier subplot vide
axes[2, 1].set_visible(False)

plt.tight_layout()
plt.show()

# =========================
# RÉSUMÉ DES PERFORMANCES
# =========================
print("\n" + "="*60)
print(" RÉSUMÉ DETR - PERFORMANCES PAR CONDITION")
print("="*60)

print(" MOYENNES PAR CONDITION:")
print(summary_detr.round(4))

print(f"\n MEILLEURES PERFORMANCES:")
print(f"  Precision: {summary_detr['Precision'].idxmax()} ({summary_detr['Precision'].max():.3f})")
print(f"  Recall: {summary_detr['Recall'].idxmax()} ({summary_detr['Recall'].max():.3f})")
print(f"  mAP50: {summary_detr['mAP50'].idxmax()} ({summary_detr['mAP50'].max():.3f})")

print(f"\n PERFORMANCE MOYENNE GLOBALE:")
print(f"  Precision: {summary_detr['Precision'].mean():.3f}")
print(f"  Recall: {summary_detr['Recall'].mean():.3f}")
print(f"  mAP50: {summary_detr['mAP50'].mean():.3f}")

print(f"\n {len(metrics_to_plot)} graphiques générés avec les VRAIES données DETR!")

 DETR - GRAPHIQUES GLOBAUX (Précision et Recall)


<Figure size 1500x1000 with 4 Axes>


 DETR - ÉVOLUTION PAR CONDITION MÉTÉO


<Figure size 1400x1000 with 4 Axes>


 DETR - GRAPHIQUES EN BARRES DÉTAILLÉS


<Figure size 1600x1200 with 6 Axes>


 RÉSUMÉ DETR - PERFORMANCES PAR CONDITION
 MOYENNES PAR CONDITION:
           Precision  Recall  F1-Score   mAP50  mAP50-95
Condition                                               
Fog           0.8977  0.9099    0.8992  0.5110    0.4505
Glare         0.8804  0.8936    0.8813  0.6248    0.5240
Night         0.8688  0.9066    0.8832  0.6077    0.5091
Rain          0.8860  0.8919    0.8830  0.6329    0.5286
Snow          0.8995  0.8932    0.8853  0.5089    0.4610

 MEILLEURES PERFORMANCES:
  Precision: Snow (0.899)
  Recall: Fog (0.910)
  mAP50: Rain (0.633)

 PERFORMANCE MOYENNE GLOBALE:
  Precision: 0.886
  Recall: 0.899
  mAP50: 0.577

 5 graphiques générés avec les VRAIES données DETR!


# RF-DETR Train

In [2]:
import os
import torch
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import RTDetrForObjectDetection, RTDetrImageProcessor
from tqdm import tqdm
from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd
from torchvision.ops import box_iou

# =========================
# CONFIGURATION
# =========================
DEVICE = "cpu"  # Utiliser "cuda" si GPU disponible
print(f"DEVICE: {DEVICE}")

TRAIN_IMAGES_DIR = "./Data3/train/images"
TRAIN_LABELS_DIR = "./Data3/train/labels"
TEST_IMAGES_DIR = "./Data3/test/images"
TEST_LABELS_DIR = "./Data3/test/labels"
SAVE_DIR = "./models_rfdetr"

os.makedirs(SAVE_DIR, exist_ok=True)

NUM_CLASSES = 5
BATCH_SIZE = 2
EPOCHS = 10
LEARNING_RATE = 1e-4

# =========================
# DATASET YOLO FORMAT
# =========================
class YoloDatasetForRTDetr(Dataset):
    def __init__(self, images_dir, labels_dir):
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.images = sorted([f for f in os.listdir(images_dir) 
                            if f.lower().endswith((".jpg", ".jpeg", ".png"))])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.images_dir, img_name)
        label_path = os.path.join(self.labels_dir, os.path.splitext(img_name)[0] + ".txt")

        img = Image.open(img_path).convert("RGB")
        w, h = img.size

        boxes = []
        labels = []
        
        if os.path.exists(label_path):
            with open(label_path, "r") as f:
                for line in f.readlines():
                    line = line.strip()
                    if not line:
                        continue
                    
                    parts = line.split()
                    if len(parts) != 5:
                        continue
                    
                    try:
                        cls, x_center, y_center, width, height = map(float, parts)
                        
                        # Validation
                        if not (0 <= x_center <= 1 and 0 <= y_center <= 1 and 
                                0 < width <= 1 and 0 < height <= 1):
                            continue
                        
                        if not (0 <= int(cls) < NUM_CLASSES):
                            continue
                        
                        # Vérifier que la box est valide
                        x1 = x_center - width / 2
                        y1 = y_center - height / 2
                        x2 = x_center + width / 2
                        y2 = y_center + height / 2
                        
                        if x2 <= x1 or y2 <= y1:
                            continue
                        
                        # RT-DETR attend format cxcywh normalisé
                        boxes.append([x_center, y_center, width, height])
                        labels.append(int(cls))
                        
                    except (ValueError, OverflowError):
                        continue

        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)
        
        target = {"boxes": boxes, "class_labels": labels}
        return img, target

# =========================
# HELPER FUNCTIONS
# =========================
def cxcywh_to_xyxy(boxes, img_w, img_h):
    if boxes.numel() == 0:
        return torch.zeros((0, 4), dtype=torch.float32)
    cx = boxes[:, 0] * img_w
    cy = boxes[:, 1] * img_h
    bw = boxes[:, 2] * img_w
    bh = boxes[:, 3] * img_h
    x_min = cx - bw/2
    y_min = cy - bh/2
    x_max = cx + bw/2
    y_max = cy + bh/2
    return torch.stack([x_min, y_min, x_max, y_max], dim=1)

def collate_fn(batch):
    imgs, targets = list(zip(*batch))
    return list(imgs), list(targets)

# =========================
# LOAD DATASETS
# =========================
train_dataset = YoloDatasetForRTDetr(TRAIN_IMAGES_DIR, TRAIN_LABELS_DIR)
test_dataset = YoloDatasetForRTDetr(TEST_IMAGES_DIR, TEST_LABELS_DIR)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f"Train images: {len(train_dataset)}, Test images: {len(test_dataset)}")

# Vérifier quelques échantillons
print("\n=== Vérification de 3 échantillons ===")
for i in range(min(3, len(train_dataset))):
    img, target = train_dataset[i]
    print(f"Échantillon {i}: {len(target['class_labels'])} objets, classes: {target['class_labels'].tolist()}")

# =========================
# MODEL INITIALIZATION
# =========================
id2label = {i: f"class{i}" for i in range(NUM_CLASSES)}
label2id = {f"class{i}": i for i in range(NUM_CLASSES)}

print("\n Chargement du modèle RT-DETR...")
model = RTDetrForObjectDetection.from_pretrained(
    "PekingU/rtdetr_r50vd",
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)
model.to(DEVICE)
print(" Modèle RT-DETR chargé")

processor = RTDetrImageProcessor.from_pretrained("PekingU/rtdetr_r50vd")
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

# =========================
# TRAINING FUNCTION
# =========================
def train_one_epoch(model, loader, optimizer):
    model.train()
    running_loss = 0.0
    n_batches = 0
    
    for imgs, annotations in tqdm(loader, desc="Train"):
        encoding = processor(images=imgs, return_tensors="pt")
        pixel_values = encoding["pixel_values"].to(DEVICE)

        # Filtrer les annotations vides
        labels = []
        for ann in annotations:
            if ann["class_labels"].numel() > 0:
                labels.append({
                    "class_labels": ann["class_labels"].to(DEVICE),
                    "boxes": ann["boxes"].to(DEVICE)
                })
        
        if len(labels) == 0:
            continue
        
        # Ajuster pixel_values si nécessaire
        if len(labels) < len(imgs):
            pixel_values = pixel_values[:len(labels)]

        optimizer.zero_grad()
        
        outputs = model(pixel_values=pixel_values, labels=labels)
        loss = outputs.loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.1)
        optimizer.step()

        running_loss += loss.item()
        n_batches += 1

    avg_loss = running_loss / max(n_batches, 1)
    return avg_loss

# =========================
# EVALUATION FUNCTION
# =========================
def evaluate(model, loader, conf_threshold=0.1):
    model.eval()
    all_matched_gt_labels = []
    all_matched_pred_labels = []
    all_ious = []
    total_detections = 0
    total_gt = 0

    with torch.no_grad():
        for imgs, annotations in loader:
            encoding = processor(images=imgs, return_tensors="pt")
            pixel_values = encoding["pixel_values"].to(DEVICE)

            outputs = model(pixel_values=pixel_values)
            
            # Post-process
            target_sizes = torch.tensor([img.size[::-1] for img in imgs])
            results = processor.post_process_object_detection(
                outputs, target_sizes=target_sizes, threshold=conf_threshold
            )

            for i, res in enumerate(results):
                pred_boxes = res["boxes"].clone() if len(res["boxes"]) > 0 else torch.zeros((0, 4), dtype=torch.float32)
                pred_labels = res["labels"].clone() if len(res["labels"]) > 0 else torch.zeros((0,), dtype=torch.int64)
                
                total_detections += len(pred_labels)
                
                ann = annotations[i]
                gt_boxes_norm = ann["boxes"]
                gt_labels = ann["class_labels"]
                
                total_gt += len(gt_labels)
                
                if len(pred_boxes) == 0 or len(gt_boxes_norm) == 0:
                    continue
                
                # Convertir GT vers xyxy absolu
                img_w, img_h = imgs[i].size
                gt_boxes_xyxy = cxcywh_to_xyxy(gt_boxes_norm, img_w, img_h)
                
                # Calculer IoU
                iou = box_iou(gt_boxes_xyxy, pred_boxes)
                
                # Pour chaque GT, trouver la meilleure prédiction
                max_iou_per_gt, best_pred_idx = iou.max(dim=1)
                all_ious.extend(max_iou_per_gt.cpu().numpy().tolist())
                
                # Matcher les labels
                matched_mask = max_iou_per_gt >= 0.5
                
                if matched_mask.sum() > 0:
                    matched_gt_labels = gt_labels[matched_mask]
                    matched_pred_labels = pred_labels[best_pred_idx[matched_mask]]
                    
                    all_matched_gt_labels.extend(matched_gt_labels.cpu().numpy().tolist())
                    all_matched_pred_labels.extend(matched_pred_labels.cpu().numpy().tolist())

    # Calcul des métriques
    if len(all_matched_gt_labels) > 0 and len(all_matched_pred_labels) > 0:
        precision = precision_score(all_matched_gt_labels, all_matched_pred_labels, 
                                    average="macro", zero_division=0)
        recall = recall_score(all_matched_gt_labels, all_matched_pred_labels, 
                             average="macro", zero_division=0)
        f1 = f1_score(all_matched_gt_labels, all_matched_pred_labels, 
                     average="macro", zero_division=0)
    else:
        precision, recall, f1 = 0, 0, 0

    mAP50 = np.mean([iou >= 0.5 for iou in all_ious]) if all_ious else 0
    mAP5095 = np.mean(all_ious) if all_ious else 0

    print(f"  -> Détections: {total_detections}, GT: {total_gt}, Matchés: {len(all_matched_gt_labels)}")

    return {
        "Box-P": round(precision, 5),
        "Box-R": round(recall, 5),
        "Box-F1": round(f1, 5),
        "mAP50": round(mAP50, 5),
        "mAP50-95": round(mAP5095, 5)
    }

# =========================
# EARLY STOPPING
# =========================
class EarlyStopping:
    def __init__(self, patience=3, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.best_epoch = 0
    
    def __call__(self, score, epoch):
        if self.best_score is None:
            self.best_score = score
            self.best_epoch = epoch
            print(f" Premier meilleur F1: {score:.4f}")
        elif score < self.best_score + self.min_delta:
            self.counter += 1
            print(f" Early stopping counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
                print(f" EARLY STOPPING à epoch {epoch}! Meilleur: epoch {self.best_epoch} (F1={self.best_score:.4f})")
        else:
            self.best_score = score
            self.best_epoch = epoch
            self.counter = 0
            print(f" Nouveau meilleur F1: {score:.4f}")

early_stopping = EarlyStopping(patience=3, min_delta=0.001)

# =========================
# TRAINING LOOP
# =========================
train_history = []
test_history = []

print("\n" + "="*70)
print("DÉBUT DE L'ENTRAÎNEMENT RT-DETR")
print("="*70)

for epoch in range(1, EPOCHS + 1):
    print(f"\n{'='*70}")
    print(f"Epoch {epoch}/{EPOCHS}")
    print('='*70)
    
    avg_loss = train_one_epoch(model, train_loader, optimizer)
    print(f"Avg train loss: {avg_loss:.4f}")

    print("\nÉvaluation sur Train:")
    train_metrics = evaluate(model, train_loader)
    print(f"Train: {train_metrics}")

    print("\nÉvaluation sur Test:")
    test_metrics = evaluate(model, test_loader)
    print(f"Test: {test_metrics}")

    # Sauvegarder le modèle
    epoch_model_path = os.path.join(SAVE_DIR, f"rtdetr_epoch{epoch}.pt")
    torch.save(model.state_dict(), epoch_model_path)
    print(f" Model saved: {epoch_model_path}")

    # Enregistrer l'historique
    train_history.append({"epoch": epoch, "loss": avg_loss, **train_metrics})
    test_history.append({"epoch": epoch, **test_metrics})
    
    # Sauvegarder l'historique à chaque époque
    pd.DataFrame(train_history).to_csv(os.path.join(SAVE_DIR, "train_history.csv"), index=False)
    pd.DataFrame(test_history).to_csv(os.path.join(SAVE_DIR, "test_history.csv"), index=False)
    
    # Early Stopping Check
    early_stopping(test_metrics['Box-F1'], epoch)
    if early_stopping.early_stop:
        print(f"\n Entraînement arrêté à l'epoch {epoch}")
        print(f" Meilleur modèle: epoch {early_stopping.best_epoch}")
        break

print(f"\n{'='*70}")
print(" ENTRAÎNEMENT TERMINÉ!")
print(f" Modèles sauvegardés dans: {SAVE_DIR}")
print(f" Meilleur epoch: {early_stopping.best_epoch} (F1={early_stopping.best_score:.4f})")
print("="*70)

DEVICE: cpu
Train images: 677, Test images: 100

=== Vérification de 3 échantillons ===
Échantillon 0: 3 objets, classes: [4, 0, 1]
Échantillon 1: 3 objets, classes: [4, 0, 1]
Échantillon 2: 3 objets, classes: [4, 0, 1]

 Chargement du modèle RT-DETR...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/172M [00:00<?, ?B/s]

Some weights of RTDetrForObjectDetection were not initialized from the model checkpoint at PekingU/rtdetr_r50vd and are newly initialized because the shapes did not match:
- model.decoder.class_embed.0.bias: found shape torch.Size([80]) in the checkpoint and torch.Size([5]) in the model instantiated
- model.decoder.class_embed.0.weight: found shape torch.Size([80, 256]) in the checkpoint and torch.Size([5, 256]) in the model instantiated
- model.decoder.class_embed.1.bias: found shape torch.Size([80]) in the checkpoint and torch.Size([5]) in the model instantiated
- model.decoder.class_embed.1.weight: found shape torch.Size([80, 256]) in the checkpoint and torch.Size([5, 256]) in the model instantiated
- model.decoder.class_embed.2.bias: found shape torch.Size([80]) in the checkpoint and torch.Size([5]) in the model instantiated
- model.decoder.class_embed.2.weight: found shape torch.Size([80, 256]) in the checkpoint and torch.Size([5, 256]) in the model instantiated
- model.decoder.cl

 Modèle RT-DETR chargé


...

Avg train loss: 8.2621

Évaluation sur Train:
  -> Détections: 1463, GT: 2100, Matchés: 1381
Train: {'Box-P': 0.99713, 'Box-R': 0.99307, 'Box-F1': 0.99507, 'mAP50': np.float64(0.65762), 'mAP50-95': np.float64(0.61843)}

Évaluation sur Test:
  -> Détections: 212, GT: 317, Matchés: 189
Test: {'Box-P': 0.99108, 'Box-R': 0.97206, 'Box-F1': 0.98109, 'mAP50': np.float64(0.59621), 'mAP50-95': np.float64(0.55962)}
 Model saved: ./models_rfdetr\rtdetr_epoch8.pt
 Early stopping counter: 1/3

Epoch 9/10


Train: 100%|█████████████████████████████████████████████████████████████████████████| 339/339 [41:47<00:00,  7.40s/it]


Avg train loss: 8.0756

Évaluation sur Train:
  -> Détections: 1261, GT: 2100, Matchés: 1238
Train: {'Box-P': 0.99912, 'Box-R': 0.99799, 'Box-F1': 0.99855, 'mAP50': np.float64(0.58952), 'mAP50-95': np.float64(0.56751)}

Évaluation sur Test:
  -> Détections: 176, GT: 317, Matchés: 170
Test: {'Box-P': 0.99608, 'Box-R': 0.98857, 'Box-F1': 0.99214, 'mAP50': np.float64(0.53628), 'mAP50-95': np.float64(0.51274)}
 Model saved: ./models_rfdetr\rtdetr_epoch9.pt
 Nouveau meilleur F1: 0.9921

Epoch 10/10


Train: 100%|█████████████████████████████████████████████████████████████████████████| 339/339 [42:07<00:00,  7.46s/it]


Avg train loss: 7.9029

Évaluation sur Train:
  -> Détections: 1098, GT: 2100, Matchés: 1082
Train: {'Box-P': 0.99912, 'Box-R': 0.99785, 'Box-F1': 0.99848, 'mAP50': np.float64(0.51524), 'mAP50-95': np.float64(0.51273)}

Évaluation sur Test:
  -> Détections: 143, GT: 317, Matchés: 145
Test: {'Box-P': 0.99608, 'Box-R': 0.98571, 'Box-F1': 0.99061, 'mAP50': np.float64(0.45741), 'mAP50-95': np.float64(0.45601)}
 Model saved: ./models_rfdetr\rtdetr_epoch10.pt
 Early stopping counter: 1/3

 ENTRAÎNEMENT TERMINÉ!
 Modèles sauvegardés dans: ./models_rfdetr
 Meilleur epoch: 9 (F1=0.9921)


# Train Test RT-DETR 

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# =========================
# FONCTIONS UTILITAIRES
# =========================
def format_with_delta(test_value, train_value):
    """Formate une valeur avec son delta par rapport au train"""
    delta = test_value - train_value
    if abs(delta) < 0.001:
        return f"{test_value:.3f}"
    elif delta > 0:
        return f"{test_value:.3f}(+{delta:.3f})"
    else:
        return f"{test_value:.3f}({delta:.3f})"

def get_color_for_delta(delta):
    """Retourne la couleur selon le delta"""
    if delta > 0.01:
        return ""  # Vert pour amélioration
    elif delta < -0.01:
        return ""  # Rouge pour détérioration
    else:
        return ""  # Blanc pour stable

# =========================
# DONNÉES RT-DETR - TRAIN vs TEST
# =========================

# Données TRAIN RT-DETR
train_metrics = {
    'Precision': 0.99912,
    'Recall': 0.99785,
    'F1-Score': 0.99848,
    'mAP50': 0.51524,
    'mAP50-95': 0.51273
}

# Données TEST RT-DETR
test_metrics = {
    'Precision': 0.99608,
    'Recall': 0.98571,
    'F1-Score': 0.99061,
    'mAP50': 0.45741,
    'mAP50-95': 0.45601
}

# Statistiques de détection
detection_stats = {
    'Train': {'Detections': 1098, 'GT': 2100, 'Matched': 1082},
    'Test': {'Detections': 143, 'GT': 317, 'Matched': 145}
}

# =========================
# TABLEAUX COMPARATIFS RT-DETR
# =========================
print("\n" + "="*80)
print(" RT-DETR - COMPARAISON TRAIN vs TEST")
print("="*80)

# Tableau principal comparatif
comparison_data = []
for metric in train_metrics.keys():
    train_value = train_metrics[metric]
    test_value = test_metrics[metric]
    delta = test_value - train_value
    
    comparison_data.append({
        'Métrique': metric,
        'Train': f"{train_value:.5f}",
        'Test': format_with_delta(test_value, train_value),
        'Δ': f"{delta:+.5f}",
        'Variation': get_color_for_delta(delta),
        'Tendance': "" if delta > 0.01 else "" if delta < -0.01 else ""
    })

df_comparison = pd.DataFrame(comparison_data)
print(df_comparison.to_string(index=False))

# =========================
# TABLEAU DES STATISTIQUES DE DÉTECTION
# =========================
print("\n" + "="*60)
print(" RT-DETR - STATISTIQUES DE DÉTECTION")
print("="*60)

stats_data = []
for dataset in ['Train', 'Test']:
    stats = detection_stats[dataset]
    stats_data.append({
        'Dataset': dataset,
        'Détections': stats['Detections'],
        'Ground Truth': stats['GT'],
        'Matchés (TP)': stats['Matched'],
        'Précision Calculée': f"{stats['Matched']/stats['Detections']:.3f}" if stats['Detections'] > 0 else "N/A",
        'Recall Calculé': f"{stats['Matched']/stats['GT']:.3f}" if stats['GT'] > 0 else "N/A"
    })

df_stats = pd.DataFrame(stats_data)
print(df_stats.to_string(index=False))

# =========================
# GRAPHIQUES COMPARATIFS RT-DETR
# =========================

# 1. Graphique comparatif Train vs Test
plt.figure(figsize=(14, 10))

# Graphique 1: Barres comparatives
plt.subplot(2, 2, 1)
metrics = list(train_metrics.keys())
train_values = [train_metrics[m] for m in metrics]
test_values = [test_metrics[m] for m in metrics]

x = np.arange(len(metrics))
width = 0.35

bars1 = plt.bar(x - width/2, train_values, width, label='TRAIN', 
                color='#1f77b4', alpha=0.8, edgecolor='black')
bars2 = plt.bar(x + width/2, test_values, width, label='TEST', 
                color='#ff7f0e', alpha=0.8, edgecolor='black')

plt.xlabel('Métriques')
plt.ylabel('Score')
plt.title('RT-DETR - Comparaison TRAIN vs TEST')
plt.xticks(x, metrics, rotation=45, ha='right')
plt.legend()
plt.grid(True, alpha=0.3)

# Ajouter les valeurs sur les barres
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2, height + 0.01, f'{height:.3f}', 
                ha='center', va='bottom', fontsize=9, fontweight='bold')

# Graphique 2: Deltas (variations)
plt.subplot(2, 2, 2)
deltas = [test_values[i] - train_values[i] for i in range(len(metrics))]
colors = ['green' if delta > 0.01 else 'red' if delta < -0.01 else 'gray' for delta in deltas]

bars3 = plt.bar(metrics, deltas, color=colors, alpha=0.8, edgecolor='black')
plt.axhline(y=0, color='black', linestyle='-', alpha=0.8)
plt.xlabel('Métriques')
plt.ylabel('Δ (Test - Train)')
plt.title('Variations RT-DETR (TEST vs TRAIN)')
plt.xticks(rotation=45, ha='right')
plt.grid(True, alpha=0.3)

# Ajouter les valeurs sur les barres de delta
for bar, delta in zip(bars3, deltas):
    height = bar.get_height()
    va = 'bottom' if height >= 0 else 'top'
    y_pos = height + 0.002 if height >= 0 else height - 0.002
    color = 'darkgreen' if height > 0.01 else 'darkred' if height < -0.01 else 'black'
    
    plt.text(bar.get_x() + bar.get_width()/2, y_pos, f'{height:+.3f}', 
            ha='center', va=va, fontsize=10, fontweight='bold', color=color)

# Graphique 3: Métriques de base (Precision, Recall, F1)
plt.subplot(2, 2, 3)
basic_metrics = ['Precision', 'Recall', 'F1-Score']
basic_train = [train_metrics[m] for m in basic_metrics]
basic_test = [test_metrics[m] for m in basic_metrics]

x_basic = np.arange(len(basic_metrics))
width_basic = 0.35

plt.bar(x_basic - width_basic/2, basic_train, width_basic, label='TRAIN', 
        color='#1f77b4', alpha=0.8)
plt.bar(x_basic + width_basic/2, basic_test, width_basic, label='TEST', 
        color='#ff7f0e', alpha=0.8)

plt.xlabel('Métriques de Base')
plt.ylabel('Score')
plt.title('RT-DETR - Métriques de Base')
plt.xticks(x_basic, basic_metrics)
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(0.98, 1.005)

# Graphique 4: Métriques mAP
plt.subplot(2, 2, 4)
map_metrics = ['mAP50', 'mAP50-95']
map_train = [train_metrics[m] for m in map_metrics]
map_test = [test_metrics[m] for m in map_metrics]

x_map = np.arange(len(map_metrics))
width_map = 0.35

bars_map1 = plt.bar(x_map - width_map/2, map_train, width_map, label='TRAIN', 
                   color='#1f77b4', alpha=0.8)
bars_map2 = plt.bar(x_map + width_map/2, map_test, width_map, label='TEST', 
                   color='#ff7f0e', alpha=0.8)

plt.xlabel('Métriques mAP')
plt.ylabel('Score')
plt.title('RT-DETR - Métriques mAP')
plt.xticks(x_map, map_metrics)
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(0.4, 0.55)

# Ajouter les valeurs sur les barres mAP
for bars in [bars_map1, bars_map2]:
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2, height + 0.005, f'{height:.3f}', 
                ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

# =========================
# ANALYSE DES PERFORMANCES RT-DETR
# =========================
print("\n" + "="*60)
print(" RT-DETR - ANALYSE DES PERFORMANCES")
print("="*60)

print(" POINTS FORTS:")
print(f"   Excellente Precision: {test_metrics['Precision']:.3f}")
print(f"   Très bon Recall: {test_metrics['Recall']:.3f}")
print(f"   F1-Score exceptionnel: {test_metrics['F1-Score']:.3f}")

print("\n  POINTS À AMÉLIORER:")
print(f"   mAP50 en baisse: -{train_metrics['mAP50'] - test_metrics['mAP50']:.3f}")
print(f"   Écart Train-Test sur mAP: {(train_metrics['mAP50'] - test_metrics['mAP50'])/train_metrics['mAP50']*100:.1f}%")

print(f"\n PERFORMANCE GLOBALE:")
print(f"  Moyenne Precision/Recall/F1: {(test_metrics['Precision'] + test_metrics['Recall'] + test_metrics['F1-Score'])/3:.3f}")
print(f"  Moyenne mAP: {(test_metrics['mAP50'] + test_metrics['mAP50-95'])/2:.3f}")

# =========================
# COMPARAISON AVEC ATTENTES
# =========================
print("\n" + "="*60)
print(" RT-DETR - ÉVALUATION DE LA PERFORMANCE")
print("="*60)

# Seuils de performance
thresholds = {
    'Excellente': 0.95,
    'Très Bonne': 0.90,
    'Bonne': 0.80,
    'Moyenne': 0.70,
    'Faible': 0.60
}

def get_performance_level(value, metric_type):
    if metric_type == 'basic':
        if value >= 0.98: return " EXCELLENTE"
        elif value >= 0.95: return " TRÈS BONNE"
        elif value >= 0.90: return " BONNE"
        elif value >= 0.80: return " MOYENNE"
        else: return " FAIBLE"
    else:  # mAP
        if value >= 0.70: return " EXCELLENTE"
        elif value >= 0.60: return " TRÈS BONNE"
        elif value >= 0.50: return " BONNE"
        elif value >= 0.40: return " MOYENNE"
        else: return " FAIBLE"

print("NIVEAUX DE PERFORMANCE SUR TEST:")
for metric, value in test_metrics.items():
    level = get_performance_level(value, 'basic' if metric in ['Precision', 'Recall', 'F1-Score'] else 'mAP')
    print(f"  {metric}: {value:.3f} - {level}")

# =========================
# SAUVEGARDE DES RÉSULTATS
# =========================
save_dir = "./models_rfdetr"

# Créer DataFrames pour sauvegarde
df_results = pd.DataFrame({
    'Dataset': ['Train', 'Test'],
    'Precision': [train_metrics['Precision'], test_metrics['Precision']],
    'Recall': [train_metrics['Recall'], test_metrics['Recall']],
    'F1-Score': [train_metrics['F1-Score'], test_metrics['F1-Score']],
    'mAP50': [train_metrics['mAP50'], test_metrics['mAP50']],
    'mAP50-95': [train_metrics['mAP50-95'], test_metrics['mAP50-95']]
})

df_detection = pd.DataFrame(stats_data)

# Sauvegarder
df_results.to_csv(f"{save_dir}/rtdetr_performance.csv", index=False)
df_detection.to_csv(f"{save_dir}/rtdetr_detection_stats.csv", index=False)

print(f"\n RÉSULTATS RT-DETR SAUVEGARDÉS")
print(f" Dossier: {save_dir}")
print(f"   - rtdetr_performance.csv")
print(f"   - rtdetr_detection_stats.csv")


 RT-DETR - COMPARAISON TRAIN vs TEST
 Métrique   Train          Test        Δ Variation Tendance
Precision 0.99912 0.996(-0.003) -0.00304                
   Recall 0.99785 0.986(-0.012) -0.01214                 
 F1-Score 0.99848 0.991(-0.008) -0.00787                
    mAP50 0.51524 0.457(-0.058) -0.05783                 
 mAP50-95 0.51273 0.456(-0.057) -0.05672                 

 RT-DETR - STATISTIQUES DE DÉTECTION
Dataset  Détections  Ground Truth  Matchés (TP) Précision Calculée Recall Calculé
  Train        1098          2100          1082              0.985          0.515
   Test         143           317           145              1.014          0.457


<Figure size 1400x1000 with 4 Axes>


 RT-DETR - ANALYSE DES PERFORMANCES
 POINTS FORTS:
   Excellente Precision: 0.996
   Très bon Recall: 0.986
   F1-Score exceptionnel: 0.991

  POINTS À AMÉLIORER:
   mAP50 en baisse: -0.058
   Écart Train-Test sur mAP: 11.2%

 PERFORMANCE GLOBALE:
  Moyenne Precision/Recall/F1: 0.991
  Moyenne mAP: 0.457

 RT-DETR - ÉVALUATION DE LA PERFORMANCE
NIVEAUX DE PERFORMANCE SUR TEST:
  Precision: 0.996 -  EXCELLENTE
  Recall: 0.986 -  EXCELLENTE
  F1-Score: 0.991 -  EXCELLENTE
  mAP50: 0.457 -  MOYENNE
  mAP50-95: 0.456 -  MOYENNE

 RÉSULTATS RT-DETR SAUVEGARDÉS
 Dossier: ./models_rfdetr
   - rtdetr_performance.csv
   - rtdetr_detection_stats.csv


# RT-DETR

In [3]:
import os
import json
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import RTDetrForObjectDetection, RTDetrImageProcessor
from tqdm import tqdm
from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd
from torchvision.ops import box_iou
import glob

# =========================
# CONFIGURATION
# =========================
DEVICE = "cpu"
print(f"DEVICE: {DEVICE}")

RESULTS_BASE_DIR = "./Data3/results"
MODELS_DIR = "./models_rfdetr"
OUTPUT_DIR = "./rtdetr_weather_evaluation"

BATCH_SIZE = 2
NUM_CLASSES = 5

WEATHER_CONDITIONS = ["fog", "glare", "night", "rain", "snow"]
LEVELS = [1, 2, 3, 4, 5]

os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================
# CONVERSION CSV -> COCO FORMAT
# =========================
def convert_csv_to_coco(images_dir, labels_csv_dir, labels_coco_dir):
    print(f"  Conversion CSV -> COCO...")
    os.makedirs(labels_coco_dir, exist_ok=True)
    
    csv_files = glob.glob(os.path.join(labels_csv_dir, "*.csv"))
    
    if len(csv_files) == 0:
        print(f"   Aucun CSV trouvé")
        return 0
    
    converted = 0
    
    # Détecter le format du premier CSV
    first_csv = csv_files[0]
    df_sample = pd.read_csv(first_csv, nrows=5)
    
    has_filename = 'filename' in df_sample.columns or 'image' in df_sample.columns
    
    for csv_file in csv_files:
        try:
            df = pd.read_csv(csv_file)
            
            if not has_filename:
                # Un CSV par image
                base_name = os.path.splitext(os.path.basename(csv_file))[0]
                img_file = None
                for ext in [".jpg", ".jpeg", ".png"]:
                    test_path = os.path.join(images_dir, base_name + ext)
                    if os.path.exists(test_path):
                        img_file = base_name + ext
                        break
                
                if img_file is None:
                    continue
                
                img_path = os.path.join(images_dir, img_file)
                img = Image.open(img_path)
                img_width, img_height = img.size
                
                boxes = []
                labels = []
                
                for _, row in df.iterrows():
                    try:
                        if 'xmin' in df.columns:
                            xmin = float(row['xmin'])
                            ymin = float(row['ymin'])
                            xmax = float(row['xmax'])
                            ymax = float(row['ymax'])
                        elif 'x1' in df.columns:
                            xmin = float(row['x1'])
                            ymin = float(row['y1'])
                            xmax = float(row['x2'])
                            ymax = float(row['y2'])
                        else:
                            continue
                        
                        if 'class' in df.columns:
                            cls = int(row['class'])
                        elif 'class_id' in df.columns:
                            cls = int(row['class_id'])
                        elif 'label' in df.columns:
                            cls = int(row['label'])
                        else:
                            continue
                        
                        if xmax <= xmin or ymax <= ymin:
                            continue
                        
                        if not (0 <= cls < NUM_CLASSES):
                            continue
                        
                        # Convertir en cxcywh normalisé
                        cx = ((xmin + xmax) / 2) / img_width
                        cy = ((ymin + ymax) / 2) / img_height
                        w = (xmax - xmin) / img_width
                        h = (ymax - ymin) / img_height
                        
                        if not (0 <= cx <= 1 and 0 <= cy <= 1 and 0 < w <= 1 and 0 < h <= 1):
                            continue
                        
                        boxes.append([cx, cy, w, h])
                        labels.append(cls)
                    except:
                        continue
                
                json_path = os.path.join(labels_coco_dir, base_name + ".json")
                target = {"boxes": boxes, "class_labels": labels}
                with open(json_path, "w") as f:
                    json.dump(target, f)
                
                converted += 1
            
            else:
                # Format avec colonne filename/image
                filename_col = 'filename' if 'filename' in df.columns else 'image'
                
                for filename in df[filename_col].unique():
                    img_data = df[df[filename_col] == filename]
                    
                    img_path = os.path.join(images_dir, filename)
                    if not os.path.exists(img_path):
                        continue
                    
                    img = Image.open(img_path)
                    img_width, img_height = img.size
                    
                    boxes = []
                    labels = []
                    
                    for _, row in img_data.iterrows():
                        try:
                            if 'xmin' in df.columns:
                                xmin = float(row['xmin'])
                                ymin = float(row['ymin'])
                                xmax = float(row['xmax'])
                                ymax = float(row['ymax'])
                            elif 'x1' in df.columns:
                                xmin = float(row['x1'])
                                ymin = float(row['y1'])
                                xmax = float(row['x2'])
                                ymax = float(row['y2'])
                            else:
                                continue
                            
                            if 'class' in df.columns:
                                cls = int(row['class'])
                            elif 'class_id' in df.columns:
                                cls = int(row['class_id'])
                            elif 'label' in df.columns:
                                cls = int(row['label'])
                            else:
                                continue
                            
                            if xmax <= xmin or ymax <= ymin:
                                continue
                            
                            if not (0 <= cls < NUM_CLASSES):
                                continue
                            
                            cx = ((xmin + xmax) / 2) / img_width
                            cy = ((ymin + ymax) / 2) / img_height
                            w = (xmax - xmin) / img_width
                            h = (ymax - ymin) / img_height
                            
                            if not (0 <= cx <= 1 and 0 <= cy <= 1 and 0 < w <= 1 and 0 < h <= 1):
                                continue
                            
                            boxes.append([cx, cy, w, h])
                            labels.append(cls)
                        except:
                            continue
                    
                    base_name = os.path.splitext(filename)[0]
                    json_path = os.path.join(labels_coco_dir, base_name + ".json")
                    
                    target = {"boxes": boxes, "class_labels": labels}
                    
                    with open(json_path, "w") as f:
                        json.dump(target, f)
                    
                    converted += 1
        
        except Exception as e:
            continue
    
    print(f"   {converted} fichiers convertis")
    return converted

# =========================
# DATASET
# =========================
class RTDetrDataset(Dataset):
    def __init__(self, images_dir, labels_dir):
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        
        all_images = []
        for ext in [".jpg", ".jpeg", ".png"]:
            all_images.extend([f for f in os.listdir(images_dir) if f.lower().endswith(ext)])
        
        self.images = sorted(list(set(all_images)))
        
        # Filtrer les images qui ont des labels
        self.images = [img for img in self.images 
                      if os.path.exists(os.path.join(labels_dir, os.path.splitext(img)[0] + ".json"))]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.images_dir, img_name)
        
        base_name = os.path.splitext(img_name)[0]
        label_path = os.path.join(self.labels_dir, base_name + ".json")

        image = Image.open(img_path).convert("RGB")

        with open(label_path, "r") as f:
            target = json.load(f)

        target = {
            "boxes": torch.tensor(target["boxes"], dtype=torch.float32) if target["boxes"] else torch.zeros((0, 4), dtype=torch.float32),
            "class_labels": torch.tensor(target["class_labels"], dtype=torch.int64) if target["class_labels"] else torch.zeros((0,), dtype=torch.int64)
        }

        return image, target

def collate_fn(batch):
    images, targets = zip(*batch)
    return list(images), list(targets)

# =========================
# CHARGER LE MEILLEUR MODÈLE RT-DETR
# =========================
print("\n Chargement du meilleur modèle RT-DETR...")

history_path = os.path.join(MODELS_DIR, "test_history.csv")
if os.path.exists(history_path):
    df_history = pd.read_csv(history_path)
    best_epoch = df_history.loc[df_history['Box-F1'].idxmax(), 'epoch']
    best_f1 = df_history.loc[df_history['Box-F1'].idxmax(), 'Box-F1']
    print(f" Meilleur epoch: {int(best_epoch)} (F1={best_f1:.4f})")
else:
    model_files = [f for f in os.listdir(MODELS_DIR) if f.startswith("rtdetr_epoch") and f.endswith(".pt")]
    epochs = [int(f.split("epoch")[1].split(".pt")[0]) for f in model_files]
    best_epoch = max(epochs) if epochs else 1
    print(f" Utilisation de l'epoch {int(best_epoch)}")

best_model_path = os.path.join(MODELS_DIR, f"rtdetr_epoch{int(best_epoch)}.pt")

processor = RTDetrImageProcessor.from_pretrained("PekingU/rtdetr_r50vd")
id2label = {i: f"class{i}" for i in range(NUM_CLASSES)}
label2id = {f"class{i}": i for i in range(NUM_CLASSES)}

model = RTDetrForObjectDetection.from_pretrained(
    "PekingU/rtdetr_r50vd",
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
model.to(DEVICE)
model.eval()

print(f" Modèle chargé: {best_model_path}")

# =========================
# HELPER
# =========================
def cxcywh_to_xyxy(boxes, img_w, img_h):
    if boxes.numel() == 0:
        return torch.zeros((0, 4), dtype=torch.float32)
    cx = boxes[:, 0] * img_w
    cy = boxes[:, 1] * img_h
    w = boxes[:, 2] * img_w
    h = boxes[:, 3] * img_h
    x1 = cx - w/2
    y1 = cy - h/2
    x2 = cx + w/2
    y2 = cy + h/2
    return torch.stack([x1, y1, x2, y2], dim=1)

# =========================
# ÉVALUATION
# =========================
def evaluate(model, loader, conf_threshold=0.1):
    model.eval()
    all_gt = []
    all_pred = []
    all_ious = []
    total_det = 0
    total_gt = 0

    with torch.no_grad():
        for images, targets in loader:
            encoding = processor(images=images, return_tensors="pt")
            pixel_values = encoding["pixel_values"].to(DEVICE)

            outputs = model(pixel_values=pixel_values)
            
            target_sizes = torch.tensor([img.size[::-1] for img in images])
            results = processor.post_process_object_detection(
                outputs, target_sizes=target_sizes, threshold=conf_threshold
            )

            for i, res in enumerate(results):
                pred_boxes = res["boxes"].clone() if len(res["boxes"]) > 0 else torch.zeros((0, 4), dtype=torch.float32)
                pred_labels = res["labels"].clone() if len(res["labels"]) > 0 else torch.zeros((0,), dtype=torch.int64)
                
                gt_boxes_norm = targets[i]["boxes"]
                gt_labels = targets[i]["class_labels"]
                
                total_det += len(pred_labels)
                total_gt += len(gt_labels)
                
                if len(pred_boxes) == 0 or len(gt_boxes_norm) == 0:
                    continue
                
                img_w, img_h = images[i].size
                gt_boxes = cxcywh_to_xyxy(gt_boxes_norm, img_w, img_h)
                
                iou = box_iou(gt_boxes, pred_boxes)
                max_iou, best_idx = iou.max(dim=1)
                all_ious.extend(max_iou.cpu().numpy().tolist())
                
                matched = max_iou >= 0.5
                if matched.sum() > 0:
                    all_gt.extend(gt_labels[matched].cpu().numpy().tolist())
                    all_pred.extend(pred_labels[best_idx[matched]].cpu().numpy().tolist())

    if len(all_gt) > 0 and len(all_pred) > 0:
        prec = precision_score(all_gt, all_pred, average="macro", zero_division=0)
        rec = recall_score(all_gt, all_pred, average="macro", zero_division=0)
        f1 = f1_score(all_gt, all_pred, average="macro", zero_division=0)
    else:
        prec, rec, f1 = 0, 0, 0

    map50 = sum(1 for iou in all_ious if iou >= 0.5) / len(all_ious) if all_ious else 0
    map5095 = sum(all_ious) / len(all_ious) if all_ious else 0

    return {
        "Detections": total_det,
        "Ground_Truth": total_gt,
        "Matched": len(all_gt),
        "Precision": round(prec, 5),
        "Recall": round(rec, 5),
        "F1": round(f1, 5),
        "mAP50": round(map50, 5),
        "mAP50-95": round(map5095, 5)
    }

# =========================
# TEST SUR TOUTES LES CONDITIONS
# =========================
all_results = []

print("\n" + "="*70)
print("ÉVALUATION RT-DETR SUR CONDITIONS MÉTÉO")
print("="*70)

for weather in WEATHER_CONDITIONS:
    print(f"\n{'='*70}")
    print(f"  CONDITION: {weather.upper()}")
    print(f"{'='*70}")
    
    for level in LEVELS:
        condition_name = f"{weather}_level{level}"
        images_dir = os.path.join(RESULTS_BASE_DIR, condition_name, "images")
        labels_csv_dir = os.path.join(RESULTS_BASE_DIR, condition_name, "labels_csv")
        labels_coco_dir = os.path.join(RESULTS_BASE_DIR, condition_name, "labels_coco")
        
        if not os.path.exists(images_dir):
            print(f" {condition_name}: Dossier non trouvé")
            continue
        
        print(f"\n {condition_name}:")
        
        # Convertir CSV -> COCO si nécessaire
        if not os.path.exists(labels_coco_dir) or len(os.listdir(labels_coco_dir)) == 0:
            num_converted = convert_csv_to_coco(images_dir, labels_csv_dir, labels_coco_dir)
            if num_converted == 0:
                print(f"   Aucun label, ignoré")
                continue
        else:
            print(f"   Labels déjà convertis")
        
        try:
            dataset = RTDetrDataset(images_dir, labels_coco_dir)
            if len(dataset) == 0:
                print(f"   Dataset vide")
                continue
            
            print(f"   {len(dataset)} images")
            
            loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
            
            print(f"   Évaluation...", end="", flush=True)
            metrics = evaluate(model, loader)
            print(" ")
            
            metrics["Weather"] = weather
            metrics["Level"] = level
            metrics["Condition"] = condition_name
            metrics["Num_Images"] = len(dataset)
            
            all_results.append(metrics)
            
            print(f"   Det: {metrics['Detections']}, GT: {metrics['Ground_Truth']}, Match: {metrics['Matched']}")
            print(f"   P: {metrics['Precision']:.4f}, R: {metrics['Recall']:.4f}, F1: {metrics['F1']:.4f}, mAP50: {metrics['mAP50']:.4f}")
            
        except Exception as e:
            print(f"   Erreur: {e}")
            continue

# =========================
# SAUVEGARDER LES RÉSULTATS
# =========================
print("\n" + "="*70)
print(" SAUVEGARDE DES RÉSULTATS")
print("="*70)

if len(all_results) > 0:
    df_results = pd.DataFrame(all_results)
    
    cols = ["Weather", "Level", "Condition", "Num_Images", "Detections", "Ground_Truth", "Matched",
            "Precision", "Recall", "F1", "mAP50", "mAP50-95"]
    df_results = df_results[cols]
    
    results_path = os.path.join(OUTPUT_DIR, "rtdetr_weather_results.csv")
    df_results.to_csv(results_path, index=False)
    print(f" Résultats: {results_path}")
    
    summary = df_results.groupby("Weather")[["Precision", "Recall", "F1", "mAP50", "mAP50-95"]].mean()
    summary_path = os.path.join(OUTPUT_DIR, "rtdetr_weather_summary.csv")
    summary.to_csv(summary_path)
    print(f" Résumé: {summary_path}")
    
    print("\n RÉSULTATS COMPLETS:")
    print(df_results.to_string(index=False))
    
    print("\n RÉSUMÉ PAR MÉTÉO:")
    print(summary.round(4))
    
else:
    print(" Aucun résultat")

print("\n" + "="*70)
print(" ÉVALUATION RT-DETR TERMINÉE!")
print(f" Résultats: {OUTPUT_DIR}")
print("="*70)

DEVICE: cpu

 Chargement du meilleur modèle RT-DETR...
 Meilleur epoch: 9 (F1=0.9921)


Some weights of RTDetrForObjectDetection were not initialized from the model checkpoint at PekingU/rtdetr_r50vd and are newly initialized because the shapes did not match:
- model.decoder.class_embed.0.bias: found shape torch.Size([80]) in the checkpoint and torch.Size([5]) in the model instantiated
- model.decoder.class_embed.0.weight: found shape torch.Size([80, 256]) in the checkpoint and torch.Size([5, 256]) in the model instantiated
- model.decoder.class_embed.1.bias: found shape torch.Size([80]) in the checkpoint and torch.Size([5]) in the model instantiated
- model.decoder.class_embed.1.weight: found shape torch.Size([80, 256]) in the checkpoint and torch.Size([5, 256]) in the model instantiated
- model.decoder.class_embed.2.bias: found shape torch.Size([80]) in the checkpoint and torch.Size([5]) in the model instantiated
- model.decoder.class_embed.2.weight: found shape torch.Size([80, 256]) in the checkpoint and torch.Size([5, 256]) in the model instantiated
- model.decoder.cl

 Modèle chargé: ./models_rfdetr\rtdetr_epoch9.pt

ÉVALUATION RT-DETR SUR CONDITIONS MÉTÉO

  CONDITION: FOG

 fog_level1:
  Conversion CSV -> COCO...
   500 fichiers convertis
   500 images
   Évaluation... 
   Det: 833, GT: 1559, Match: 824
   P: 0.9985, R: 0.9951, F1: 0.9968, mAP50: 0.5285

 fog_level2:
  Conversion CSV -> COCO...
   500 fichiers convertis
   500 images
   Évaluation... 
   Det: 723, GT: 1559, Match: 714
   P: 0.9980, R: 0.9877, F1: 0.9927, mAP50: 0.4580

 fog_level3:
  Conversion CSV -> COCO...
   500 fichiers convertis
   500 images
   Évaluation... 
   Det: 613, GT: 1559, Match: 597
   P: 0.9946, R: 0.9141, F1: 0.9454, mAP50: 0.3837

 fog_level4:
  Conversion CSV -> COCO...
   500 fichiers convertis
   500 images
   Évaluation... 
   Det: 689, GT: 1559, Match: 561
   P: 0.9528, R: 0.8545, F1: 0.8840, mAP50: 0.3691

 fog_level5:
  Conversion CSV -> COCO...
   500 fichiers convertis
   500 images
   Évaluation... 
   Det: 1225, GT: 1559, Match: 537
   P: 0.9622, R: 

# RT-DETR tableaux weather 

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =========================
# DONNÉES RT-DETR PAR NIVEAU
# =========================

rtdetr_results = {
    'fog': {
        1: {'P': 0.99852, 'R': 0.99513, 'F1': 0.99680, 'mAP50': 0.52854, 'mAP50-95': 0.51711},
        2: {'P': 0.99804, 'R': 0.98765, 'F1': 0.99268, 'mAP50': 0.45799, 'mAP50-95': 0.45824},
        3: {'P': 0.99464, 'R': 0.91406, 'F1': 0.94540, 'mAP50': 0.38368, 'mAP50-95': 0.39634},
        4: {'P': 0.95284, 'R': 0.85451, 'F1': 0.88402, 'mAP50': 0.36908, 'mAP50-95': 0.37769},
        5: {'P': 0.96221, 'R': 0.86820, 'F1': 0.90579, 'mAP50': 0.36983, 'mAP50-95': 0.37608}
    },
    'glare': {
        1: {'P': 0.99882, 'R': 0.99722, 'F1': 0.99801, 'mAP50': 0.58307, 'mAP50-95': 0.56115},
        2: {'P': 0.99882, 'R': 0.99714, 'F1': 0.99797, 'mAP50': 0.57729, 'mAP50-95': 0.55681},
        3: {'P': 0.99852, 'R': 0.99634, 'F1': 0.99742, 'mAP50': 0.57409, 'mAP50-95': 0.55303},
        4: {'P': 0.99882, 'R': 0.99691, 'F1': 0.99785, 'mAP50': 0.55997, 'mAP50-95': 0.54211},
        5: {'P': 0.99882, 'R': 0.99669, 'F1': 0.99774, 'mAP50': 0.55035, 'mAP50-95': 0.53443}
    },
    'night': {
        1: {'P': 0.99882, 'R': 0.99603, 'F1': 0.99740, 'mAP50': 0.49583, 'mAP50-95': 0.49106},
        2: {'P': 0.99882, 'R': 0.99549, 'F1': 0.99713, 'mAP50': 0.47081, 'mAP50-95': 0.47011},
        3: {'P': 0.99881, 'R': 0.99474, 'F1': 0.99674, 'mAP50': 0.44901, 'mAP50-95': 0.45113},
        4: {'P': 0.99612, 'R': 0.99071, 'F1': 0.99336, 'mAP50': 0.42224, 'mAP50-95': 0.42712},
        5: {'P': 0.99502, 'R': 0.98551, 'F1': 0.99010, 'mAP50': 0.39806, 'mAP50-95': 0.40596}
    },
    'rain': {
        1: {'P': 0.99882, 'R': 0.99483, 'F1': 0.99679, 'mAP50': 0.44901, 'mAP50-95': 0.44581},
        2: {'P': 0.99842, 'R': 0.98750, 'F1': 0.99276, 'mAP50': 0.39384, 'mAP50-95': 0.40212},
        3: {'P': 0.99643, 'R': 0.89706, 'F1': 0.93339, 'mAP50': 0.34326, 'mAP50-95': 0.36069},
        4: {'P': 0.99609, 'R': 0.78125, 'F1': 0.80359, 'mAP50': 0.32688, 'mAP50-95': 0.34019},
        5: {'P': 0.58053, 'R': 0.55000, 'F1': 0.56142, 'mAP50': 0.31903, 'mAP50-95': 0.32542}
    },
    'snow': {
        1: {'P': 0.99554, 'R': 0.99161, 'F1': 0.99351, 'mAP50': 0.47723, 'mAP50-95': 0.47407},
        2: {'P': 0.99804, 'R': 0.97222, 'F1': 0.98409, 'mAP50': 0.39064, 'mAP50-95': 0.40165},
        3: {'P': 0.99545, 'R': 0.82692, 'F1': 0.86535, 'mAP50': 0.34948, 'mAP50-95': 0.36126},
        4: {'P': 0.72009, 'R': 0.62500, 'F1': 0.65117, 'mAP50': 0.31815, 'mAP50-95': 0.32040},
        5: {'P': 0.73992, 'R': 0.75000, 'F1': 0.74486, 'mAP50': 0.24004, 'mAP50-95': 0.24963}
    }
}

# =========================
# FONCTION POUR DÉTERMINER LA QUALITÉ
# =========================
def get_quality(value):
    """Détermine la qualité basée sur la valeur mAP50"""
    if value >= 0.90:
        return "Excellent"
    elif value >= 0.75:
        return "Très Bon"
    elif value >= 0.60:
        return "Bon"
    elif value >= 0.40:
        return "Moyen"
    else:
        return "Faible"

def format_with_delta(current_value, reference_value):
    """Formate une valeur avec son delta par rapport à une référence (level 1)"""
    delta = current_value - reference_value
    if abs(delta) < 0.001:
        return f"{current_value:.3f}"
    elif delta > 0:
        return f"{current_value:.3f}(+{delta:.3f})"
    else:
        return f"{current_value:.3f}({delta:.3f})"

# =========================
# TABLEAUX PAR CONDITION MÉTÉO
# =========================
save_dir = "./rtdetr_weather_evaluation"

for weather in ['fog', 'glare', 'night', 'rain', 'snow']:
    print("\n" + "="*100)
    print(f" RT-DETR - RÉSULTATS {weather.upper()}")
    print("="*100)
    
    # Référence (level 1)
    ref_metrics = rtdetr_results[weather][1]
    
    # Tableau avec deltas
    weather_data = []
    for level in range(1, 6):
        metrics = rtdetr_results[weather][level]
        
        weather_data.append({
            'Level': f'{weather}_level{level}',
            'P (Test)': format_with_delta(metrics['P'], ref_metrics['P']),
            'R (Test)': format_with_delta(metrics['R'], ref_metrics['R']),
            'F1 (Test)': format_with_delta(metrics['F1'], ref_metrics['F1']),
            'mAP50 (Test)': format_with_delta(metrics['mAP50'], ref_metrics['mAP50']),
            'mAP50-95 (Test)': format_with_delta(metrics['mAP50-95'], ref_metrics['mAP50-95']),
            'Qualité': get_quality(metrics['mAP50'])
        })
    
    df_weather = pd.DataFrame(weather_data)
    print(df_weather.to_string(index=False))
    
    # Sauvegarder
    df_weather.to_csv(f"{save_dir}/rtdetr_{weather}_results.csv", index=False)

# =========================
# TABLEAU COMPARATIF TOUTES CONDITIONS (LEVEL 1)
# =========================
print("\n" + "="*100)
print(" RT-DETR - COMPARAISON TOUTES CONDITIONS (LEVEL 1)")
print("="*100)

level1_data = []
for weather in ['fog', 'glare', 'night', 'rain', 'snow']:
    metrics = rtdetr_results[weather][1]
    level1_data.append({
        'Condition': weather.upper(),
        'Precision': f"{metrics['P']:.3f}",
        'Recall': f"{metrics['R']:.3f}",
        'F1-Score': f"{metrics['F1']:.3f}",
        'mAP50': f"{metrics['mAP50']:.3f}",
        'mAP50-95': f"{metrics['mAP50-95']:.3f}",
        'Qualité': get_quality(metrics['mAP50'])
    })

df_level1 = pd.DataFrame(level1_data)
print(df_level1.to_string(index=False))
df_level1.to_csv(f"{save_dir}/rtdetr_level1_comparison.csv", index=False)

# =========================
# TABLEAU COMPARATIF TOUTES CONDITIONS (LEVEL 5)
# =========================
print("\n" + "="*100)
print(" RT-DETR - COMPARAISON TOUTES CONDITIONS (LEVEL 5)")
print("="*100)

level5_data = []
for weather in ['fog', 'glare', 'night', 'rain', 'snow']:
    metrics = rtdetr_results[weather][5]
    level5_data.append({
        'Condition': weather.upper(),
        'Precision': f"{metrics['P']:.3f}",
        'Recall': f"{metrics['R']:.3f}",
        'F1-Score': f"{metrics['F1']:.3f}",
        'mAP50': f"{metrics['mAP50']:.3f}",
        'mAP50-95': f"{metrics['mAP50-95']:.3f}",
        'Qualité': get_quality(metrics['mAP50'])
    })

df_level5 = pd.DataFrame(level5_data)
print(df_level5.to_string(index=False))
df_level5.to_csv(f"{save_dir}/rtdetr_level5_comparison.csv", index=False)

# =========================
# TABLEAU RÉSUMÉ PAR CONDITION MÉTÉO
# =========================
print("\n" + "="*100)
print(" RT-DETR - RÉSUMÉ MOYEN PAR CONDITION MÉTÉO")
print("="*100)

summary_data = []
for weather in ['fog', 'glare', 'night', 'rain', 'snow']:
    # Calculer les moyennes sur les 5 niveaux
    avg_p = np.mean([rtdetr_results[weather][l]['P'] for l in range(1, 6)])
    avg_r = np.mean([rtdetr_results[weather][l]['R'] for l in range(1, 6)])
    avg_f1 = np.mean([rtdetr_results[weather][l]['F1'] for l in range(1, 6)])
    avg_map50 = np.mean([rtdetr_results[weather][l]['mAP50'] for l in range(1, 6)])
    avg_map5095 = np.mean([rtdetr_results[weather][l]['mAP50-95'] for l in range(1, 6)])
    
    summary_data.append({
        'Condition': weather.upper(),
        'Avg Precision': f"{avg_p:.3f}",
        'Avg Recall': f"{avg_r:.3f}",
        'Avg F1': f"{avg_f1:.3f}",
        'Avg mAP50': f"{avg_map50:.3f}",
        'Avg mAP50-95': f"{avg_map5095:.3f}",
        'Qualité': get_quality(avg_map50)
    })

df_summary = pd.DataFrame(summary_data)
print(df_summary.to_string(index=False))
df_summary.to_csv(f"{save_dir}/rtdetr_weather_summary.csv", index=False)

# =========================
# TABLEAU DE DÉGRADATION (LEVEL 1 vs LEVEL 5)
# =========================
print("\n" + "="*100)
print(" RT-DETR - DÉGRADATION (LEVEL 1 → LEVEL 5)")
print("="*100)

degradation_data = []
for weather in ['fog', 'glare', 'night', 'rain', 'snow']:
    level1 = rtdetr_results[weather][1]
    level5 = rtdetr_results[weather][5]
    
    degradation_data.append({
        'Condition': weather.upper(),
        'ΔP': f"{(level5['P'] - level1['P']):.3f}",
        'ΔR': f"{(level5['R'] - level1['R']):.3f}",
        'ΔF1': f"{(level5['F1'] - level1['F1']):.3f}",
        'ΔmAP50': f"{(level5['mAP50'] - level1['mAP50']):.3f}",
        'ΔmAP50-95': f"{(level5['mAP50-95'] - level1['mAP50-95']):.3f}",
        'Loss %': f"{((level1['mAP50'] - level5['mAP50']) / level1['mAP50'] * 100):.1f}%"
    })

df_degradation = pd.DataFrame(degradation_data)
print(df_degradation.to_string(index=False))
df_degradation.to_csv(f"{save_dir}/rtdetr_degradation_analysis.csv", index=False)

# =========================
# GRAPHIQUES
# =========================
print("\n Génération des graphiques...")

# 1. Graphique par condition météo
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('RT-DETR - Performance par Condition Météo', fontsize=18, fontweight='bold')

colors = {
    'fog': '#1f77b4',
    'glare': '#ff7f0e',
    'night': '#2ca02c',
    'rain': '#d62728',
    'snow': '#9467bd'
}

levels = [1, 2, 3, 4, 5]
metrics_to_plot = [
    ('P', 'Precision', axes[0, 0]),
    ('R', 'Recall', axes[0, 1]),
    ('F1', 'F1-Score', axes[0, 2]),
    ('mAP50', 'mAP@50', axes[1, 0]),
    ('mAP50-95', 'mAP@50-95', axes[1, 1])
]

for metric_key, metric_name, ax in metrics_to_plot:
    for weather in ['fog', 'glare', 'night', 'rain', 'snow']:
        values = [rtdetr_results[weather][l][metric_key] for l in levels]
        ax.plot(levels, values, color=colors[weather], marker='o', 
                linewidth=2.5, markersize=10, label=weather.upper())
    
    ax.set_xlabel('Niveau de Perturbation', fontsize=12, fontweight='bold')
    ax.set_ylabel(metric_name, fontsize=12, fontweight='bold')
    ax.set_title(metric_name, fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.legend(loc='best', fontsize=10)
    ax.set_xticks(levels)
    ax.set_ylim([0, 1.05])

# Graphique de dégradation
ax_deg = axes[1, 2]
weather_names = [w.upper() for w in ['fog', 'glare', 'night', 'rain', 'snow']]
degradation_values = [
    (rtdetr_results[w][1]['mAP50'] - rtdetr_results[w][5]['mAP50']) / rtdetr_results[w][1]['mAP50'] * 100
    for w in ['fog', 'glare', 'night', 'rain', 'snow']
]

bars = ax_deg.bar(weather_names, degradation_values, 
                  color=[colors[w] for w in ['fog', 'glare', 'night', 'rain', 'snow']], 
                  alpha=0.7, edgecolor='black', linewidth=1.5)

ax_deg.set_ylabel('Perte mAP50 (%)', fontsize=12, fontweight='bold')
ax_deg.set_title('Dégradation Level 1→5', fontsize=13, fontweight='bold')
ax_deg.grid(axis='y', alpha=0.3, linestyle='--')

for bar in bars:
    height = bar.get_height()
    ax_deg.text(bar.get_x() + bar.get_width()/2., height,
               f'{height:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(f"{save_dir}/rtdetr_weather_performance.png", dpi=300, bbox_inches='tight')
print(f" Graphique sauvegardé : rtdetr_weather_performance.png")

# 2. Heatmap des performances
fig, ax = plt.subplots(figsize=(12, 8))
fig.suptitle('RT-DETR - Heatmap mAP@50 par Condition et Niveau', fontsize=16, fontweight='bold')

weather_list = ['fog', 'glare', 'night', 'rain', 'snow']
matrix = np.zeros((len(weather_list), 5))

for i, weather in enumerate(weather_list):
    for j, level in enumerate(range(1, 6)):
        matrix[i, j] = rtdetr_results[weather][level]['mAP50']

im = ax.imshow(matrix, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)

ax.set_xticks(range(5))
ax.set_yticks(range(len(weather_list)))
ax.set_xticklabels([f'Level {l}' for l in range(1, 6)], fontsize=12)
ax.set_yticklabels([w.upper() for w in weather_list], fontsize=12)
ax.set_xlabel('Niveau de Perturbation', fontsize=13, fontweight='bold')
ax.set_ylabel('Condition Météo', fontsize=13, fontweight='bold')

# Ajouter les valeurs
for i in range(len(weather_list)):
    for j in range(5):
        text = ax.text(j, i, f'{matrix[i, j]:.3f}',
                      ha="center", va="center", 
                      color="black", fontsize=11, fontweight='bold')

plt.colorbar(im, ax=ax, label='mAP@50')
plt.tight_layout()
plt.savefig(f"{save_dir}/rtdetr_heatmap.png", dpi=300, bbox_inches='tight')
print(f" Heatmap sauvegardé : rtdetr_heatmap.png")

plt.show()

print("\n" + "="*100)
print(" TABLEAUX ET GRAPHIQUES RT-DETR GÉNÉRÉS")
print(f" Dossier: {save_dir}")
print("="*100)


 RT-DETR - RÉSULTATS FOG
     Level      P (Test)      R (Test)     F1 (Test)  mAP50 (Test) mAP50-95 (Test) Qualité
fog_level1         0.999         0.995         0.997         0.529           0.517   Moyen
fog_level2         0.998 0.988(-0.007) 0.993(-0.004) 0.458(-0.071)   0.458(-0.059)   Moyen
fog_level3 0.995(-0.004) 0.914(-0.081) 0.945(-0.051) 0.384(-0.145)   0.396(-0.121)  Faible
fog_level4 0.953(-0.046) 0.855(-0.141) 0.884(-0.113) 0.369(-0.159)   0.378(-0.139)  Faible
fog_level5 0.962(-0.036) 0.868(-0.127) 0.906(-0.091) 0.370(-0.159)   0.376(-0.141)  Faible

 RT-DETR - RÉSULTATS GLARE
       Level P (Test) R (Test) F1 (Test)  mAP50 (Test) mAP50-95 (Test) Qualité
glare_level1    0.999    0.997     0.998         0.583           0.561   Moyen
glare_level2    0.999    0.997     0.998 0.577(-0.006)   0.557(-0.004)   Moyen
glare_level3    0.999    0.996     0.997 0.574(-0.009)   0.553(-0.008)   Moyen
glare_level4    0.999    0.997     0.998 0.560(-0.023)   0.542(-0.019)   Moyen
glare

<Figure size 1800x1000 with 6 Axes>

<Figure size 1200x800 with 2 Axes>


 TABLEAUX ET GRAPHIQUES RT-DETR GÉNÉRÉS
 Dossier: ./rtdetr_weather_evaluation


In [1]:
%matplotlib inline

import os
from pathlib import Path
import matplotlib.pyplot as plt
import cv2  # ou PIL
import random

images_dir = Path("./Data/val/images")
labels_dir = Path("./Data/val/labels")

# 1. Vérifier correspondance image <-> label
image_files = list(images_dir.glob("*.jpg"))
print("Nombre d’images :", len(image_files))

missing_labels = []
for img_path in image_files:
    label_path = labels_dir / (img_path.stem + ".txt")
    if not label_path.exists():
        missing_labels.append(img_path.name)

if missing_labels:
    print(" Quelques images sans label:", missing_labels[:10])
else:
    print(" Tous les fichiers ont un label correspondant.")

# 2. Vérifier classes utilisées
class_ids = set()
for label_file in labels_dir.glob("*.txt"):
    with open(label_file, "r") as f:
        for line in f:
            parts = line.strip().split()
            if parts:
                class_ids.add(parts[0])

print("Classes trouvées dans les labels : ", sorted(class_ids))

# 3. Afficher aléatoirement 3 images + leurs boîtes pour vérification visuelle
for _ in range(3):
    img_path = random.choice(image_files)
    label_path = labels_dir / (img_path.stem + ".txt")

    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    plt.figure(figsize=(6,6))
    plt.imshow(img)
    plt.axis("off")

    with open(label_path, "r") as f:
        for line in f:
            cls, cx, cy, bw, bh = line.strip().split()
            cls = int(cls); cx = float(cx); cy = float(cy); bw = float(bw); bh = float(bh)

            # Convert to pixel coords
            x1 = int((cx - bw/2) * w)
            y1 = int((cy - bh/2) * h)
            x2 = int((cx + bw/2) * w)
            y2 = int((cy + bh/2) * h)

            plt.gca().add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1,
                                              fill=False, edgecolor='red', linewidth=2))
            plt.text(x1, y1-5, f"cls {cls}", color='red', fontsize=12)

    plt.show()


Nombre d’images : 100
 Tous les fichiers ont un label correspondant.
Classes trouvées dans les labels :  ['0', '1', '2', '3', '4']


<Figure size 600x600 with 1 Axes>

<Figure size 600x600 with 1 Axes>

<Figure size 600x600 with 1 Axes>

In [4]:
import torch
import torch.nn as nn
import torchvision
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.ops import MultiScaleRoIAlign
import numpy as np
from PIL import Image
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from typing import List, Dict, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

class FasterRCNNPipeline:
    """
    Pipeline complet pour Faster R-CNN avec contrôle étape par étape
    """
    
    def __init__(self, num_classes: int, backbone: str = "resnet50", min_size: int = 800, 
                 max_size: int = 1333, box_score_thresh: float = 0.05):
        
        self.num_classes = num_classes
        self.min_size = min_size
        self.max_size = max_size
        self.box_score_thresh = box_score_thresh
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        # 1⃣ Initialisation du backbone avec FPN
        self.backbone_name = backbone
        self.backbone = self._create_backbone_with_fpn(backbone)
        
        # 2⃣ Configuration du RPN (Region Proposal Network)
        self.anchor_generator = self._create_anchor_generator()
        
        # 3⃣ Configuration du RoI Pooling
        self.roi_pooler = self._create_roi_pooler()
        
        # 4⃣ Configuration du classifieur de boîtes
        self.box_head, self.box_predictor = self._create_box_head()
        
        # 5⃣ Assemblage du modèle Faster R-CNN complet
        self.model = self._assemble_faster_rcnn()
        
        print(f" Pipeline Faster R-CNN initialisé avec {backbone}")
        print(f" Nombre de classes: {num_classes}")
        print(f"  Device: {self.device}")
    
    def _create_backbone_with_fpn(self, backbone_name: str) -> nn.Module:
        """Étape 1: Création du backbone avec FPN"""
        
        return torchvision.models.detection.backbone_utils.resnet_fpn_backbone(
            backbone_name=backbone_name,
            weights="DEFAULT",
            trainable_layers=3  # Les 3 dernières couches entraînables
        )
    
    def _create_anchor_generator(self) -> AnchorGenerator:
        """Configuration des ancres pour le RPN"""
        
        # Tailles des ancres (côté du carré) pour chaque niveau de la FPN
        anchor_sizes = ((32,), (64,), (128,), (256,), (512,))
        
        # Ratios d'aspect des ancres
        aspect_ratios = ((0.5, 1.0, 2.0),) * len(anchor_sizes)
        
        return AnchorGenerator(
            sizes=anchor_sizes,
            aspect_ratios=aspect_ratios
        )
    
    def _create_roi_pooler(self) -> MultiScaleRoIAlign:
        """Configuration du RoI Pooling"""
        
        return MultiScaleRoIAlign(
            featmap_names=['0', '1', '2', '3'],  # Noms des feature maps de la FPN
            output_size=7,  # Taille de sortie fixe
            sampling_ratio=2
        )
    
    def _create_box_head(self) -> Tuple[nn.Module, nn.Module]:
        """Création du classifieur de boîtes"""
        
        # Réseau pour extraire les features des RoIs
        box_head = torchvision.models.detection.faster_rcnn.TwoMLPHead(
            in_channels=256,  # Sortie de la FPN
            representation_size=1024
        )
        
        # Prédicteur final (classes + bounding boxes)
        box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(
            in_channels=1024,  # representation_size
            num_classes=self.num_classes
        )
        
        return box_head, box_predictor
    
    def _assemble_faster_rcnn(self) -> FasterRCNN:
        """Assemblage final du modèle Faster R-CNN"""
        
        # Créer le modèle sans spécifier num_classes puisque nous fournissons box_predictor
        model = FasterRCNN(
            backbone=self.backbone,
            num_classes=None,  # Doit être None quand box_predictor est spécifié
            
            # Configuration RPN
            rpn_anchor_generator=self.anchor_generator,
            rpn_pre_nms_top_n_train=2000,
            rpn_pre_nms_top_n_test=1000,
            rpn_post_nms_top_n_train=2000,
            rpn_post_nms_top_n_test=1000,
            rpn_nms_thresh=0.7,
            rpn_fg_iou_thresh=0.7,
            rpn_bg_iou_thresh=0.3,
            rpn_batch_size_per_image=256,
            rpn_positive_fraction=0.5,
            
            # Configuration RoI
            box_roi_pool=self.roi_pooler,
            box_head=self.box_head,
            box_predictor=self.box_predictor,  # Nous fournissons notre propre box_predictor
            box_score_thresh=self.box_score_thresh,
            box_nms_thresh=0.5,
            box_detections_per_img=100,
            box_fg_iou_thresh=0.5,
            box_bg_iou_thresh=0.5,
            box_batch_size_per_image=512,
            box_positive_fraction=0.25,
            
            # Taille d'image
            min_size=self.min_size,
            max_size=self.max_size
        )
        
        return model.to(self.device)
    
    def preprocess_image(self, image: np.ndarray) -> torch.Tensor:
        """Prétraitement de l'image pour l'inférence"""
        
        # Conversion en PIL si nécessaire
        if isinstance(image, np.ndarray):
            if len(image.shape) == 3 and image.shape[2] == 3:
                image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            image = Image.fromarray(image)
        
        # Transformation
        transform = torchvision.transforms.Compose([
            torchvision.transforms.ToTensor(),
        ])
        
        return transform(image).unsqueeze(0).to(self.device)
    
    def visualize_feature_maps(self, image_tensor: torch.Tensor, num_maps: int = 16):
        """Visualisation des feature maps après le CNN"""
        
        self.model.eval()
        
        # Forward pass jusqu'au backbone
        with torch.no_grad():
            # Extraire les features avec FPN
            features = self.model.backbone(image_tensor)
            
            # Prendre la première feature map (plus haute résolution)
            if '0' in features:
                first_feature_map = features['0'][0].cpu().numpy()
            else:
                # Si la structure est différente, prendre le premier élément
                first_key = list(features.keys())[0]
                first_feature_map = features[first_key][0].cpu().numpy()
            
        # Visualisation
        fig, axes = plt.subplots(4, 4, figsize=(12, 12))
        fig.suptitle(' Feature Maps après le Backbone FPN', fontsize=16, fontweight='bold')
        
        for i, ax in enumerate(axes.flat):
            if i < min(num_maps, first_feature_map.shape[0]):
                feature_map = first_feature_map[i]
                ax.imshow(feature_map, cmap='viridis')
                ax.set_title(f'Channel {i}')
                ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        return features
    
    def detect_objects(self, image: np.ndarray, confidence_threshold: float = 0.5) -> Dict:
        """
        Détection d'objets complète avec toutes les étapes
        """
        
        # 1⃣ Prétraitement
        image_tensor = self.preprocess_image(image)
        original_image = image.copy()
        
        # 2⃣ Mode évaluation
        self.model.eval()
        
        with torch.no_grad():
            # 3⃣ Forward pass complet
            predictions = self.model(image_tensor)
            
            # 4⃣ Extraction des résultats
            boxes = predictions[0]['boxes'].cpu().numpy()
            scores = predictions[0]['scores'].cpu().numpy()
            labels = predictions[0]['labels'].cpu().numpy()
            
            # 5⃣ Filtrage par confidence
            keep = scores >= confidence_threshold
            filtered_boxes = boxes[keep]
            filtered_scores = scores[keep]
            filtered_labels = labels[keep]
        
        results = {
            'boxes': filtered_boxes,
            'scores': filtered_scores,
            'labels': filtered_labels,
            'num_detections': len(filtered_boxes),
            'original_image': original_image
        }
        
        return results
    
    def visualize_detection_step_by_step(self, image: np.ndarray):
        """
        Visualisation étape par étape du pipeline
        """
        
        print(" LANCEMENT DU PIPELINE ÉTAPE PAR ÉTAPE")
        print("=" * 60)
        
        # 1⃣ Feature Extraction
        print("1⃣ ÉTAPE 1: EXTRACTION DES FEATURES (CNN + FPN)")
        image_tensor = self.preprocess_image(image)
        feature_maps = self.visualize_feature_maps(image_tensor)
        print(" Features extraites avec succès")
        
        # 2⃣ Détection finale
        print("\n2⃣ ÉTAPE 2: DÉTECTION FINALE (RPN + Classification)")
        results = self.detect_objects(image)
        
        # 3⃣ Visualisation des résultats
        print("\n3⃣ ÉTAPE 3: POST-PROCESSING ET VISUALISATION")
        self.plot_detection_results(results)
        
        return results
    
    def plot_detection_results(self, results: Dict, class_names: Optional[List[str]] = None):
        """Visualisation des résultats de détection"""
        
        image = results['original_image']
        boxes = results['boxes']
        scores = results['scores']
        labels = results['labels']
        
        # Conversion RGB si nécessaire
        if len(image.shape) == 3 and image.shape[2] == 3:
            image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        else:
            image_rgb = image
        
        fig, ax = plt.subplots(1, 1, figsize=(12, 8))
        
        # Afficher l'image
        ax.imshow(image_rgb)
        
        # Couleurs pour différentes classes
        colors = ['red', 'blue', 'green', 'orange', 'purple', 'yellow', 'cyan', 'magenta']
        
        # Dessiner les bounding boxes
        for i, (box, score, label) in enumerate(zip(boxes, scores, labels)):
            x1, y1, x2, y2 = box
            
            # Sélection de la couleur
            color = colors[label % len(colors)]
            
            # Rectangle
            rect = patches.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                linewidth=2, edgecolor=color, facecolor='none'
            )
            ax.add_patch(rect)
            
            # Label
            class_name = class_names[label] if class_names else f"Class {label}"
            label_text = f"{class_name}: {score:.3f}"
            
            ax.text(
                x1, y1 - 10, label_text,
                bbox=dict(boxstyle="round,pad=0.3", facecolor=color, alpha=0.7),
                fontsize=8, color='white', fontweight='bold'
            )
        
        ax.set_title(f' Résultats de Détection - {len(boxes)} objets détectés', 
                    fontsize=16, fontweight='bold')
        ax.axis('off')
        plt.tight_layout()
        plt.show()
        
        # Affichage des statistiques
        print(f"\n STATISTIQUES DE DÉTECTION:")
        print(f"   • Nombre d'objets détectés: {len(boxes)}")
        if len(boxes) > 0:
            print(f"   • Score moyen: {np.mean(scores):.3f}")
            print(f"   • Score max: {np.max(scores):.3f}")
            print(f"   • Score min: {np.min(scores):.3f}")
            
            unique_labels, counts = np.unique(labels, return_counts=True)
            print(f"   • Répartition des classes:")
            for label, count in zip(unique_labels, counts):
                class_name = class_names[label] if class_names else f"Class {label}"
                print(f"     - {class_name}: {count} objets")
        else:
            print("   • Aucun objet détecté au-dessus du seuil de confiance")

    def explain_pipeline_steps(self):
        """Explication détaillée des étapes du pipeline"""
        
        print("\n" + "="*70)
        print(" EXPLICATION DÉTAILLÉE DU PIPELINE FASTER R-CNN")
        print("="*70)
        
        steps = [
            {
                "step": "1⃣ EXTRACTION DES FEATURES",
                "description": "Le backbone CNN (ResNet50 avec FPN) extrait des feature maps multi-échelles",
                "output": "Feature maps à 4 échelles différentes ['0', '1', '2', '3']",
                "channels": "256 canaux par feature map"
            },
            {
                "step": "2⃣ RPN (REGION PROPOSAL NETWORK)",
                "description": "Génère ~2000 propositions de régions contenant potentiellement des objets",
                "output": "Boîtes candidates + scores d'objectivité",
                "anchors": "5 tailles × 3 ratios = 15 ancres par position"
            },
            {
                "step": "3⃣ ROI POOLING/ALIGN",
                "description": "Redimensionne chaque région d'intérêt à une taille fixe (7×7)",
                "output": "Features normalisées pour la classification",
                "purpose": "Permet de traiter des régions de tailles variables"
            },
            {
                "step": "4⃣ CLASSIFICATION ET RÉGRESSION",
                "description": "Classe les objets et affine les coordonnées des bounding boxes",
                "output": "Classes prédites + boxes raffinées + scores de confiance",
                "architecture": "TwoMLPHead + FastRCNNPredictor"
            },
            {
                "step": "5⃣ POST-PROCESSING",
                "description": "NMS supprime les doublons, filtrage par score de confiance",
                "output": "Boîtes finales détectées",
                "algo": "Non-Maximum Suppression (NMS)"
            }
        ]
        
        for step_info in steps:
            print(f"\n{step_info['step']}")
            print(f"    {step_info['description']}")
            print(f"    Sortie: {step_info['output']}")
            if 'channels' in step_info:
                print(f"    {step_info['channels']}")
            if 'anchors' in step_info:
                print(f"    {step_info['anchors']}")
            if 'purpose' in step_info:
                print(f"    {step_info['purpose']}")
            if 'architecture' in step_info:
                print(f"     {step_info['architecture']}")
            if 'algo' in step_info:
                print(f"     {step_info['algo']}")

    def get_model_summary(self):
        """Affiche un résumé de l'architecture du modèle"""
        
        print("\n" + "="*70)
        print("  RÉSUMÉ DE L'ARCHITECTURE FASTER R-CNN")
        print("="*70)
        
        total_params = sum(p.numel() for p in self.model.parameters())
        trainable_params = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        
        print(f" Paramètres totaux: {total_params:,}")
        print(f" Paramètres entraînables: {trainable_params:,}")
        print(f" Backbone: {self.backbone_name} avec FPN")
        print(f" Nombre de classes: {self.num_classes}")
        print(f" Ancres par position: {self.anchor_generator.num_anchors_per_location()[0]}")
        
        # Compter les sous-modules
        modules = {
            'Backbone': len(list(self.model.backbone.modules())),
            'RPN': len(list(self.model.rpn.modules())),
            'RoI Heads': len(list(self.model.roi_heads.modules()))
        }
        
        print(f" Sous-modules: {modules}")

# =========================
# EXEMPLE D'UTILISATION SIMPLIFIÉ
# =========================

def demo_pipeline_simple():
    """Démonstration simplifiée du pipeline"""
    
    # Configuration
    NUM_CLASSES = 5  # Exemple: 4 classes + background
    CLASS_NAMES = ["Personne", "Véhicule", "Animal", "Objet", "Autre"]
    
    try:
        # 1⃣ Initialisation du pipeline
        print(" Initialisation du pipeline Faster R-CNN...")
        pipeline = FasterRCNNPipeline(
            num_classes=NUM_CLASSES,
            backbone="resnet50",
            min_size=800,
            max_size=1333,
            box_score_thresh=0.05
        )
        
        # 2⃣ Explication du pipeline
        pipeline.explain_pipeline_steps()
        
        # 3⃣ Résumé de l'architecture
        pipeline.get_model_summary()
        
        # 4⃣ Création d'une image de test
        print("\n Création d'une image de test...")
        sample_image = create_test_image()
        print(f" Dimensions de l'image: {sample_image.shape}")
        
        # 5⃣ Détection simple
        print("\n Détection d'objets...")
        results = pipeline.detect_objects(sample_image, confidence_threshold=0.3)
        pipeline.plot_detection_results(results, CLASS_NAMES)
        
        return pipeline, results
        
    except Exception as e:
        print(f" Erreur: {e}")
        import traceback
        traceback.print_exc()
        return None, None

def create_test_image(width: int = 640, height: int = 480) -> np.ndarray:
    """Crée une image de test avec des formes colorées"""
    
    # Image noire
    image = np.zeros((height, width, 3), dtype=np.uint8)
    
    # Ajouter quelques formes
    cv2.rectangle(image, (100, 100), (200, 200), (0, 255, 0), -1)  # Rectangle vert
    cv2.circle(image, (400, 150), 50, (255, 0, 0), -1)  # Cercle rouge
    cv2.rectangle(image, (300, 300), (450, 400), (0, 0, 255), -1)  # Rectangle bleu
    cv2.ellipse(image, (150, 350), (80, 40), 0, 0, 360, (255, 255, 0), -1)  # Ellipse cyan
    
    return image

# =========================
# VERSION ULTRA SIMPLIFIÉE
# =========================

def create_simple_faster_rcnn(num_classes: int = 5):
    """Crée un modèle Faster R-CNN simple et fonctionnel"""
    
    # Charger un modèle pré-entraîné et modifier le classifieur
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(
        weights=torchvision.models.detection.FasterRCNN_ResNet50_FPN_Weights.DEFAULT
    )
    
    # Remplacer le classifieur pour le nombre de classes souhaité
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(
        in_features, num_classes
    )
    
    return model

def simple_demo():
    """Démonstration ultra-simplifiée"""
    
    print(" DÉMONSTRATION FASTER R-CNN ULTRA-SIMPLIFIÉE")
    print("=" * 50)
    
    # 1. Créer le modèle
    model = create_simple_faster_rcnn(num_classes=5)
    model.eval()
    
    # 2. Créer une image de test
    image = create_test_image()
    
    # 3. Prétraitement
    transform = torchvision.transforms.Compose([
        torchvision.transforms.ToTensor(),
    ])
    
    image_tensor = transform(image).unsqueeze(0)
    
    # 4. Détection
    with torch.no_grad():
        predictions = model(image_tensor)
    
    # 5. Visualisation
    boxes = predictions[0]['boxes'].numpy()
    scores = predictions[0]['scores'].numpy()
    labels = predictions[0]['labels'].numpy()
    
    # Filtrer par score
    keep = scores > 0.3
    boxes = boxes[keep]
    scores = scores[keep]
    labels = labels[keep]
    
    # Afficher les résultats
    print(f" Objets détectés: {len(boxes)}")
    for i, (box, score, label) in enumerate(zip(boxes, scores, labels)):
        print(f"   {i+1}. Classe {label}, Score: {score:.3f}, Box: {box.astype(int)}")
    
    # Visualiser
    visualize_results(image, boxes, scores, labels)
    
    return model, predictions

def visualize_results(image, boxes, scores, labels):
    """Visualisation simple des résultats"""
    
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    fig, ax = plt.subplots(1, 1, figsize=(10, 6))
    ax.imshow(image_rgb)
    
    colors = ['red', 'blue', 'green', 'orange', 'purple']
    
    for box, score, label in zip(boxes, scores, labels):
        x1, y1, x2, y2 = box
        color = colors[label % len(colors)]
        
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, 
                               linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        
        ax.text(x1, y1-10, f'Class {label}: {score:.3f}', 
               bbox=dict(boxstyle="round,pad=0.3", facecolor=color, alpha=0.7),
               fontsize=8, color='white', fontweight='bold')
    
    ax.set_title(f'Résultats de Détection - {len(boxes)} objets', fontsize=14)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    print(" CHOISISSEZ LA DÉMONSTRATION:")
    print("1. Pipeline complet avec explications")
    print("2. Version ultra-simplifiée")
    
    choice = input("Entrez votre choix (1 ou 2): ").strip()
    
    if choice == "1":
        print("\n" + "="*60)
        print(" LANCEMENT DU PIPELINE COMPLET")
        print("="*60)
        pipeline, results = demo_pipeline_simple()
    elif choice == "2":
        print("\n" + "="*60)
        print(" LANCEMENT DE LA VERSION SIMPLIFIÉE")
        print("="*60)
        model, predictions = simple_demo()
    else:
        print(" Choix invalide. Lancement de la version simplifiée...")
        model, predictions = simple_demo()

 CHOISISSEZ LA DÉMONSTRATION:
1. Pipeline complet avec explications
2. Version ultra-simplifiée


Entrez votre choix (1 ou 2):  1



 LANCEMENT DU PIPELINE COMPLET
 Initialisation du pipeline Faster R-CNN...
 Pipeline Faster R-CNN initialisé avec resnet50
 Nombre de classes: 5
  Device: cpu

 EXPLICATION DÉTAILLÉE DU PIPELINE FASTER R-CNN

1⃣ EXTRACTION DES FEATURES
    Le backbone CNN (ResNet50 avec FPN) extrait des feature maps multi-échelles
    Sortie: Feature maps à 4 échelles différentes ['0', '1', '2', '3']
    256 canaux par feature map

2⃣ RPN (REGION PROPOSAL NETWORK)
    Génère ~2000 propositions de régions contenant potentiellement des objets
    Sortie: Boîtes candidates + scores d'objectivité
    5 tailles × 3 ratios = 15 ancres par position

3⃣ ROI POOLING/ALIGN
    Redimensionne chaque région d'intérêt à une taille fixe (7×7)
    Sortie: Features normalisées pour la classification
    Permet de traiter des régions de tailles variables

4⃣ CLASSIFICATION ET RÉGRESSION
    Classe les objets et affine les coordonnées des bounding boxes
    Sortie: Classes prédites + boxes raffinées + scores de confianc

Traceback (most recent call last):
  File "<tmp>\3685099294.py", line 432, in demo_pipeline_simple
    results = pipeline.detect_objects(sample_image, confidence_threshold=0.3)
  File "<tmp>\3685099294.py", line 202, in detect_objects
    predictions = self.model(image_tensor)
  File "<env>\lib\site-packages\torch\nn\modules\module.py", line 1775, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
  File "<env>\lib\site-packages\torch\nn\modules\module.py", line 1786, in _call_impl
    return forward_call(*args, **kwargs)
  File "<env>\lib\site-packages\torchvision\models\detection\generalized_rcnn.py", line 118, in forward
    detections, detector_losses = self.roi_heads(features, proposals, images.image_sizes, targets)
  File "<env>\lib\site-packages\torch\nn\modules\module.py", line 1775, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
  File "<env>\lib\site-packages\torch\nn\modules\module.py", line 1786, in _call_impl
    return forward_call(*args,

In [ ]:
"""
 DINO COMPLETE PIPELINE
Train → Test → Weather Evaluation
Ultra-robuste, tout en un, prêt pour GPU
"""

import os
import json
import glob
import random
from pathlib import Path
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import pandas as pd
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score
from torchvision.ops import box_iou
import matplotlib.pyplot as plt

# =========================
# CONFIGURATION GLOBALE
# =========================
print("\n" + "="*80)
print(" DINO OBJECT DETECTION - PIPELINE COMPLET")
print("="*80)

# Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n  Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Chemins - DATA3
BASE_DIR = "./Data3"
TRAIN_DIR = os.path.join(BASE_DIR, "train")
TEST_DIR = os.path.join(BASE_DIR, "test")
RESULTS_DIR = os.path.join(BASE_DIR, "results")

# Sauvegarde
SAVE_DIR = os.path.join(BASE_DIR, "models_dino")
WEATHER_EVAL_DIR = os.path.join(BASE_DIR, "weather_evaluation_dino")

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(WEATHER_EVAL_DIR, exist_ok=True)

# Hyperparamètres
NUM_CLASSES = 5
BATCH_SIZE = 4  # Ajuster selon votre GPU
EPOCHS = 30
LR = 1e-4
CONF_THRESHOLD = 0.5
EARLY_STOP_PATIENCE = 7

print(f"\n  Configuration:")
print(f"   Classes: {NUM_CLASSES}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Learning Rate: {LR}")
print(f"   Early Stop Patience: {EARLY_STOP_PATIENCE}")

# =========================
# INSTALLATION TRANSFORMERS (si nécessaire)
# =========================
try:
    from transformers import AutoImageProcessor, AutoModelForObjectDetection
    print("\n Transformers installé")
except ImportError:
    print("\n  Installation de transformers...")
    import subprocess
    subprocess.check_call(["pip", "install", "transformers", "-q"])
    from transformers import AutoImageProcessor, AutoModelForObjectDetection
    print(" Transformers installé")

# =========================
# VÉRIFICATION DES CHEMINS
# =========================
print(f"\n Vérification des chemins...")
for path_name, path in [("Train", TRAIN_DIR), ("Test", TEST_DIR), ("Results", RESULTS_DIR)]:
    if os.path.exists(path):
        print(f"    {path_name}: {path}")
    else:
        print(f"    {path_name} introuvable: {path}")
        exit(1)

# =========================
# CONVERSION YOLO → COCO
# =========================
def convert_yolo_to_coco(base_path, force=False):
    """Convertit les labels YOLO en format COCO JSON"""
    images_dir = os.path.join(base_path, "images")
    labels_dir = os.path.join(base_path, "labels")
    coco_dir = os.path.join(base_path, "labels_coco")
    
    if not os.path.exists(images_dir) or not os.path.exists(labels_dir):
        print(f"     Dossiers manquants dans {base_path}")
        return 0
    
    os.makedirs(coco_dir, exist_ok=True)
    
    # Vérifier si déjà converti
    json_files = glob.glob(os.path.join(coco_dir, "*.json"))
    if len(json_files) > 0 and not force:
        print(f"    Déjà converti ({len(json_files)} fichiers)")
        return len(json_files)
    
    print(f"    Conversion YOLO → COCO...")
    
    label_files = glob.glob(os.path.join(labels_dir, "*.txt"))
    converted = 0
    
    for label_file in tqdm(label_files, desc="   ", leave=False):
        base_name = Path(label_file).stem
        
        # Chercher l'image correspondante
        image_file = None
        for ext in ['.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG']:
            test_path = os.path.join(images_dir, base_name + ext)
            if os.path.exists(test_path):
                image_file = test_path
                break
        
        if not image_file:
            continue
        
        try:
            # Charger l'image pour obtenir les dimensions
            img = Image.open(image_file)
            img_width, img_height = img.size
            
            # Lire les annotations YOLO
            boxes = []
            labels = []
            
            with open(label_file, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) < 5:
                        continue
                    
                    cls = int(float(parts[0]))
                    x_center = float(parts[1])
                    y_center = float(parts[2])
                    width = float(parts[3])
                    height = float(parts[4])
                    
                    # Validation
                    if not (0 <= cls < NUM_CLASSES):
                        continue
                    if not (0 <= x_center <= 1 and 0 <= y_center <= 1):
                        continue
                    if not (0 < width <= 1 and 0 < height <= 1):
                        continue
                    
                    boxes.append([x_center, y_center, width, height])
                    labels.append(cls)
            
            # Sauvegarder en JSON
            json_path = os.path.join(coco_dir, base_name + ".json")
            with open(json_path, 'w') as f:
                json.dump({
                    "boxes": boxes,
                    "class_labels": labels
                }, f, indent=2)
            
            converted += 1
            
        except Exception as e:
            print(f"     Erreur {base_name}: {e}")
            continue
    
    print(f"    {converted} fichiers convertis")
    return converted

print(f"\n Conversion des labels...")
convert_yolo_to_coco(TRAIN_DIR)
convert_yolo_to_coco(TEST_DIR)

# =========================
# DATASET CUSTOM
# =========================
class DINODataset(Dataset):
    def __init__(self, base_path):
        self.images_dir = os.path.join(base_path, "images")
        self.labels_dir = os.path.join(base_path, "labels_coco")
        
        # Lister toutes les images
        self.images = []
        for ext in ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']:
            self.images.extend(glob.glob(os.path.join(self.images_dir, ext)))
        
        self.images = sorted(self.images)
        
        # Filtrer celles qui ont des labels
        self.valid_images = []
        for img_path in self.images:
            base_name = Path(img_path).stem
            json_path = os.path.join(self.labels_dir, base_name + ".json")
            if os.path.exists(json_path):
                self.valid_images.append(img_path)
        
        print(f"      Dataset: {len(self.valid_images)} images valides")
    
    def __len__(self):
        return len(self.valid_images)
    
    def __getitem__(self, idx):
        img_path = self.valid_images[idx]
        base_name = Path(img_path).stem
        json_path = os.path.join(self.labels_dir, base_name + ".json")
        
        # Charger image
        image = Image.open(img_path).convert("RGB")
        
        # Charger annotations
        with open(json_path, 'r') as f:
            data = json.load(f)
        
        boxes = torch.tensor(data["boxes"], dtype=torch.float32) if data["boxes"] else torch.zeros((0, 4))
        labels = torch.tensor(data["class_labels"], dtype=torch.int64) if data["class_labels"] else torch.zeros((0,), dtype=torch.int64)
        
        return image, {"boxes": boxes, "class_labels": labels}

def collate_fn(batch):
    images, targets = zip(*batch)
    return list(images), list(targets)

# =========================
# CRÉER LES DATALOADERS
# =========================
print(f"\n Création des dataloaders...")

train_dataset = DINODataset(TRAIN_DIR)
test_dataset = DINODataset(TEST_DIR)

train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=True if torch.cuda.is_available() else False
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=True if torch.cuda.is_available() else False
)

print(f"    Train: {len(train_dataset)} images, {len(train_loader)} batches")
print(f"    Test: {len(test_dataset)} images, {len(test_loader)} batches")

# =========================
# CHARGER LE MODÈLE DINO
# =========================
print(f"\n Chargement du modèle DINO...")

try:
    # Essayer DINO-4scale
    processor = AutoImageProcessor.from_pretrained("IDEA-Research/grounding-dino-tiny")
    model = AutoModelForObjectDetection.from_pretrained(
        "IDEA-Research/grounding-dino-tiny",
        num_labels=NUM_CLASSES,
        ignore_mismatched_sizes=True
    )
    print("    Modèle: Grounding DINO Tiny")
except Exception as e:
    print(f"     Erreur Grounding DINO: {e}")
    try:
        # Fallback: DETR comme DINO
        processor = AutoImageProcessor.from_pretrained("facebook/detr-resnet-50")
        model = AutoModelForObjectDetection.from_pretrained(
            "facebook/detr-resnet-50",
            num_labels=NUM_CLASSES,
            ignore_mismatched_sizes=True
        )
        print("    Modèle: DETR-ResNet50 (fallback)")
    except Exception as e2:
        print(f"    Impossible de charger un modèle: {e2}")
        exit(1)

model.to(DEVICE)
print(f"    Modèle sur {DEVICE}")

# Optimiseur
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.7)

# =========================
# HELPER FUNCTIONS
# =========================
def cxcywh_to_xyxy(boxes, img_w, img_h):
    """Convertir format YOLO vers xyxy"""
    if boxes.numel() == 0:
        return torch.zeros((0, 4))
    
    cx = boxes[:, 0] * img_w
    cy = boxes[:, 1] * img_h
    w = boxes[:, 2] * img_w
    h = boxes[:, 3] * img_h
    
    x1 = cx - w/2
    y1 = cy - h/2
    x2 = cx + w/2
    y2 = cy + h/2
    
    return torch.stack([x1, y1, x2, y2], dim=1)

# =========================
# TRAIN FUNCTION
# =========================
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0
    n_batches = 0
    
    pbar = tqdm(loader, desc="   Training")
    for images, targets in pbar:
        try:
            # Préparer les inputs
            encoding = processor(images=images, return_tensors="pt")
            pixel_values = encoding["pixel_values"].to(device)
            
            # Filtrer targets vides
            valid_targets = []
            valid_indices = []
            for i, t in enumerate(targets):
                if t["class_labels"].numel() > 0:
                    valid_targets.append({
                        "class_labels": t["class_labels"].to(device),
                        "boxes": t["boxes"].to(device)
                    })
                    valid_indices.append(i)
            
            if len(valid_targets) == 0:
                continue
            
            # Ajuster pixel_values
            pixel_values = pixel_values[valid_indices]
            
            # Forward
            optimizer.zero_grad()
            outputs = model(pixel_values=pixel_values, labels=valid_targets)
            loss = outputs.loss
            
            # Backward
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.1)
            optimizer.step()
            
            total_loss += loss.item()
            n_batches += 1
            
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
            
        except Exception as e:
            print(f"\n     Erreur batch: {e}")
            continue
    
    return total_loss / max(n_batches, 1)

# =========================
# EVAL FUNCTION
# =========================
def evaluate(model, loader, device, conf_threshold=CONF_THRESHOLD, desc="Eval"):
    model.eval()
    
    all_gt = []
    all_pred = []
    all_ious = []
    total_det = 0
    total_gt = 0
    
    with torch.no_grad():
        pbar = tqdm(loader, desc=f"   {desc}", leave=False)
        for images, targets in pbar:
            try:
                encoding = processor(images=images, return_tensors="pt")
                pixel_values = encoding["pixel_values"].to(device)
                
                outputs = model(pixel_values=pixel_values)
                
                # Post-process
                target_sizes = torch.tensor([img.size[::-1] for img in images])
                results = processor.post_process_object_detection(
                    outputs, 
                    target_sizes=target_sizes, 
                    threshold=conf_threshold
                )
                
                for i, res in enumerate(results):
                    pred_boxes = res["boxes"] if len(res["boxes"]) > 0 else torch.zeros((0, 4))
                    pred_labels = res["labels"] if len(res["labels"]) > 0 else torch.zeros((0,), dtype=torch.int64)
                    
                    gt_boxes_norm = targets[i]["boxes"]
                    gt_labels = targets[i]["class_labels"]
                    
                    total_det += len(pred_labels)
                    total_gt += len(gt_labels)
                    
                    if len(pred_boxes) == 0 or len(gt_boxes_norm) == 0:
                        continue
                    
                    # Convertir GT
                    img_w, img_h = images[i].size
                    gt_boxes = cxcywh_to_xyxy(gt_boxes_norm, img_w, img_h)
                    
                    # IoU matching
                    iou = box_iou(gt_boxes, pred_boxes)
                    max_iou, best_idx = iou.max(dim=1)
                    all_ious.extend(max_iou.cpu().numpy().tolist())
                    
                    matched = max_iou >= 0.5
                    if matched.sum() > 0:
                        all_gt.extend(gt_labels[matched].cpu().numpy().tolist())
                        all_pred.extend(pred_labels[best_idx[matched]].cpu().numpy().tolist())
                
            except Exception as e:
                print(f"\n     Erreur eval: {e}")
                continue
    
    # Calculer métriques
    if len(all_gt) > 0 and len(all_pred) > 0:
        prec = precision_score(all_gt, all_pred, average="macro", zero_division=0)
        rec = recall_score(all_gt, all_pred, average="macro", zero_division=0)
        f1 = f1_score(all_gt, all_pred, average="macro", zero_division=0)
    else:
        prec, rec, f1 = 0, 0, 0
    
    map50 = np.mean([1 for iou in all_ious if iou >= 0.5]) if all_ious else 0
    map5095 = np.mean(all_ious) if all_ious else 0
    
    return {
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1-Score": round(f1, 4),
        "mAP@50": round(map50, 4),
        "mAP@50-95": round(map5095, 4),
        "Detections": total_det,
        "Ground_Truth": total_gt,
        "Matched": len(all_gt)
    }

# =========================
# EARLY STOPPING
# =========================
class EarlyStopping:
    def __init__(self, patience=7, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.best_epoch = 0
    
    def __call__(self, score, epoch):
        if self.best_score is None:
            self.best_score = score
            self.best_epoch = epoch
            return True
        elif score < self.best_score + self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
            return False
        else:
            self.best_score = score
            self.best_epoch = epoch
            self.counter = 0
            return True

early_stopping = EarlyStopping(patience=EARLY_STOP_PATIENCE)

# =========================
# ENTRAÎNEMENT PRINCIPAL
# =========================
print("\n" + "="*80)
print(" DÉBUT DE L'ENTRAÎNEMENT")
print("="*80)

train_history = []
test_history = []
best_f1 = 0

for epoch in range(1, EPOCHS + 1):
    print(f"\n{'='*80}")
    print(f" Epoch {epoch}/{EPOCHS}")
    print(f"{'='*80}")
    
    # Train
    loss = train_one_epoch(model, train_loader, optimizer, DEVICE)
    lr_scheduler.step()
    print(f"\n    Loss: {loss:.4f}")
    
    # Eval Train
    print(f"\n    Évaluation Train:")
    train_metrics = evaluate(model, train_loader, DEVICE, desc="Train")
    print(f"      P={train_metrics['Precision']:.3f} | R={train_metrics['Recall']:.3f} | "
          f"F1={train_metrics['F1-Score']:.3f} | mAP={train_metrics['mAP@50']:.3f}")
    
    # Eval Test
    print(f"\n    Évaluation Test:")
    test_metrics = evaluate(model, test_loader, DEVICE, desc="Test")
    print(f"      P={test_metrics['Precision']:.3f} | R={test_metrics['Recall']:.3f} | "
          f"F1={test_metrics['F1-Score']:.3f} | mAP={test_metrics['mAP@50']:.3f}")
    
    # Sauvegarder checkpoint
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'train_metrics': train_metrics,
        'test_metrics': test_metrics,
        'loss': loss
    }
    
    torch.save(checkpoint, os.path.join(SAVE_DIR, f"dino_epoch{epoch}.pt"))
    
    # Meilleur modèle
    is_best = early_stopping(test_metrics['F1-Score'], epoch)
    if is_best:
        torch.save(checkpoint, os.path.join(SAVE_DIR, "best_dino_model.pt"))
        best_f1 = test_metrics['F1-Score']
        print(f"\n    Meilleur modèle sauvegardé! (F1={best_f1:.4f})")
    
    # Historique
    train_history.append({"epoch": epoch, "loss": loss, **train_metrics})
    test_history.append({"epoch": epoch, **test_metrics})
    
    pd.DataFrame(train_history).to_csv(os.path.join(SAVE_DIR, "train_history.csv"), index=False)
    pd.DataFrame(test_history).to_csv(os.path.join(SAVE_DIR, "test_history.csv"), index=False)
    
    # Early stopping
    if early_stopping.early_stop:
        print(f"\n    Early stopping à l'epoch {epoch}")
        print(f"    Meilleur: Epoch {early_stopping.best_epoch} (F1={early_stopping.best_score:.4f})")
        break

print("\n" + "="*80)
print(" ENTRAÎNEMENT TERMINÉ")
print(f" Modèles: {SAVE_DIR}")
print(f" Meilleur F1: {best_f1:.4f} (Epoch {early_stopping.best_epoch})")
print("="*80)

# =========================
# ÉVALUATION SUR WEATHER
# =========================
print("\n" + "="*80)
print("  ÉVALUATION SUR CONDITIONS MÉTÉO")
print("="*80)

# Charger le meilleur modèle
print("\n Chargement du meilleur modèle...")
best_checkpoint = torch.load(os.path.join(SAVE_DIR, "best_dino_model.pt"))
model.load_state_dict(best_checkpoint['model_state_dict'])
model.eval()
print("    Meilleur modèle chargé")

# Détecter tous les dossiers weather
weather_folders = [f for f in os.listdir(RESULTS_DIR) 
                   if os.path.isdir(os.path.join(RESULTS_DIR, f))]

weather_conditions = ["fog", "glare", "night", "rain", "snow"]
all_weather_results = []

print(f"\n Dossiers détectés: {len(weather_folders)}")

for folder in sorted(weather_folders):
    folder_lower = folder.lower()
    
    # Filtrer uniquement weather_levelX
    is_weather = any(w in folder_lower for w in weather_conditions)
    if not is_weather:
        continue
    
    print(f"\n{'='*80}")
    print(f" {folder}")
    print(f"{'='*80}")
    
    weather_path = os.path.join(RESULTS_DIR, folder)
    
    # Convertir si nécessaire
    convert_yolo_to_coco(weather_path)
    
    # Créer dataset
    try:
        dataset = DINODataset(weather_path)
        if len(dataset) == 0:
            print("     Dataset vide")
            continue
        
        loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, 
                           collate_fn=collate_fn, num_workers=0)
        
        # Extraire weather et level
        weather, level = None, None
        for w in weather_conditions:
            if w in folder_lower:
                weather = w
                for l in range(1, 6):
                    if f"level{l}" in folder_lower or f"_{l}" in folder_lower:
                        level = l
                        break
                break
        
        # Évaluer
        metrics = evaluate(model, loader, DEVICE, desc=folder)
        metrics["Weather"] = weather or "unknown"
        metrics["Level"] = level or 0
        metrics["Folder"] = folder
        
        all_weather_results.append(metrics)
        
        print(f"\n    P={metrics['Precision']:.3f} | R={metrics['Recall']:.3f} | "
              f"F1={metrics['F1-Score']:.3f} | mAP={metrics['mAP@50']:.3f}")
        
    except Exception as e:
        print(f"    Erreur: {e}")
        continue

# Sauvegarder résultats weather
if len(all_weather_results) > 0:
    df_weather = pd.DataFrame(all_weather_results)
    df_weather = df_weather.sort_values(["Weather", "Level"])
    
    results_path = os.path.join(WEATHER_EVAL_DIR, "weather_results.csv")
    df_weather.to_csv(results_path, index=False)
    
    # Résumé par condition
    summary = df_weather.groupby("Weather")[["Precision", "Recall", "F1-Score", "mAP@50"]].mean()
    summary_path = os.path.join(WEATHER_EVAL_DIR, "weather_summary.csv")
    summary.to_csv(summary_path)
    
    print("\n" + "="*80)
    print(" RÉSUMÉ PAR CONDITION")
    print("="*80)
    print(summary.round(4))
    
    print(f"\n Résultats: {WEATHER_EVAL_DIR}")
else:
    print("\n  Aucun résultat weather")

# =========================
# RÉSUMÉ FINAL
# =========================
print("\n" + "="*80)
print(" PIPELINE COMPLET TERMINÉ")
print("="*80)

print(f"\n RÉSULTATS FINAUX:")
print(f"   Train - F1: {train_history[-1]['F1-Score']:.4f}, mAP: {train_history[-1]['mAP@50']:.4f}")
print(f"   Test  - F1: {test_history[-1]['F1-Score']:.4f}, mAP: {test_history[-1]['mAP@50']:.4f}")
print(f"   Weather: {len(all_weather_results)} conditions évaluées")

print(f"\n FICHIERS GÉNÉRÉS:")
print(f"   Modèles: {SAVE_DIR}")
print(f"   Weather: {WEATHER_EVAL_DIR}")

print("\n" + "="*80)
print(" TOUT EST TERMINÉ!")
print("="*80)


 DINO OBJECT DETECTION - PIPELINE COMPLET

  Device: cpu

  Configuration:
   Classes: 5
   Batch Size: 4
   Epochs: 30
   Learning Rate: 0.0001
   Early Stop Patience: 7

 Transformers installé

 Vérification des chemins...
    Train: ./Data3\train
    Test: ./Data3\test
    Results: ./Data3\results

 Conversion des labels...
    Conversion YOLO → COCO...


    677 fichiers convertis
    Conversion YOLO → COCO...


    100 fichiers convertis

 Création des dataloaders...
      Dataset: 1354 images valides
      Dataset: 200 images valides
    Train: 1354 images, 339 batches
    Test: 200 images, 50 batches

 Chargement du modèle DINO...


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


     Erreur Grounding DINO: Unrecognized configuration class <class 'transformers.models.grounding_dino.configuration_grounding_dino.GroundingDinoConfig'> for this kind of AutoModel: AutoModelForObjectDetection.
Model type should be one of ConditionalDetrConfig, DFineConfig, DabDetrConfig, DeformableDetrConfig, DetaConfig, DetrConfig, RTDetrConfig, RTDetrV2Config, TableTransformerConfig, YolosConfig.


...

   Training: 100%|████████████████████████████████████████████████████████████████████████████████████████| 339/339 [1:01:33<00:00, 10.90s/it, loss=2.2582]



    Loss: 2.5727

    Évaluation Train:


      P=1.000 | R=1.000 | F1=1.000 | mAP=1.000

    Évaluation Test:


      P=1.000 | R=1.000 | F1=1.000 | mAP=1.000

    Meilleur modèle sauvegardé! (F1=1.0000)

 Epoch 3/30


   Training:  31%|████████████████████████████▏                                                             | 106/339 [20:56<45:19, 11.67s/it, loss=1.9212]

In [7]:
import matplotlib.pyplot as plt
import numpy as np

# =======================
#  Métriques Résumées
# =======================
models = ["RT-DETR", "DETR", "Faster-RCNN", "YOLOv8"]

f1_weather = {
    "Fog":     [0.9449, 0.8992, 0.9522, 0.8922],
    "Glare":   [0.9978, 0.8813, 0.9935, 0.9361],
    "Night":   [0.9949, 0.8832, 0.9106, 0.8669],
    "Rain":    [0.8576, 0.8829, 0.8318, 0.7616],
    "Snow":    [0.8478, 0.8852, 0.9533, 0.7578]
}

plt.figure(figsize=(12,6))
for i, model in enumerate(models):
    plt.plot(f1_weather.keys(), [f1_weather[w][i] for w in f1_weather.keys()], marker='o', label=model, linewidth=3)

plt.title("Comparaison Robustesse Weather des 4 modèles", fontsize=16)
plt.xlabel("Condition météo")
plt.ylabel("F1 Score")
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend(title="Modèles", fontsize=10)
plt.show()


<Figure size 1200x600 with 1 Axes>